# Datascape 2026: Klasifikasi Kondisi Paket (Aman / Rusak / Lainnya)

Notebook ini menggabungkan **seluruh isi `paket-untuk-dennis.zip`** ke dalam satu file:
27 skrip Python, `BRIEF.md`, `LAPORAN.md`, `BACA-INI.md`, dan semua `REPORT.md` per eksperimen,
ditambah file data kecil yang dibutuhkan (`split/train.csv`, `split/val.csv`,
`test_pseudo_labels_strong.csv`, dan `submission.csv` final) yang ditanam di dalam notebook.

**Cara kerja notebook**

- Setiap skrip ditaruh utuh di sel yang diawali `%%simpan <path>`. Saat sel dijalankan, kodenya
  ditulis ke `ROOT/<path>` (path `/Users/yopiesuryadi/lomba-cv` otomatis diganti dengan `ROOT` Anda).
  Kode di sel **sama persis** dengan file di zip.
- Sel `!{PY} runs/.../train.py ...` menjalankan skrip itu sebagai proses Python terpisah, dengan argumen
  yang tercatat di REPORT masing-masing. Cara ini dipilih (bukan menempel kode langsung ke sel) karena
  skrip memakai `argparse`, `multiprocessing.Pool`, DataLoader dengan worker, dan `from common import *`
  yang tidak berjalan benar bila dieksekusi langsung di kernel Jupyter.
- Sel yang diberi tanda **[jembatan]** adalah tambahan kecil yang tidak ada di zip: langkah yang dulu
  dikerjakan manual dan perlu ada supaya pipeline bisa jalan dari awal sampai akhir.

**Urutan pipeline**

| # | Bagian | Isi | Butuh GPU |
|---|---|---|---|
| 1 | Setup | konfigurasi path, pulihkan split & file data kecil | - |
| 2 | `eda` | audit data: md5/phash, duplikat test-train, label konflik, kontak sheet | tidak |
| 3 | `yolo-cls` | YOLO11s-cls 320px | ya |
| 4 | `effnet-focal` | EfficientNetV2-b0 + focal loss | ya |
| 5 | `timm-convnext` | ConvNeXt-tiny putaran 1 (192px) | ya |
| 6 | `fm-probe` | embedding CLIP ViT-B/16 + DINOv2, logistic regression | ekstraksi saja |
| 7 | `emb-heads` | SVM / MLP / kNN di atas embedding putaran 1 | tidak |
| 8 | `convnext-r2` | ConvNeXt-tiny putaran 2 (224px, letterbox) | ya |
| 9 | `emb-r2` | CLIP ViT-L/14 + multi-crop 448, SVM / MLP | ekstraksi saja |
| 10 | `ensemble` | gabungan akhir + bias kelas + override duplikat | tidak |
| 11 | Verifikasi | bandingkan hasil dengan `submission.csv` di paket | tidak |

Bagian 3, 4, dan 6 tidak dipakai di ensemble final, tapi tetap disertakan supaya semua kode ada di sini.
Ensemble final memakai: **MLP multi-crop (emb-r2) 2 : MLP putaran 1 (emb-heads) 1 : ConvNeXt p1 1 : ConvNeXt p2 1**.

## Ringkasan dari paket

### BACA-INI.md

### Paket Datascape 2026 untuk Dennis

- `submission.csv` : jawaban final, 3.057 baris, format sama dengan sample_submission.csv. Langsung bisa di-submit.
- `LAPORAN.md` : ringkasan pendekatan, angka validasi, temuan data, dan pelajaran.
- `kode/` : semua skrip supaya bisa dijalankan ulang. Tidak berisi data dan bobot model.
  - `BRIEF.md` aturan kerja, `split/` pembagian train/val yang dipakai (seed 42).
  - `runs/<pendekatan>/` kode + REPORT.md tiap model. Yang paling bernilai: `emb-r2` (CLIP multi-crop), `emb-heads` (SVM/MLP di atas embedding), `convnext-r2`.
  - `runs/eda/` audit data: duplikat test-train (`test_train_matches.csv`), 467 override label (`test_pseudo_labels_strong.csv`), label konflik di train (`train_cross_label_pairs.csv`), dua contoh kontak sheet.
  - `ensemble.py` penggabung akhir.

Lingkungan: Python 3.13, torch (MPS/CUDA), timm, ultralytics, open_clip_torch, scikit-learn. Dataset diekstrak ke `data/Train_Set`, `data/Test_Set`, path di skrip menunjuk `~/lomba-cv/`, sesuaikan.

Dua hal yang perlu Dennis cek sendiri sebelum submit: aturan lomba soal model pretrained, dan soal memanfaatkan gambar test yang identik dengan train (467 jawaban memakai itu; kalau dilarang, matikan blok override di `ensemble.py`).

### LAPORAN.md

#### Datascape 2026: klasifikasi kondisi paket. Hasil percobaan 4 Oktober 2026 (dua putaran)

File jawaban: `submission-final-ensemble.csv` (3.057 baris, format sama dengan sample_submission.csv).

##### Angka
Validasi = 15% data train yang disisihkan (1.833 gambar), tidak pernah dipakai melatih.
Karena banyak gambar kembar di dalam dataset, angka juga dihitung pada "validasi bersih":
1.327 gambar yang tidak punya kembaran di data latih. Angka bersih lebih dekat ke skor sebenarnya.

| Pendekatan | Macro F1 val | Macro F1 val bersih |
|---|---|---|
| YOLO11s-cls, 320px, 7 epoch | 0,812 | 0,787 |
| EfficientNetV2-b0, focal loss, 224px | 0,827 | 0,799 |
| CLIP ViT-B/16 + DINOv2 embedding, logistic regression | 0,841 | 0,826 |
| ConvNeXt-tiny fine-tune, 192px, 5 epoch, TTA | 0,874 | 0,844 |
| CLIP+DINOv2 embedding, MLP (rata-rata 3 seed) | 0,909 | 0,879 |
| CLIP+DINOv2 embedding, SVM RBF | 0,912 | 0,884 |
| Ensemble putaran 1 (SVM 1 : MLP 2 : ConvNeXt 2) + bias kelas + override duplikat | 0,921 | 0,899 |
| *Putaran 2* | | |
| ConvNeXt-tiny ulang, 224px, 7 epoch, sampler lembut, letterbox, tanpa noise/rotasi | 0,868 | 0,839 |
| CLIP ViT-L/14 + fitur putaran 1 (1664-d), SVM RBF | 0,922 | 0,889 |
| CLIP ViT-B/16 **multi-crop 448** + fitur putaran 1, MLP 3 seed | 0,921 | 0,894 |
| **Ensemble final (MLP multi-crop 2 : MLP p1 1 : ConvNeXt p1 1 : ConvNeXt p2 1) + bias + override duplikat** | **0,933** | **0,913** |

Per kelas pada validasi bersih (ensemble final): Aman 0,96, Rusak 0,83, Lainnya 0,96.
Kelemahan utama: 49 dari 246 paket Rusak masih ditebak Aman (putaran 1: 60).

##### Yang paling berpengaruh
1. **Fitur foundation model + classifier non-linear** menang atas fine-tune CNN kecil dalam waktu terbatas.
   Embedding CLIP (laion2b) dan DINOv2 (ViT-S/14) digabung jadi 896 dimensi, lalu SVM/MLP di atasnya.
   Lompatan dari logistic regression (0,84) ke SVM (0,91) datang murni dari head non-linear.
2. **Audit data.** Temuan penting:
   - 253 gambar test identik byte dengan gambar train, 1.054 (34%) near-duplicate. 467 di antaranya punya label
     train yang konsisten dan dipakai sebagai override jawaban.
   - 26 grup gambar identik berlabel Aman DAN Rusak sekaligus di train. Label train tidak bersih.
   - 1.371 file .jpg sebenarnya WEBP, 1 AVIF. Baca dengan PIL `convert("RGB")`, jangan cv2.
   - Distribusi test berbeda: lebih banyak gambar besar (render AI 2816x1536, foto 3000x4000) dan gambar
     224x224 ber-noise. Di train kelompok ini hampir semua Aman.
   - Ada "jalan pintas" berbahaya: di beberapa kelompok, Aman = versi ber-noise/rotasi, Rusak = foto bersih,
     padahal objeknya sama-sama utuh. Augmentasi noise/rotasi berat justru memperkuat jalan pintas ini.
3. **Bias per kelas untuk Macro F1.** Setelah ensemble, logit Rusak digeser -0,25 dan Lainnya -1,0
   (dipilih di validasi bersih). Classifier dengan class_weight balanced cenderung over-predict Rusak.
4. **Rata-ratakan prediksi antar gambar test yang kembar** (366 grup) supaya konsisten.

##### Pelajaran putaran 2
- Backbone lebih besar (CLIP ViT-L/14) sendiri TIDAK membantu (0,914 vs 0,912); nilainya baru muncul saat digabung fitur lama.
- Yang menolong kelas Rusak adalah **multi-crop 448px** (5 crop, rata-rata embedding): F1 Rusak bersih 0,76 ke 0,80. Masalahnya detail lokal, bukan kapasitas model.
- ConvNeXt dengan sampler lebih lembut dan tanpa augmentasi noise/rotasi: Aman dan Rusak membaik, tapi recall Lainnya turun (17 dari 63 jadi Aman). Di ensemble ini tertutup oleh model embedding yang kuat di Lainnya.
- Bottleneck ekstraksi fitur di macOS adalah DataLoader worker (spawn), bukan GPU. `num_workers=0` 3x lebih cepat.

##### Yang belum dicoba dan layak dicoba berikutnya
- SVM di atas fitur multi-crop (belum sempat), dan multi-crop untuk DINOv2.
- Fine-tune ConvNeXt di 256-288px dengan GPU kosong (putaran 2 terpaksa 224 karena waktu).
- Membersihkan label konflik di train sebelum melatih ulang.
- Cek aturan lomba: apakah pretrained model dan pemanfaatan duplikat train-test diperbolehkan.

##### Reproduksi
Semua di `~/lomba-cv/` di MacBook Yopie: `BRIEF.md`, `split/`, `runs/<pendekatan>/{train.py,predict.py,REPORT.md}`,
`ensemble.py`. Venv `~/.venvs/lomba-cv`. Total waktu komputasi sekitar 50 menit di Apple M5, semua model dilatih bersamaan.

### BRIEF.md (aturan kerja semua agen)

#### Lomba: klasifikasi kondisi paket (Datascape 2026) — brief bersama semua agen

##### Tugas
Klasifikasi gambar 3 kelas: Aman, Rusak, Lainnya (OOD: food packaging, kantong plastik, paper bag).
Metrik: **Macro F1**. Kelas Lainnya hanya 3% data -> paling menentukan Macro F1.

##### Data (jangan dipindah/diubah)
- /Users/yopiesuryadi/lomba-cv/data/Train_Set/{Aman,Rusak,Lainnya}/  (8800 / 3001 / 423)
- /Users/yopiesuryadi/lomba-cv/data/Test_Set/  (3057 gambar: test_00001.jpg ... campuran jpg/jpeg/png, ada RGBA, ukuran sangat beragam 82px s/d 2816px)
- /Users/yopiesuryadi/lomba-cv/data/sample_submission.csv  (kolom: filename,label)
- Split TETAP, wajib dipakai semua agen supaya hasil bisa dibandingkan:
  /Users/yopiesuryadi/lomba-cv/split/train.csv dan val.csv (kolom path,label; val 15% stratified, seed 42)
  val = Aman 1320, Rusak 450, Lainnya 63.
- Gambar harus di-convert ke RGB saat dibaca (ada RGBA/P). Tangani file korup dengan skip + log.

##### Lingkungan
- Python: /Users/yopiesuryadi/.venvs/lomba-cv/bin/python  (torch, torchvision, timm, ultralytics, open_clip_torch, transformers, sklearn, pandas, opencv). Boleh pip install tambahan ke venv ini.
- Apple M5 32 GB, device "mps". Ada beberapa agen training bersamaan -> pakai batch sedang, num_workers 4, dan imgsz wajar. Hemat waktu: target tiap agen selesai < 40 menit. Jangan menunggu epoch tak terbatas; pakai early stopping / jumlah epoch tetap.
- Cache model pretrained akan diunduh dari internet (HF/ultralytics) — itu normal.

##### Output WAJIB tiap agen, di /Users/yopiesuryadi/lomba-cv/runs/<nama-agen>/
1. val_probs.csv : path,label,p_Aman,p_Rusak,p_Lainnya  (prediksi di val set, untuk ensemble)
2. test_probs.csv: filename,p_Aman,p_Rusak,p_Lainnya  (SEMUA 3057 file test, urutan bebas)
3. submission.csv: filename,label  (format sama persis sample_submission.csv, 3057 baris)
4. REPORT.md: pendekatan, hyperparameter, Macro F1 val (argmax) + per-class F1 + confusion matrix,
   waktu training, dan catatan masalah. Angka harus dari eksekusi nyata, bukan perkiraan.
5. Kode (train.py / predict.py) yang dipakai, supaya bisa dijalankan ulang.

##### Larangan
- Jangan ubah split. Jangan latih memakai val. Jangan sentuh folder lain di mesin ini.
- Jangan klaim angka yang belum dihitung.

## 1. Setup

Sesuaikan `ROOT` di sel berikut. Struktur yang diharapkan di dalam `ROOT`:

```
ROOT/
  data/Train_Set/{Aman,Rusak,Lainnya}/   <- dataset lomba (tidak termasuk di paket)
  data/Test_Set/
  data/sample_submission.csv
```

Folder `split/`, `runs/<pendekatan>/` beserta skripnya akan dibuat oleh notebook ini.

Lingkungan asli: Python 3.13, torch (MPS di Apple M5), timm, ultralytics, open_clip_torch,
scikit-learn, imagehash, scipy, pandas. Beberapa skrip menulis `torch.autocast("mps", ...)` dan
`--device mps` secara eksplisit. Di mesin CUDA/Colab, ubah `DEVICE` di bawah untuk YOLO, dan untuk skrip
lain ganti `"mps"` menjadi `"cuda"` di sel kodenya bila perlu.

In [ ]:
# !pip install timm ultralytics open_clip_torch scikit-learn imagehash scipy pandas  # bila belum terpasang
import os, sys, base64, gzip, shutil
from IPython.core.magic import register_cell_magic

ROOT = os.path.expanduser("~/lomba-cv")        # <-- SESUAIKAN: folder kerja yang berisi data/
ORIG_ROOT = "/Users/yopiesuryadi/lomba-cv"   # path asli di skrip, diganti otomatis
PY = sys.executable                              # interpreter untuk menjalankan skrip
DEVICE = "mps"                                   # "mps" (Mac), "0" (GPU CUDA pertama untuk YOLO), atau "cpu"

os.makedirs(ROOT, exist_ok=True)
os.chdir(ROOT)  # beberapa skrip (eda, ensemble) memakai path relatif terhadap ROOT

@register_cell_magic
def simpan(line, cell):
    """%%simpan <path relatif ROOT>: tulis isi sel ke file, ganti ORIG_ROOT -> ROOT."""
    path = os.path.join(ROOT, line.strip())
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(cell.replace(ORIG_ROOT, ROOT))
    print("ditulis:", path)

for d in ["data/Train_Set", "data/Test_Set", "data/sample_submission.csv"]:
    print(("OK      " if os.path.exists(os.path.join(ROOT, d)) else "BELUM ADA ") + os.path.join(ROOT, d))

### 1b. Pulihkan file data kecil dari paket

File berikut ditanam di notebook (gzip + base64) supaya notebook ini mandiri:

- `split/train.csv`, `split/val.csv`: pembagian train/val tetap (val 15% stratified, seed 42). Path di dalamnya
  diganti ke `ROOT`.
- `runs/eda/test_pseudo_labels_strong.csv`: 467 override label test dari audit duplikat (dipakai `ensemble.py`).
  File ini dibuat manual saat EDA, tidak dihasilkan oleh skrip mana pun di paket.
- `submission_paket.csv`: `submission.csv` final dari paket, untuk verifikasi di bagian akhir.

In [ ]:
DATA = {
"split/train.csv": """
...
Y6/tjRRECwA=
""",
"split/val.csv": """
H4sIAAAAAAACA63dSw8kyW0E4LsB/xNZmyQzmZlH333y4yyMrIEtW1ottJIB/XtXr6YXho/q7zK7M4dCdeWDZDAi+MOXP/3nL373
5ddff/f3f/fdv/349Y8/fveXP/zw268//vmPf/nym99+97s//P7XX/7h3//nu998+dOX7/71j19++/2v/uXrn777x99/+f6nP341
xt37l//1w3/84vXXv/kpM+f95Q/ff/iUk6c+f5eYOz5/Sp1xPn9K7lqfP2XFns9Tvn76eSPFy9QCn3d2gm2XMfLzp+z7bN6Pn9LP
lgHfZR3xdeMMcAS6wNddecDXHc9vAr9obnDBZC3wlLM2OI2VAc5Rjb7imuoJTuMZLb7LHCQIpPi6A7zLiTlFQLrgXTIP2LtzXnCm
o2qKu+6CM12xQAxoksDsdcS7rBbxaK8hksxRIMlcBX7RybnEmRZrVJEi84gGd92ZC3yXWBNEtbEn+EXjtes+/0UxNrnrQGxcV+yX
7GyxX0QdMLeIsGuJqFZTZEFrif0yV4PvsqPBd8lZ4Bytc8Gu6yty75Ob5LsiI4sK8JTeIjY+S00yD3Eax70iwpKvO88gX1fs3ech
JTLVFAjBzCvq6SgCZE6xRgvc3qfF/dJnkwxeIASnRB2wtljpsQQa+hSfKerpTeJ0iRiQAjuMDlHbzyNyzCeFB/vliCzoVSCBNWpx
jkaTeHQEvnsmQYkjRa12BBZ0osFT9hRo6E7xi1aJExBb1AGdYu/WENVE1iQIwRU3wxD57r7TvAvJGS7BVEWP7zkCAsl/KkeR7wpc
6rl4Rb47yf2yCKZKKpvVorIZU3zdJ5KIOD2DYM1DZEEjrqhJjkDaaoIYMKvW572Jvba4GcbdIvMQ+csgUS1T9NX2Olvsly26LRlg
784nUos1ChLVxF0XS7zLNj0+snejRF+tYpFqQiCzOQXmMa8g+80WbLQOcTM8pdogSFsS/EVE+yt+0boCUeoQ+e6zdY+oAwQa2uRm
mEeg5016WZUlIuww3Iol0PNp+vbiKUUYPTVF5hFNOu5ZhI1WpJ5eBPcmrKswNewlOYPgeTwXr8jIJtkvWzAG9yBd7hKVzSJMylPk
KYQH30vslxyGXdQCgw/BjNsRpCe8BO69SQZ/pokkR2DwAiWuJtKhMUhUE7nUc9ktgp4LVgR5lxmbVBOiw7GP+EVP6i2+yya41N4C
Uw3R4+tLmE5xRX00RP4y7iG8Q8KB3AQ9P4KhsQlDI1qcgOcwCq3lJuLcKTDVJOrRPgLF6X1IR4x0FW4SdrSI9k8hQBACgdfNFjnm
SoHMPhcMqSaEmPuUYDrFFis9CHo+QuAvu8QJWIu8C6k+5xXxaI5BdOUCsd5T6I9qELYriWpjC25FbxIDSrDRqsS9e/qQrmWR/hHh
+mWT2EhYekecgDjDMG9Fh2OJu65TaBWScImD8A6fG1NENaJvfFJMoZ2LJnmdUElmCZzhuaQIAknWqAg3lNRHUQIje77uJOpRkQVd
ke9WC3XXJlrLOMSfgagMJun8xBhEVy5yqWiiwm7xi7IJ97xENbF7EZ2N6Lj3FYjSHiLzWMZdZ4vY2EvwX9Lg3ik4s0XYrofcu7mW
6BRmEm2L4DN0C65fXMFG2y2YCEkUeGeJ/nTsaXiqIlMl+qNFIskkHbFYom+fsZIw2AWKE6T6zDD3i9gvZ5LOjzjTuYdxKhQepkQR
njsJn6GIwpfUR3VM3SgUeCX0ankv0RRucqYH4TMIBvsTjohnnNDOPVcdMLJ+zpHIDk1lEz9F+6+fNz/NZScSGFJQv3JVAUEOY5BJ
yixBOnnyOkKEb7FfyFN6kVCyBbyV1cauVpR8JVo/1WFs5Y1YQbQtCQDz1GomgVlE3GIEJUmot2LXbUIODQI2L0GCDCJynyeJLcgi
AEwQQmaTZIoMedhJ8hcBQe5xCe2KNEqSSMLJL6ot4IoXdiKKNUHR2EQm2QZqKyLMIm3uJkYEexrCt6iPBjEkKxKnB2mJ9RW51CSD
FU4cItcRoEcagJeMUdqkhn0+LhmsIL5L7CIka3EzPMeImN5OQosQgxUmsRSOGKZVKEzAhiC1DUIbjyD2xkfsuiJQ/jqkDUXsRzeR
VK0tdl1sUX02MRc5hMLehHyei8hSBpFJBhFOXBFJ9k5iFiGM2uoQfHeT8QxbyLAXkQOOKW7MvoJe9CSqAoEkgtjeoiaJJBZrhtZZ
BPe+xvj8EjmgMAE7i/TVBiF6xCUWJcTShpiw5z6EeCiIqqsIkp9iyMOTkIl3IVbLz6YzjXsx9XlMwYvoNmw/wfg+fci0lJ+c2j7m
RbQwW+4jbru4IoOJIgRGguU/qQcR1ora5sxFDGuN3bgxehD7pRaxUtpklFiQgYdNbFcOsfY74he9fCmFxFHgOLEMwkvwiiZjE+8h
1ENiEEiEb03w81lkqHZPgnqIM51LnICTIpLUDNEtJwNg9hA063UXERoQcwVC1o4hKoHXfEBRf5Jx2IcI1JsM1SZj8JL0OLKJgVcZ
Hk0T1ENUNs/WJaNQn5zhY1r+2MbkntgBHxJhLxmS3GQEdYuM7ClJLjGhbjJCQPShSOc+iT39INyVXshOL8nIz0UQXiL/F4bY6yZh
ERwzSEmw2gi3PwjyfRaRzw2Rex+CtEVdYs6Nup8kx9xE/CzYOM+1K7p8Z5Gx8qRzP4PYo4lBJ2cNwmwWezeXYOOUsYQfIsLuY8ZU
icqmr8gx50xixkEUJS1WOm4TMzGBwbfB67aIjXHI2GfSW36uOjLeeJOR5aJP0hkkZ0CjLRcZqi14NCXUlkV0tYMYVdW5ZOyQyOtq
CZ3BIffL7mF4VwL3PkkYdoNUNmRAA+HeNvkuM8PYgImvS3KGvELjG4uYlBBcKoPYjV+CtKUwFV53kzqAmMQ20T4TVv7covqsMYjh
m1DEPhWfoMYR/WgQjcxrqQm3X9zeS3BvNxmzGUlGlNxNcinCAyYM3jVEB3WQUYXzEJZeCKQth8DIntqe2MaJW2ptMgy4FjHlE7Va
EpvkDsIhIOzFQSwPI0X3v0ogbbkF/6XIvdtbMCnPED5488XH/JwVMQUyW4SU//wikmOKE1BXoKGLKBxjCeVn1SCDrEVHrIgXx0mB
eZzZBmkTiNImPni7yeBQ4vc2RVRr4vMwr8gxawtcapCht91kkFIIzuw8RRQyxISaWBNHCB7ZnsbrR1QTeZLgDGWcQwli3UaHfYif
JBmfLno2ecgvSpE1rxTfpUcY3QRR4BEmQjbh14ncO6folQfpH2WSsUM9iH+XqPj2KaJVEPfuzKf6/LhuPOhMiyrrNWlWRNgi1cQl
zP4kasBBPMuD6BsLjFZ4Eo8gQ9iJBpXoA2YQhYzpuIshPYvkmOsaTz5yAghel0sMR8vZZI3IcPqJnKzN+C7hAtZJesICrxvXMJJJ
lXVE73OSIaZB9LAzCXtxmL4amWaQAsXZRH0xm1TlRfRqpOJr4tS2U3CJq4xaR+hh1yQM0zOJnyQZCH/Facwmk6FCoKGbaP/bDPZb
RPtfomcziGPWJG6dQdyj9ybzgkj+cuqaM03QrSCmcUKz3MQVpJbApebaZORnkG4uUY/mIVaFxLGZRNiKQ/xfSJVFGD2T4AyTOAjk
ESfgRYAR3Vyx0ofobIIon59CgFTlh7jrEB3fIVnQuKSGJRMcCStiXaJxD6JVmEGmJhqkTah1Tk6CKJHannij1SLOc0cwBkcSTlsK
dvTzWUhvQnTE4ohIsqao+CZhL+Ygcyuu4LSdEAq8dUTm8eSYhE1POPlT5HVPNXGJP6ZAcbLi897nk9aRbotBK4ooqsg8+AxyGkXF
91zegku8RJ/kEE/t5yCJe/cSraWZmLWb+NcRleQhniuEjfZEWDKxXOS7TTpiUYJbsUhNMojnypwCOWmiNJtV/XlUm5kLqJjGBE+J
MPMORS/rSb7JLBpR22+SeTxnmnhqB/EwJTcmUYTXEY6fi7AXm2SHZwjWeBDn/yAzZBZR1fYhfTXC8xjkBOy/Kja/fnwEhNAmCCV5
hkibawoKe25Bo0lyHJ9YssnQW0KyFgYwB8HEJCC1oBdN811Iq7CKGPBOMVI4lpBJHkJ3bZIeNjE8jEuMIFukh3UFCL+L7N0U0Gy2
oMY1AYOixAkYs4kkfBq7FPJ1kYWjGedBZAai9ZNXQCfdguqUVSaXKiJwFJD1SGOmR6yWk4wAMzbhxGQwRRa0iWV5EhjnrGtAVRED
hqDR7CINpBuEHCrK8j2FZDOuAHj3msTqJ02cJtmh2HWviXwiTgsR6pNM5eeQdV5xv8xDyBVXtFCLDDPYZCxxJBl3SCjsvUSVtcg5
ynmD2DFtYqZHLG0GqSbI0OdNxliXiAF9BMU05iIjBjex5CM35kCWn2KYwRbWLauTDFpOYgZMTCnJINTYolY7SejRZugzaRU+IbaJ
lHUSChjBdwVincfQF8lprDaEhkPGWIuaJAlCcItYuU9iUSJq2FhkWEqIlc6JxqkeYghPTBHIeKkjbsy1hZh7L4EF7dykni5iCD9N
P4DIAUVf7amnSa1GbIeGGGjTKXgeHSSvm8S+ixidzL7iu6To8e3aRAoyzbsQwX2QfoC5pYbZu+Rm2KTLbWjjBCEYJJca4sZc5AQ8
VfklLJokFR8ZwV6GiyN2HckOn68ruDhkpeuSXXfILfUkDcKWVQhKnkRVdAqJyD2IBdJGw7uIicYS/YAdpLJpYjtkMDIymn6UqMrX
EvhLFJG9LZHXZYhMtQkHson1XA7Rhz2T2MoTXnOEMMjMKzocSZiUO8Vd14N0UJOY0aSoJioFR+lEkqxZVDZBOoWDmPQcMgq4cpL7
hQxNXGREzyDvQgaJVQlcqs2ZHkQLVUILdYIMfb7CouRswn9p0p8mDNOzjHkRGdNGzICjyL1rDD3MEJkmEt9DMo8UgxWeY2TMgAWf
gdj4nhIYfAaJ00X606SrsItgzZfI0xcZibBEF+oQy6x1hQx77UmwINL7JNzzmaKvFrkJi0awgJ8lasJGI9a5TawViuyXMMahi7Cu
BHJC9PaT4HXP5hWctkUGuQdRpbTIXw5R4HUINHQSft0poVmepCqvIvU0YSTHEO8SJXLvVaQPW03uuiY24eJmGAT1a6KFOiXiUZKa
ZJPBlocY8L4opmLvispmLHIzXBHti/B3V1wyltg4cRB+3RKsiE1qtadUI3t3Gx682LvH+DMIa8vnVUgvi9iyEm+RJ8AG8XQ6BMck
41+Iy8MR1cRpMk51TLNfjPkiGZZCOJB7EvNoYrW8yBjrRXKGHMSaW3DaIkzWLFa6GzE0BIcgSNdyCg7kU9sTfYAYlhJbYEF1BGI9
r1AZBFGlDDKW+ASx2yfauX0mcfwUiPVzAoSfB3G27LVJ/4j0PneQLreo+F6iQrBfUmTw0WSs3xER9jnTRH9E3JiS1CRF/KWGYKPt
KTKPOUkk6SbKZ6I2JqMXB9FlVRIuMenO5RJdy3FFB/W04LTNK5wtg2hbiqgBF0HPdws2WpIe365D9I1kiAz5RYPEo3FF1rzIGKVZ
hzApDZc4RJW1hf5ox0+Kh6+fE6ZEUWKsuVss0vODiOWnaImdJsMMmhiqJpk1fklrmaSHlWKl1xKl8CSE7z7EhD0FdNLElnUsY/fW
RFAyCNBwyST3RQymtmncE2M/AcCcGMRaoYjtEBl+sQ75upMIYg+R4AkpSDaxLF+H2HcJ6OQpP0UMIBLfJxyJBnWJtmWdIvY6pLFG
cswzCFxBTPvnEpHkTGOKIOiuQcxFklg4ziugk0Oo2ocYb3WQOmAQm81NYkAdYlxhRiL8NC7o6+eaTZNkEgxSJHYZQcoJw1QlJd8V
BdKKSfze0DBgwTzcZqU3cbATydQqwxls4gKGPNYW6ZSIm4EMbJ57E799wnclAMyLGiR8h47oc+8iZfkl0wGP8aAexDWOFGvirjvE
U3gRP6ZBOiVBnKyb9MTyoJkrpFgTmvAswvYj8zySzFF6AhJROJL+HNGnr7uI6ucQb6hLIEgyA6xEhN1D9NyLTGVdR8yZfT6LaMQS
3VuvRSZDkZbYKDIZqgirTax0bHO/iHy3yGTLLHG/dP/N+e4///nHL//91z9fYP63L/PT3z94UK5vqdCnDxpv37dPHxSjzBvl/oZS
ffyg2+ZBcb45a3z8sWOYB0V9y9U+/kaJ3uipfMr8tLep88dvVGW+0c/gyMfLvxY6tPUNbf/4Qeeos1YbPeigVTuBHvTm/378oEh0
+b+9Ij5e/jvRWXt7+X78jTa6s3ME2pBvi6fPL7YV6NBuFNcKJRF5ptmQMVHs/1me8flVu9AbvcGIz38a+tg5bqFrJNGD1r5qZ091
H6FE66C4ltHoYjsXZSO90IaMq+7sXqg6Ulnt22bk8w1ZKIl4j4f/PByptOYtZf78jVQFOVE28pJnoKv2oHL97av1eXqM7qOXNZxZ
tZ7qYtvo0N6p0hoURcZWWe0YKkB2KSAKXf5DAQhDLf9MhNYUSv3yDYx+viETrdpEIEtWqcIvUeEXKM+OYJk/A30LIREdKK6Fyo92
ovR4bnRox0Ebcg+0alMB4ytUorXQ6Z/oY8cslIzmRmVWXlaLoJ/W7BpBpWiNoVC/i+6jPAhkWWofNXpQnK16RyrPrkTV0UY9yLzo
Y/8sQQT5EQqQtVRao5oHB7XFghEH+qCM7W3BDhJ2dWjRVRs1UOxPlEM+SQTa2YnS4zEnSkbvQDt7qDa9Qo+jEcb2XCPqqlU3ZKFa
ZEzVFjtThexAPy1UcayamVlHQfWlkFGEaI1VKhwp9Hioem2HWrVQmT9qHWYqkk6iIxLqjTJVDzIuu9gUgICw2rwoiXiN9FStQ1Vm
HVX4oav2JVlSFaTqiqYie231jS4ralTHT/Ug30OSPw/ZqjgeqKZ93ihVgEQ/bapyfatsJFR79S61aq3q/r0UQRedtaU4WhsRdF9E
f5TWoAfF23QDNOrQnX2WytjYDakEAwdBrDEQSWdsBfukYh+WYh8qmUeEah020tTEW637+YMU168RfvScNZWwJ+uvoTv7ojz7RUBC
yehiHK2lCr+tKMOK6csu/1KxHx3aLNTOyK0oMXdu1alRCbtqig9Vri9EP/vZD+nzO1v1RaZK2MdR3BrFP1JHJK5Ka5TEMxV+NNZS
TBZVQTKu31aHVvVFIpWEQUnz8qr0OBAbOhSTJQ6qjp5yHRG9h9KKXsYaY4S4qwAElbEp0mBORatWsrNodfqrW/20VFRPVa5vdmhV
W0yJ4Fezmlap6VV1tNXF1kpNr6R5Pw+RAYIB9UataNXKlWGr1qHyiMo1lJxaIVrv+bPAcUC1fBCkEdXKAAdlbHlUn/bt1v55XFvs
GymNH4v9qjq6FwXIeVXdr6Qw9yqovlU74zKsVlEZGCWmlWWhEp0uBbEeRYlZint8puqLoPwor6rXVMiORGlNDMaJQD8teyvGuOKM
jqNEFQjPTnX6syYrs7Yi6CozTtWFmIzovZmPlqqyFX60huKwK8w/FPf4BSD88L3IIZWCZQWzwFD2oMoAp0stPzK/y0SpX7RS+CZT
sCgAYShF3WTadZX5T5QfvUbcq1VTF9tmhlxKcq66EEeJTgdCjwdTZ2xFGtwIG8lQZZazmlOy/KHIFamAqFaKuquGJ1zlV9vKPp1J
83Yrhzh1jdyhPBBUpA3lxKzCUaokYkBRhbK9RhS9J4lQggHGrFO8kVSCgaE8EI5Cj0N1s1SAzK08olrRYY/KswcbC7LU5BwVskN5
Q29HZRiqglRHZCpTDuUyXKxRh2RnMZUS6iotxFG0oVtqmofqiqrlf3a22pDISyNbseon+kZPta4s5gOlNTWUMlNlbK1kZ0vJYE8x
YS4jVrIWtPJja6WmLzVdSFGrjipq7lDddaXvT9aCVklEKSqDconJMVRxjPCjOMq07CgLjFZuQ1cpoXop3ZEad5WMe6ymMAQTeanW
4bmKpdHsYlPzIJmfv2Joqo/94nuoZLSUdQErahTZy3nVq+pI6bJVxhZZKmSrVQuVH+VVc/yU6X0pc+CljG9bkU+Ve/44jH+keGxb
NQ+UNfhQhhODmd/Nv1148k/P/3z/ly/v//6fgS7f/uXTx73t79HjYtC3ew8uRI/La3/st06uWopv7DL0uPHtQKm3S7qyb76ZWopL
v10GXdm3lxJbiqZvV3SjRNuV/dYLZt8OLwW9oKY9s8O+XdrLfdiNUjj0xLAra89slD1ky0ayYTeKvdxr2tBj4+yxOcqddt/ZM7u2
ze/oRqmwgfHYyx2H7Wuvz2/wErs+7dvZK+BtBKjezubG49osIOyNcnDmbnOUso+zYbvmshvFPm7blbVZQHTYEs+eClvivQ0i1L4r
Wx7blGfk/1uK/wUWs3NbxvwBAA==
""",
"runs/eda/test_pseudo_labels_strong.csv": """
H4sIAAAAAAACA4VbTY8cuQ2957cMFiIpUuQxARYL5LAwcskxcIDZIIltLGIvUD8/7ClVd/UMS89A+eJnSuLH44fUP16///jHb//+
8vrt89fXl/+9fn/57euPly+f//n65U8/bv/W8g//9J/f//UyxLZwevnrp59/efnz18/fHgB7A1hvW3474G9/fP/83wci3hCksnGj
l7///JdP7xA0wCIUaBHuO4J4s9YLETyQCCGI8DcEc9/yKxbp8gZw040kShk2Ia1tXUchY+wnEW8ba7WI97u28vsIoNY+HuQZsO9B
JM+hlUWoDSCCEIDbfZcpTSuErLVJrLvjSOqKmlUyRNdmp74ri1Nv2+2vCqKPnXKNQIftyD9J7bEPEuFCiMZPv39LSDrORiP45dOv
7xDTd673YQ/XSK15hVAkYj+Kkm7iXh3FGS3iBizrKBi5OViEWZAMVohAdmPhtb5Ypsq5bflVAKBx7vsSnWzLrwLMk46xRSsl7Ors
bWw8qnjm7uiginicR4OInUFppPOEVzsdcJXpXaLjgpt48t9KBjxtyJpDOfpUacarlLwhtMu4ccZmUiKQAwrZI/NV3iMM/E9mTuoj
Y01GoS/RgxR061GJGId/ae1fAq0mThDRwUF8lgk3jZsVrNEbMmun3UXZ0nlsKuNJBCwkOtN6n/3ISunLm0evZJwyyt27nhGKfLjb
uFttaJEuuvmsEyIDJQrD9xmOl3btA+STDgOphwKe7jHJyehqnxHrYNQ2eYW6b2TVUbQ9ApqoqgSU+vSvzEpluGoVa087VQYKU0aR
ojNfqMiW34ccr3ffuYhW1V0ZSY7p4+2jj6uCkl2h86kqROyeEWF1sKoPYNRAtYpOz5FBF6lAY/rnjaRLXUzXutxmFo27VxhfrGFn
vxlW7NP4dJIWxVkNthYmT/VOVDIQdVknoFHrDGUgy5tBGTYgYjZS7SrJ22ggnm2AzGgzJ70Z7vZXBXlYTqMCBK8rWYujlbKLkwyY
tQasU8esIfnGoUYf/Xwo6FCGwzUg1w9HHjgC8OMIxI+OmldvKOM4IQ91WJg5o4j1GdOLCtH7PK1kdyq9AgSIete2bnAdUrUPvZcB
dX/rDuYvHg0wnAeoNaKh7iHmqOl6n9EMdOpBqCkMaPogFAghcJVplUveCNv1JcGPiH4WMTv1bMzaRlpWTmG2TrIxwIwlHCp92r7T
qaZ+3kWgeIvJCyR9z6HvCYwailk6xnOXGSER6xYj/9O6MkrAAIAjHC/cnNqcwFwpnNogdFDUbVFzKMNPJJnHLvaB6D51TMtK9GaE
Ozf17hUg0Bo0i1mTdC756BdEuvRwIpQ8E7HOjUSCFE4y1manLL3eZndOqXDNjb6r6+mYq70NM0Ork0zPkN7qUiIRM4yMLhE2aeVS
xlEVvQ1V82TVcV2AxmJNKwmYVmHbRItAYpQOEsFry3MTKGIG49hH6R+1wW2s6Y/eDSKjQsRaWzw7iEvv4SrjPAHmMH3vXZ0rBAoD
RmHA7GvyOyaZO6DaRG9giZkWu2Y1MwrG4EmfnTSNXinbEG3xbPkutS0NpAlBFTsJAd+UabDeriJVpjmu7oFI5DzLKlKmTHOQ3iLZ
yqN2eBIdCGG0dguxQxmyZZFQAQSucT6KlkeBeVNmQ7hfnFmhsFBg+DlwuCq2qTcBJN7bepZF/cgUV7c8iVjfEFKfs5d+y7vEFQAw
Ukc9fCJmpDZ5eN8zYgA+6YHMfgwP05Oz/S5OqlObt/Zq61WPRUoBMq9CajymdteTgoSMt2VuK+yQD8sIXGYWj9p806JxpWOothAB
Q0lPV4DVjQKpw32iej4Rtra9NUQrRuvJChkD3jEGRZkxSt0m4CqTjrnctd1t3jCH+unO/xnhDyYvk4GhiSyZIY0f97baN9VKn5Me
g9qmFWvYgDYbgFfMzzyeBW0BCXjUWDe1KRalgjHHANlqJrF4JULB8CYhA64S63gepzcQpQuPmeMl7NH+PK/RQcM5dD0GzY7AgYQj
b3Z9uOcTwB0YBDK9c4MIKOM0ACiZzWc0L2jcwWU8eYcbPS7b+8XEl9zaWl8+WfzqmjIBiKMdzdjJB1ToaTBYeoYfw6FbtVMD4Bpz
tJiOHOWTAPKAZ52EIHzxXImiIRkBRxFwdEgx73m0980rdQSsNIIDJJw4KIEuBoOJAP1LCGgnA3FGoMcgiQCpIJJ21nVugNtnCkcH
RZkiAszzYjaLQlJrm1tbjzoSACIxEb4uUfl4IilDaw/ncj76JAK8oeTGjHYxXY+oXUxDEiLrhJUIcKXBTY93M903L17CJUKRSjUQ
wqBZ/GjmrCe9ebEPf/iXR6WOQFYJ5D0BtxnoqNQ6ROj6toFp+tfedBYnOV5rXr1SY0KXIkydIALU40yzRbriYCZraBEzhJizhqs3
fUwot/IxWLyiJmaCIo4LxiuNs6wJlPn8EqRXEo5mMetka1QBUCTCV4GJAEHC4xjTJu24FAA3ICF4WRom4LhMS57XYglpdHoJ4BUA
BRkcG6ZBB0SsS+00OXJemRXs496wwpxuH8t8I33dnvN9tDg8CaMgcVGoDgOOIQZFTA7v55fsTyJ8/XIiAXCNaBDBy3qe+3zm+fZU
tM6sfU4zsjUv23Pu892XcgLYK8D6dwEJeHo5MSpEgE2YgjUGCNU+fK0rPV3yl4WMEnjmlIjD6F6OzpMX15VlApBX6GRnDtlEqjUE
rQEzldrxFlov0raO0xsnpWqVARKRTvfmcz/7DEC6MNRc5ebXr2rZkEUMdO5shqoLsw4RY816Zg52MeAuhoI1JiPNSwatZKCXWJmV
1zdtCRhQRNxLwrLUgnO4zPxAn+M0oi3T7lCQEAeyyJgBcHlnw8d8S9weF0fP55jlg2ewlzNe9gYa+2y7Jm2R5zZGBVj/JoHvb9Iu
5jUJAOTrk5OudeGnO97qRWQCZquqXD/nZu/Is7w7UlaPE6+N/pGVXAGj+Gwzu15Z1U+/e6iTsg94lIHI0cFDaIYDMI4GCv1ous6Z
MaO908UIjY8B2X7zXrhn0PplU9bg6xtths/mErG+PeXoUMQ0+9Mo5UmEgiA5Ht5dhlnA0ULMlmf/aVAp4yiDbtvsVmzT2/JlVAIQ
+97f9l0jsOvFml4Fjr/k/stbti2/QgQHWAM9n08EL22WAEUi+mGz0ctJdSICybi+x/s/M7HVWmw9AAA=
""",
"submission_paket.csv": """
H4sIAAAAAAACA23dzY4kyXFF4b2ehSDSrv+Y+1J7rfQCwggoCZTIAQFKC729Jqe7QDDj62Ugs6utPCKPXXPP0//xpz9//frLX77+
8Odf/v3rz//0P19/+59/e/32p/74X3/9zz/8819++fXv14Jr4/dr//q/f/vlv/9+ceKF67drX58Xt97dunjwV168sF7PF1b98a+/
fl5DMaViCsXU0gs3Xti4hlJKpQSlBOsSlJKBa6gkqiRalmhZcp6/2dznjxkoZRT+woFahpZloJihYoaKGY27cWBhhhZmopqJhZko
ZqqYOXVx4d0qZvZzEebRC1XMej3fvVDMQjFLxSwVs7Q0C0/MwhOzVMzCbbZfeOHWfbaji3hotqrZWJqtpdl6aDYemo1qGrdZq5jG
0jRqaTwzjVIa69JYl8YD0yjkoJCDW+ygjoM6Duo4usEOCjko5OgGO6jk6ga7WpOLWu54LvxFLVe1XNRyUctVLRdPfr2ey1KvJyoL
3C9xv8D9ei1c23pz44UH/xqVAupXFa6hlJ/U/5df/vTrr//3y98vT/0cVFOqBuCvOnrh8y4rgL+Cm6wSXRx4N1YmqEXkr6CWHFxD
KQMPTIn8BfKXyF9DCzNQzNi4hlrA/RL3C9yvqVoA/ppYlollAfZL2K+JUiZKEfVroRRQv5busIVSRP1aqAXQL0C/Fkr5jfmPz4ON
SjYqEfELxK+NRQHwa6OQjUK2HvutRWk9LI1aAPxqPSutVfkN+Y/fYusO69ZFfCYD+nVUzE/qf37WAvwF8BfAXwdLA+7XUTEHNxm4
Xxc32cXCXN1kF5X8hv3Hb1HYL2C/gP0S9us+S8kL6xLE/fzE/sda5TXw0qm/c+GFG9ca1w6uoRhwP+B+wP0UCkHYD5gfMT/Vunh0
8fJXC+4HgT/CfjJ0UQujzJ9gZRT5ExUUfJwFoT9Cf4D+DKwOMn8A/gD8GaplqJbxhEymHhpE/oD8AfkD8gfkz2/kf/5jVIoCf4T+
IPAH6I/QH6A/a+IvRC1LD81SMWB/kPcD9kdxPxvLAvYH7M/W4wL4B/CP4B+E/Yj9AfvTWhbBP4J/WtWI/kHiT6uc1l2GzB9k/hyV
A/bnqBql/hxVg9gfxf4A/1HsD/gfpP4A/0Hqz8UjA/oH9I/oH4X+gdA/QP+B0D+A/iH0D6B/KPQPsH+8jl74XJUB+A+M+gfgPwD/
AfgPTfoHJv0DgX9g0j+Q9we4P8D9gUH/EPYH4v4Q9QeoP0T9gbw/kPcHmD/E/DGiixzFDCX+oWH/0LB/CP1D6B9K/QOpfwD9Y6qi
+Rz5DU37B+A/FPsHYv+YBz8Fq4PUP5ZWB8P+oWH/WLjTwP6B1D+Q+gfIP5ZWBegfQv8A+sdWKWD/EPuHJv3jJ/w/b13xfyj8DzUA
Q+P+oQ5gYN4/MO8fmPcP4X8A/0P4H8L/UPofR9X81gA87l3wfyD7D+F/CP9D6X+I/wP8H+L/AP8H+D+ubjY0AAMNwLgqBh3AUAcw
kP+n8v9EBzDRAUx0ABNT/4kGYKoBmGgAphqAqbH/RAcwEf9nRW9GLRr6TwwAJlqAiRZgogWY2uyf0bKgCZhoAmaelJnoAaZ6gIke
YGLkP9ECTLQAEy3AHChEHcBE7J/C/xT+p/A/hf+Jmf8cKAbwnxr5TwT/ieA/Bf85VYzoP5X9p7L/BP8n+D8R/SfwP5H8p4b+U1v9
E/yf4P8E/6f4P8H/Kf5P8H8i+k/gfwr/U/ifyP5T7J/I/hMb/RPRfwr8E+CfAP9U8J8i/wT5J7b6J6b+E7F/Cvvz99z/9fnKoVfq
JsPUf2LqP8X9Ke7Po7sMc/8J7k9wf4r7E9yf4P68z7nfvKpF3J/g/kLwX8D+euEmW+D+AvfXa+nNGy9sXDu4hkoKqFxVuqhawP2F
6L8U/Re4v8D9pbH/QvZfP7n/kRuWtvuX5v4L2/0rz4nsEvuX9vuXBgBLY/8F/C/gfwH/C0P/BfovDP2X4L8E/zWeZ+OWov8C/Jem
/kv0X6D/Av0Xxv4LyX9N3GZC/0LyX5O7S2upGtB/gf4L9F+i/xL9lyb/C/hfwP8S/hfwv7Drv0D/pfS/gP+FXf8l+i/Qf4n+a3tp
lP2Xsv/Czv9CB7A0+1/I/gtn/RYagIUGYPVzp2wp+C8M/hf2/Jf4vxD8l4L/Av8Xxv5L+F+I/Qv0X6D/Av0Xpv4L8F/a9F9K/Uv4
XxfLAvpvpf79Kl0M3j1wberNCy/cuNZ68/OzbCv2b8T+Lf5vTP63Dvlv8H+L/xv83+D/Fv83+L+V+zdy/0bu32D/Ru7fOOq3hf6N
3L+R+zfAvwH+rdH/VvDfOuq3wf4t9m+xf+Os30bu35r6bwT/jan/1tR/A/0b6N/K/Vu5f2PqvxX7t7b8N3L/1tx/r+cwdgP9+3f0
f31exF2G3L8B/g3wb4F/64j/VvDfOvC3kfy3jvhvRf+9n6ls64j/Fvw3ov9G9N8C/wb4t3b9N7L/Vvbfyv67n/3y/h39n4utqf9G
+t9g/8bMf4v9G+zfYv/W0H8D/hsn/rai/77PYzJbJ/036L+V/Tfwv5H9Nzb9N0b++4KX2vNvwb+R/VvZvwH/RvZvZf/W0L9fz5us
NfRv0b8V/1v4b8X/Rvxvjf0bY/8G/hv4b+G/gf8W/lvhvxX+Gw1AowFohf/O89Os0QC0on+jA2gd9m+1AI3w3+oAGh1A48hfqwFo
Tf4bHUAj/DcagFb2b2T/RgPQaAAa2b+R/VvZv3HYv4X/Bv4bwb8R/Bv0b2z6N+Dfiv0N+jfo3zjv14J/I/a32N/K/S34t+DfSv6N
5N8b1Tj4t4J/d/m1qkktQCv8t3qAVg/QOvbf2vpvNQGNJqC1BdDoAho7/40moNUEtJqAPoCNJgCtLqAxAmiMAFoH/xszgEYT0JoB
tGYAjTagtQPQ6gMOtgAO2oCDEcDRF/6OZgAHM4CDGcDRDOBgD+BgD+BgBHCw83/UAhy0AEctwNEI4Ojs/0ETcNQEHDQBR03AURNw
MAQ4OgF4sANw1AQc7P4fbQAcdQFH5/4PBgEHOwAHTcBRE3DQBBw0AQfn/g+GAActwNEQ4GgD4GAKcDQFOPqq/5lYGHQBB13AwRDg
oAk42Po/6AGORgBHx/4PuoCDLuBo+n8wAzg69n/0Pf+DPuBg+n/QBhy1AUdtwFEbcDD/Pzj4fzQCOOgCjg7+HYwAjkYAByOAI/4f
jAAOZv9H9D+i/8H0/2D7/2j6fwT/g/H/AfwP4H+w+X8w/D/I/+f8HGZ8/Gu0MGD/AfsPBgAH6D9A/7nP72Iekf+A/AfD/yPwXw0A
rqb/F+i/GABckf9qAnCB/iv0X6D/agBwwf4L9l+x/2r8fzH+v2L/FftvPTvMK/bf39n/MUi7gv8V/K8mAFf0vxgBXOwBXMD/YgJw
sQVwhf6rCcDVBOBqE+CK/hf0v6D/1QjgagRwwf8r/l/MAC7wf4X/O/llhosxwEUDcLUNcDUHuGgBruYAFz3AxRzgYg5wdfr/ogW4
GATc9Zw0XXUAVx3ARQdw0QFc7QJctQD39xbg80H82QJ8rpdmARddwFUXcNEFXOwDXA0CLs4AXk0BLk4AXDQBF03AVRNw1QRcNAEX
TcBVE3CxDXDRA1xtA1w0AVcTgIsu4KILuNgFuOoBrk7+X20DXE0ALtqAizbgagJw1Qdc9AEXGwH3eQigXs/8XxD91QvbAPVt+vvH
x6NeaATq9RwB1OvZB9TreQqwXs9TACXTX8H0V69CNaVq0AbUq1RLoRa0AfXC1//rhT6gXoW1CaoJqnlOAOoVFfPsAUq2v5Ltr17B
ygQrE63MQC0DtQzUMlTLQC1jPf81A/fYQCXPBqBeA4uC/F8w/RVMf/V67gGURH/1Lfr7fLImipko5if+H2/XyiysDKYABdtfyfZX
r4WlWXhklu6yhXKWnpiFxdkv1r3reV9sVLOxPBvFbBSzsTRbj/9WMRvFtO601so0amnU0rrV2rdaa3Eai9P4CGjdaAc32lE5R5/O
B/Wc+fzRB2tzNl6HUo6W5mBpLkq5xd/jVTEXxVwtztWn89XSXN1rFx9q6AKg/atCFwDtX9VzFFCw/lVhElDS/hW0f1XPSUBJ+1fQ
/hW0f1VqAapQS6EWNABy/lWpAZD0r6pUDL8JUBL/lcR/VWoDCgagKvUB9RwGFNR/BfVfQf1XhS5A5r+qoWIGVmeolqFahtYHnUAN
Lc/Q8kyUM5+sqalqppZGzQD0fyX9X0H/VzVVjRqBwpcBCgLAKjQCpUZAAsCqpcVZuNGWnh21AnAAFhyABQdgFfoAKADrrQB8/gws
zPM8YJX6gNp4ZtAGyABYpTYACsCSArCkAKxqFKMmQArAqkY16gK+HYCfH2lqBOAALDgACw7AqqPb7OA2QyMAB2DVUTVoBOqqlItS
0AXURSkXK6MeABLAKrQAcACWHIAV7AdU0AQERwFKCsCCArCiLgAOwIIDsOAArJSKQRcQdQGwABYsgBVsB1SexwFLFsBKoRY1AVAA
VtQByAFYcgCWHIAlB2BFk4CgBQhaACkACwrACiYBMABWNAmIWgA4AAsOwAoaACkAK5gFBPyHAbDeBsBPJsAAWBH9oQCsYA6Qp/K/
ZACsTFSCIUA0BJABsCL4QwFYUABWgH4ZAAsGwMrSHQb0SwFYUABWtmrBEEAOwMrW47Jxj2kKAAlgpVGM6P+WAD5+36J/GsUA/m8D
4PMvRClAv/x/Bf9fwf9X8v9VjkoB+IMBgPR/Bf1fSf9XEfnl/6sI/RAAFgSABQFgQQBYEAAWBIAVkD8gP/R/Bf1fQf9XQ9wfSP/Q
/9XA+H/gGEBJ/1dD6R/+vxr4HkBBAFhDGwBDGwBSABYUgDVE/qH8P4T+gYMABQ1gQQNY4/lFgBoCvzSABQ1gDXAfFsAa4j40gDXA
/QHuDyT/gQ2AAerL/1cD2P/W/30kEwkAa2gXAAbAGtgFGEr+PwyAX5+vRD1K/mPqPhP94QAsOADr7QD8/PQfgr8cgDWU/IeSPyyA
NYT/oeQvD2C9PYCPfzqiPzSAJQ1gja1qEP6lAawB/MsBWAP4lwKwpACsgfQPA2DBAFgwANZoLIyi/wD/vwWAn88SOoCBDmAg+EMA
WBAAlgSAJQFgDTQAb//f497BDsBA8B+i/9D8X/6/GtoAgACwBhoA+P9qoAGA/q8mGgDY/2rqDAD0fzUV/OH/K/n/aqoFkACwJqL/
VPSHALAkAKypFmAi+78FgJ/3CQSABQFgTfF/4gDARPKH/6/g/6up3P9DAPj1+cqlV6KWaGHA/yn+z6GFQQMw0QDAAVgTDcBEA/Bt
APz4LJpoAaaSPxyANcX/idE/HIAFB2DJAVhyAJYcgCUHYMkBWFOj//n0/9dU/IcFsKYagIkzALIAFiyANcV/aABr4gwALIAlC2DN
rQ8A8R8ewJIHsCYOAUADWNIAFjSABQ1gQQNYE/CHBbCmJv/SAJY0gAUNYEEDWFPxHx7Akgew5vNLgDVBf1kACxbAmqL/BP1lAawJ
/M+LB0b4n8K/NIA1hX94AGsC//IA1gT/oQEsaABrafIvD2AtjQCWOoCFDgAiwIIIsH6IAL8+L6IYTAAW8C8PYMEDWG8P4OdSL+z/
QwNY0ADWwtz/bQF8/AzAfwH+MgDWEv0Xjv9JAVgL8H8bAJ//RJQi9i+xf2njf2njHw7AggOw5ACshan/AvulACwoAEsKwFrI/kvb
/lAAFhSABQVgLaF/YdN/TSwMFYC1kP1hAKyl7A8FYC2Qf2HuvwB++P9qKfcvgB/+v4L/r+D/K/n/Cv6/gv+vFqi/tOW/sOW/tp4X
YF/uv4L7r5aG/pL/FeR/BflfLcV+2P9q9cWnNLgP+18tjf2l/6sl8Mv/V/D/1TqqRuRfGvwvDf6X2L/EfkgACxLAWiD/AvkXgj8M
gAUDYMkAWPv53/0WBIC1xX0ZAGvr5B8UgCUFYG0lfzkAayv6SwJYkADW5v/5W9AA1kb4f1sAn9dUD/i/NfzfPvwHEWBBBFhbLQBM
gAUTYO2gGkz/IQIsiABrqwGQCbA20r9EgLW17b+R/yUCrD20NPgOwEYHABFgQQRYEgEWRIAFEWBJBFgbG//fHsDP+0Tpf2P6DxNg
bXQAWx0ATIC1NfyHCbC2vgAAFWBt9QAbPYBUgCUVYEkFWFttgFyAtdEHbPQBUgHWRh8AE2DBBFgyARZMgCUTYG21ARttwEYbsNUG
7NbSNIpBGwARYG3M/t8iwMfHEWb/G+FfGsCCBrCgASxpAGtj9i8NYG01APIA1lb4hwiwIAKsbxHg5wcAuoCNLqAR/2ECrEYTABFg
tXoAmQCr0QO0eoDGub9G+G+Ef2kAq9EAwAJYDfpDAliQAFaL/rIAFiyA1Yj/DfQ30N9Cf2vrv8H+1pE/SACrAf9W/JcEsBqj/1b6
b6T/FvwbX/+TBLAa8b8V/yEBLEgAqwH/FvwhAazW6L9F/1b+b5z7gwawGuiHBbBgASxYAOttAXz8XGR/WABLFsBqTf1b5G/t+jfi
f4v8EgGWRIAlEWA1yN8g/1sD+Pj9APwN8DfA3wJ/A/yNuX9j07+B/Qb2u1GIqC/1X/XRs6+xfyv/Q/5XkP9Vg/yN8A/1XzXAL/Nf
wfxX7a/9NQ79tcgP+V81wC/3X7XG/pL/VQP8R/Ef8r86mvvL/ley/9UR+6H/K+n/6uDL/9D/FfR/Bf1fHaD/aNv/gP0H7D9gP9x/
dRD8of4rqf/qAP1Hk3+p/+qI/XL/Fdx/JfdfHcBf6r86gj/cfwX3Xx2wH+q/gvqvjtB/lPsh/yvJ/0ryvzowANfBvv8B/OH+K7j/
6gj9R7v+sP/V0ab/0ff9Dugv/V9B/1dHwV/+vzpqAA6S/7f/7+PjFQbAggGwDr75LwNgHXzlTwLAOtr3hwGwZAAsGABLBsA6Ovd3
sPN/WtUo+0MBWFAAlhSAdbDzDwNgHfUAB9H/IPpDAFhHGwAwABYMgHWU/aEArIMW4GDf/2j4f5T9YQAsGAALBsA6+ta/FIAFBWBB
AVgH+L/I/RIA1hX+YQCsi4P/EgDW1bm/i23/q+G/DIAFA2BdRf8L/ssAWDAA1gX/L/h/wf8L/l8N/qX/q4sG4CL7X5z6h/uvrugP
+V9B/leQ/9XF2F/uv5L7ry7gfwF/mP/qatv/Ytv/IvdD/FcXsf8K/RD/1cWX/a5O/MP6V7D+lax/JetfXWt/LpI/vH8F71/B+1fy
/tUV+9/iv8fvQuiX+a9k/iuY/wrmv5L5ry62/i/If0V+Wf8K1r+6OPH3lv49fhMC/0X0h/SvJP0rSP8K0r+C9K8k/asL7F9l/yvu
y/pXsP4VrH91kfwh/StJ/+oK/BfZ/2rf/4L8sv6VrH91QX5I/+ot/Xv8cy4eGIz8ofyrq9x/4f4NpH+B9C+v58Q/ryf383pyP/D9
Bb6/vJD5A+FfJPwLhH95gfp5FUoplIKv++WFr/tFwr+8ntyPfH95laoB+APjX97Gv8fvB8E/MP7lFaxMsDLRyjzJn9eT/IHvL6+h
hRlYmKFSBkoZWpihhRlYmKGFGShmopipYiaKmbjLJmqZWJaJZeF2f14TCzNRy9I9tnCPIfXntVDMQjFLC7O0MAsLs1DL0gOzsTAb
tWzVsnWXbRSzsTIbK7P1wGzUsrEujVIapbRK6cF7olFMa2VaK9Mqp1FOo5yj2+x39H99vhLPzMFtdlDMwcocrMzBTXb09B+UclXK
1eN/UcpFKVeQeaI/r4tarpblqhixH6q/FFJ/4PpLYeafwsw/sv1Ftr/A9hfY/iLbX2D7S4n/Bf7D9hfY/gLbX+oZ+iPZX6pQiuBf
gr9Mf5HpLxUUA/rL8xd4/lLCP0R/qaga8L+eQ/8U8F/C/1v09/l4/PD8fX1eRC2DzJTpL4UGoKbWRh1AoQOQ6S+FFqCm1gY9QE2t
zVQ1aAEk+kstVaMeoNADyPQXmf4C018KPUChByj1ALVVDZqAH6a/z7tloxj0AFD9pbY+zjZq2VqZrWLQBRS6gFIXINdf5PpLqQmQ
7C/VKAc9QKEHqINiDopBBwDNX+qoFLQAPzR/nyt99MwclIIeAJ6/1EUp6ADqalnUAtTVslwti5qAUhMA11+CHkCqv0D1l2AAEAwA
IPqLRH+B6C/B3D8w/UWmv0QdgFR/CQb/gesvQQ8A1V+k+gtUf0mhmGBd1AEkWBc0AHn+l38J4n/Af0j+Aslffkj+Ph42SP4CyV8g
+UsGCsF5v8DxFzj+kqE1GSgF6I/QH6E/QH+E/kysCtI/LH+JyB9kf1j+AstfguifhUqWKhH2IfkLJH+R5C9ZevAXahH2A+xH2T/A
PiR/keQveSp+A8df5PgLHH9JqxZgP42FEfXh+Ascf4mYHwX/tIoB9WH5S46eF2A/wD4kf8nRuiD5R9SX5S+y/CXgviR/geQvEfgj
8EPzF2j+As1foPnLW/P3eYdC8xdo/jKw258B6A/l/qHcD9FfJPqLRH8ZwD48fxmFYgT9gdw/wPwB5kPyF0n+MsB8Of4ix1/g+Asc
f4HjL3L8ZYD634q/j1QMy19g+cvA2H8MFaO5PzR/GQr+Q+iH6C8S/QWiv7xFf4+nY2hpxP4h9kPzl4HB/0Dqh+QvA+gfE6WA/G/D
36M8Rf6hyP92/D3frXUR+wfYD8NfBhK/BH+R4C9D7B9b1WysC9gvw19g+IsMfxka/Evxl6HQL8df5PgLHH95O/4eCwb8D2X+If7/
sPx9fb5d5YD/cPxliP+Q/AWSv0Dyl4HUL8df4PjLEP4h+cvQ5H8g9g/QX46/jItalPqHUv9Q6oflL0OT/4nj/pmK/ROxf2r0P5H7
4fnLRAMgzV+k+Qs0f3lr/j7v+qnYPzH4l+Uv35a/DxrOWr6MirD5P9EGzKftJ1NdAFx/gesvcv1lqg2Q6y9w/WWiCYDqLxNNwET4
h+gv36K/z18t8j9cf4HrLxPxfyr+T8R/iP4i0V/mRDH4ul+m4v+36u+zbLQBE23ARBsw0QZMDADg+cvEAACav0yM/WH5y0QLAMlf
JloAOP4Cx1/g+MtE9p/Avwx/geEvE/iH4C8Q/EWCv0Dwlyn2w/AXGP4Cw18m0D+R/GejEgV/+P0ywf2pHf+Jcf8E96H3ywT3YffL
ROyH3C8Ts364/TJFfcn9MoF9uf0it1+muD/Ffdn9MjXth94vS9yX3y9vv9/nx8NC8pfdL7D7BXa/wO6XJezD7pcfdr+vz4uFF+Lj
GHq/LB34g98v8PtlKfkvnfdbiv5L0IfjLwvQX4j+MPxlIflD8Jel/f5vw9/HRSyMgv8C9CX4y1Lwh+EvC8yH4C8Q/GUB+W+/3+OO
x2k/6P0ivV/eer/nX4hCQPsF2sPuF9j9YrtfYPcL7H5ZAP5S6ofdL0v7/EuH/eD3i/x+gd8vS6kfgr8sTfwXUj8Ef1ma+MPwFxn+
ssR9Kf6yBP4F8C9t9cvxlwX0L6V+OP6ysNG/nv+1bxbQv4R+KP6ygP4F9EvwFwn+AsFfFuC/AP+33u9RHgb+kvtFcr/8kPt94knw
h94v0PtlCf1L6IfgLxsz/y3yQ/AXCf6ykfg3yA+9XzbIv3HST3K/SO6XDfK/3X6fq7oBfoj9ssF9iP0isV8g9stG1N+i/tY2P7R+
2cA+rH7ZSvob3IfWL99avw+abJ3020j7Mvtli/zfar/PnwT2w+2XDfZL7Reo/bI1898K/JD7ZWPkv5X3IfeL5H7Z+KJfNhqAjQYA
Zr9sTf2h9ssG/rfwL7VfoPaL1H7Z2vF/u/0ej8j6eabkH1+pBkByv2zk/o0GQGq/bHUAcPtlI/hD7Zct/sPtF7n9Ardfts/7y+4X
2P0Cu182GgDJ/QK5X95yv8ciaui/jz4E0AJA75dvvd9n2egBIPjL1txfhr/A8BcZ/rIxAZDgLxL8BYK/bA0AtroA+P0Cv1/g90s/
Jb+B3y/w++Xt93u+d+F1qER6v7TiP/x+aTUBEvyllf9bc38o/tL6wh8cf5HjL41GAIq/tBqBViMAyV8k+Yskf4HkL5L8BZK/QPKX
xgCg0Qa02gA5/tIaADSO/DeagOZ/8JNWH9DoA2D5Cyx/ae38S/MXaP4izV8aY4DGGKDRBUDyF0j+AslfWlv/sPylMfWH5S+w/KUx
9W9MABpT/8YAAIa/yPCXBv4l+Mu34O/zzlEHAMdfGh0AHH+B4y9w/KWR/6H4Syv+w/GXVvxvbfo38C/LX2T5iyx/aUwAGhMAOP4C
x1/g+Ascf2md+WvBX5a/wPKXxqY/FH9pfNuvLz7GQH4Z/tI48dcAf2vH/4D8P/x+X58XgxcOvXDqxyxd3LrYunjww58PzBH8YfgL
DH85mAAcjf5h+AsMfzki/9HoH46/yPEXOP4ix1/k+MvBEOBb8ffx+QbJXyT5CyR/OYD/Afyh+AsUf4HiL1D8BYq/HMT/g9H/0Rf9
ZPjLAffP8//1C/x+OUr/x7v9cPzlAPxS/AWKvxyQX4a/wPCXA/If5X8Z/iLDXw7gD8Ffjsb/R6f+juL/Ef8P+A/BXw7m/2+/32Ox
gf4D9MPulwP0H6D/KPsfsP9g0/9b7fd5kyH6H0T/o33/I/hL75cj+B+lfwj+IsFfIPgLBH+R4C8Q/AWCv0DwlwP4H8Afer9A75ej
3C+/X+T3C/x+gd8vR/i/2viX4S8y/AWGv8Dwl4vs/0Pw9/V5sfVTDt79XBj4/SK/X+D3i/x+gd8vV8Efgr9A8Jcr+l/Q/+J7fvD7
5Qr+EPzl4nt+F6n/KvVD8JerfX8Y/vJt+Pv45Pl2/D0uY20U/aH5y0UDAMtfrnI/NH+B5i/S/OUq+MPzF3n+chX8IfrLBf8vgv8F
/i8O/V/h/yr5w/KXi+QvyV+u8H+B/wv8y/EXOP4Cx18uNv+h+MtV9r+A/8WRv6sT/1eb/xf4l+MvV3v/Vw3ARQNwFf6vO4C36O9x
W6ADuOoALk7+yfOXiyP/V/y/mv5D9BeI/iLRXy7y/1X+l+gvV/lfpr/I9JerMwBX43+4/nLVBVx1AbD95aILkOxvQPY3XmgCBmx/
A7a/8cIMYED3N6D7G9L9jdezBxhv3d/Hb2zA9jdezwHAgOxvQPY3Xs8OYLwKhUD1N6D6G69CIc8GYLyCQoJC0ACMV1BJtCZBKcGa
BJXwBMCA6m+8cPRvyPU34PobLx4AGK+hgqD7G6+BigYqwgxgwPY3XlP1TNUzUQ9agAHf3/j2/X0WPnWvTT01Ezfb1CfAwt22cLct
lLNwsy2tzdLaLKzNPxwC+H+iAQZ7a/QAAA==
""",
}

for rel, b64 in DATA.items():
    text = gzip.decompress(base64.b64decode("".join(b64.split()))).decode()
    path = os.path.join(ROOT, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(text.replace(ORIG_ROOT, ROOT))
    print(f"{rel:45s} {text.count(chr(10)) - 1:6d} baris")

## 2. EDA / audit data (`runs/eda`)

Urutan: `scan.py` (md5, metadata, phash/dhash semua gambar, CPU) -> **[jembatan]** `meta.csv` ke `meta.pkl`
-> `dups.py` (duplikat & label konflik) -> `augscore.py` (proxy noise/rotasi) -> `sheets.py` (kontak sheet PNG).
`runs/eda/meta.csv` juga dipakai `emb-r2/train.py` dan `ensemble.py` untuk menghitung "val bersih".

<details><summary><b>REPORT.md asli: <code>runs/eda/REPORT.md</code></b> (klik untuk buka)</summary>

#### EDA audit — agen `eda` (CPU only, data tidak diubah)

Semua angka di bawah berasal dari eksekusi `scan.py`, `dups.py`, `sheets.py`, `augscore.py` di folder ini (4 Okt 2026).
File turunan: `meta.csv` (metadata + md5 + phash/dhash semua 15.281 file), `test_train_matches.csv`,
`train_cross_label_pairs.csv`, `test_pseudo_labels_strong.csv`, `test_pseudo_labels_strict.csv`, `augscore.csv`, dan 16 kontak sheet `sheet_*.png`.

##### 1. Integritas file, mode, ukuran

**File korup: 0.** Semua 12.224 train + 3.057 test terbuka dan ter-decode penuh (`im.load()`), tidak ada yang truncated.
Yang perlu diwaspadai bukan korup, tapi **ekstensi bohong**: 793 Aman + 578 Rusak + 597 test ber-ekstensi `.jpg` padahal isinya WEBP; 17 Aman + 18 test `.jpg` isinya PNG; 1 Aman (`.jpg`) isinya **AVIF**. PIL 12 membukanya (AVIF ikut terbuka), tapi loader lain (cv2.imread, torchvision.io.decode_jpeg, ultralytics cache) bisa gagal diam-diam. Pakai PIL + `convert("RGB")`.

| grup | n | RGB | RGBA | P | JPEG | WEBP | PNG | AVIF |
|---|---|---|---|---|---|---|---|---|
| Aman | 8800 | 8776 | 18 | 6 | 7915 | 793 | 91 | 1 |
| Rusak | 3001 | 2997 | 3 | 1 | 2419 | 578 | 4 | 0 |
| Lainnya | 423 | 423 | 0 | 0 | 423 | 0 | 0 | 0 |
| TEST | 3057 | 2953 | 101 | 3 | 2315 | 597 | 145 | 0 |

Alpha: dari 102 test RGBA/P-transparan hanya 12 yang alpha-nya benar-benar < 255; sisanya alpha penuh (convert RGB aman). 

Ukuran (median w×h): Aman 512×512, Rusak 640×640, **Lainnya 200×200 (95% ≤ 512px, 67% sisi terpanjang ≤ 256)**, Test 512×450.
Orientasi: Aman 66% persegi, Rusak 63% persegi, **Lainnya hanya 17% persegi** (40% landscape, 43% portrait), Test 47% persegi.
Lainnya = 100% JPEG, median 12 KB; 276 dari 423 (65%) punya salah satu sisi tepat 200 px (thumbnail gaya OpenFoodFacts). Di test hanya 54 file punya sisi 200 px, 9 punya sisi 400 px.

**Test berbeda distribusinya dari train** (persen per grup):

| sisi terpanjang | ≤256 | 256-512 | 512-768 | 768-1024 | 1024-1536 | 1536-2048 | >2048 |
|---|---|---|---|---|---|---|---|
| Aman | 10.6 | 44.4 | 30.2 | 7.9 | 5.1 | 0.9 | 1.0 |
| Rusak | 4.9 | 32.6 | 47.0 | 15.1 | 0.4 | 0.1 | 0.1 |
| Lainnya | 67.1 | 29.8 | 3.1 | 0 | 0 | 0 | 0 |
| TEST | 21.7 | 27.8 | 19.6 | 17.6 | 6.6 | 1.3 | **5.5** |

Val (split resmi) hanya punya 27 gambar sisi ≥1500 px; test punya 240. Val tidak mewakili porsi gambar besar di test.

Resolusi kunci (jumlah train per label vs test):

| res | Aman | Rusak | Lainnya | TEST | % test | isi (dilihat di sheet) |
|---|---|---|---|---|---|---|
| 224×224 | 253 | 0 | 0 | **520** | 17.0 | kardus stock-photo + salt-pepper noise berat, kotak kecil di latar abu (`sheet_test_224x224.png`, `sheet_train_224x224.png`) |
| 640×640 | 1478 | 1305 | 2 | 347 | 11.4 | stock photo alamy/shutterstock, rusak & utuh campur |
| 675×900 | 108 | 391 | 0 | 280 | 9.2 | foto HP paket berlabel GHTK Express (Vietnam) + stempel timestamp "02-15-2025" |
| 416×416 | 87 | 211 | 0 | 193 | 6.3 | render 3D conveyor abu + kotak "Isacyt" label hijau |
| 3000×4000 / 4000×3000 | 20 | 0 | 0 | **92** | 3.0 | foto HP paket plastik oranye (flyer Shopee) di lantai keramik / meja oranye / kain biru (`sheet_test_3000x4000.png`, `sheet_train_3000x4000.png`) |
| 2816×1536 | 20 | 0 | 0 | **40** (39 RGBA PNG 5–8 MB) | 1.3 | **render AI**: kotak mulus di studio/lantai, label "SHIP TO: JANE DOE 123 MAIN ST" (`sheet_test_2816x1536.png`; contoh `test_00093.png`, `test_00666.png`, `test_03009.png`) |
| 1536×1024 / 1792×1024 / 2390×1792 | 34 | 0 | 0 | 57 | 1.9 | **render AI** kotak mulus, bubble-wrap + label FRAGILE, 1 iklan "CETAK KARDUS ARSIP" (`test_01638.jpg`) (`sheet_test_1536x1024_etc.png`) |
| 150×200 / 200×150 | 4 | 0 | 53 | 14 | 0.5 | thumbnail produk makanan Eropa |

Kesimpulan: di train, **semua** gambar ≥1500 px dan semua 224×224 berlabel Aman (0 Rusak, 0 Lainnya). Test punya 2-5× lebih banyak gambar di kelompok itu. Secara visual isinya memang kotak utuh (render AI / paket oranye utuh), jadi label Aman masuk akal, tapi model tidak pernah melihat Rusak pada domain ini; kalau ada paket oranye yang penyok di test, model tidak akan tahu.

##### 2. Duplikat

###### Eksak (md5) di dalam train
- 607 grup md5 identik, 1.260 file terlibat, **653 file redundan** (Rusak 738 file terlibat, Aman 522). Ukuran grup: 579×2, 17×3, 7×4, 3×5, 1×8.
- **26 grup dengan label konflik, semuanya Aman↔Rusak** (byte-identik, dua label). Daftar lengkap ada di output `dups.py`; contoh: `Aman_07821.jpg`=`Rusak_02372.jpg`, `Aman_06056.jpg`=`Rusak_01561.jpg` (infografik popcorn), `Aman_07807.jpg`=`Rusak_02101.jpg`, `Aman_06795.jpg`=`Rusak_01900.jpg` (iklan Tokodus 36×24×21). Pola nama: `Aman_078xx` ↔ `Rusak_019xx–024xx` = blok yang sama disalin ke dua kelas.
- 159 grup md5 identik menyeberang split train/val resmi → **164 file val punya kembaran byte-identik di fold train**. Val F1 akan terinflasi.

###### Near-duplicate (phash 64-bit, Hamming ≤ 6, verifikasi visual di `sheet_test_train_neardup_examples.png`)
- Di dalam train: 3.425 pasangan (730 eksak + 2.695 near; jarak 0: 1.306, 2: 622, 4: 382, 6: 385). Near-dup umumnya = gambar sama di resolusi beda, versi grayscale, atau versi ber-noise/rotasi. Setelah merge komponen: **9.954 gambar unik dari 12.224** (18.6% redundan). Klaster terbesar 14 gambar (conveyor scene, 10 Rusak + 4 Aman).
- **203 pasangan lintas label** (`train_cross_label_pairs.csv`): 193 Aman↔Rusak, 10 Aman↔Lainnya. Dilihat di `sheet_train_crosslabel_pairs.png`: hampir semuanya gambar yang sama persis (kotak ebay, kotak makanan kraft, lunch box, iklan kardus, kotak kosong terbuka) — kotak **utuh** yang diberi label Rusak. Ini label noise di Rusak, bukan ambiguitas. Lintas Aman↔Lainnya: `Aman_06277.jpg`=`Lainnya_00403.jpg` (beef jerky & snack), `Aman_05225/05226.jpg`≈`Lainnya_00405.jpg` (mailer box teal, bukan makanan — label Lainnya meragukan), `Aman_07707.jpg`=`Lainnya_00416.jpg` (set kotak suku cadang TruckLite), `Aman_04505/06272`≈`Lainnya_00402` (rak produk pembersih).
- Val: dari 1.833 gambar val, **506 (27.6%) punya tetangga phash ≤ 6 di fold train** (338 dengan jarak 0). Per kelas: Aman 299/1320, Rusak 204/450 (45%!), Lainnya 3/63. Macro F1 val semua agen terinflasi, terutama Rusak.

###### Test ↔ train (kebocoran yang bisa dipakai)
- **253 test byte-identik (md5) dengan train**: 155 Aman, 96 Rusak, 2 konflik Aman+Rusak.
- Jarak phash tetangga terdekat test→train: 0: 512 file, 2: 204, 4: 164, 6: 174, 8: 137, 10: 181, ≥12: 1.685. Jadi **1.054 test (34.5%) punya near-dup ≤ 6**, dan 2.003 (65.5%) jarak ≥ 8 (tidak ada kembaran).
- `test_train_matches.csv` (1.989 pasangan, 1.054 test unik, kolom `test_filename,train_path,label,hash_distance,dhash_distance,exact_md5,test_res,train_res,train_fold`). Label konsisten untuk 974 test (Aman 616, Rusak 356, Lainnya 2); **77 test cocok ke Aman DAN Rusak**, 3 ke Aman+Lainnya.
- Konflik itu sebagian besar scene conveyor 416×416 (`sheet_test_conflicting_matches.png`): latar sama, kotak sedikit beda posisi → phash tidak membedakan. Untuk conveyor, phash ≤ 6 **tidak** boleh dipakai sebagai pseudo-label.
- Set pseudo-label yang saya anggap aman: `test_pseudo_labels_strong.csv` = **467 test** (Aman 284, Rusak 183): md5 identik ATAU (phash 0 dan dhash ≤ 4), tanpa tetangga ≤ 6 yang berlabel lain. Set longgar `test_pseudo_labels_strict.csv` (phash ≤ 4 konsisten) = 816 (Aman 535, Rusak 279, Lainnya 2) — pakai hanya jika mau ambil risiko.
- Di dalam test sendiri: 121 grup md5 identik (285 file) → prediksi harus konsisten antar kembaran (bisa dirata-rata).

##### 3. Kontak sheet dan apa yang terlihat (dilihat langsung)

- `sheet_train_Aman.png` (48 acak): kardus coklat stock-photo/listing e-commerce (IKEA, "Pack of 50", Tokopedia-style banner "Tambahan packing kardus"), kotak di latar putih, pallet. **Augmentasi berat terlihat**: salt-pepper noise, rotasi dengan sudut hitam, mirror (teks terbalik "Kardus terpotong"), bahkan gambar yang murni noise (`Aman_04694.jpg` 70×53, `Aman_04647.jpg` 53×66) dan kotak 57×46 yang isinya noise (`Aman_03736.jpg`). Ada yang bukan kardus: kurir motor (`Aman_06602.jpg`), lunch box plastik warna-warni (`Aman_04486.jpg` — ini mirip Lainnya), air-column bubble (`Aman_06561.jpg`), amplop kraft (`Aman_08644.jpg`).
- `sheet_train_Rusak.png`: kardus sobek/penyok/basah, vektor/kartun kardus rusak, "BARANG RUSAK ↔ BARANG BARU". Tapi juga banyak yang **tidak tampak rusak**: render conveyor Isacyt (`Rusak_00085/00077/00195/00096/00047`), foto GHTK berlabel yang tampak utuh (`Rusak_00252, 00409, 00581, 00258, 00323, 00267`), pallet di gudang (`Rusak_01412`), kotak Clip Cord Sleeves (`Rusak_02287`), infografik daur ulang (`Rusak_01584` — jelas salah), **wadah makanan kraft + cup takeaway (`Rusak_01361` — ini Lainnya menurut definisi)**, kardus vektor utuh (`Rusak_01764`).
- `sheet_train_Lainnya.png`: produk ritel bermerek Eropa/Prancis (Nestlé Chocapic, Quaker Cruesli, Gerblé, Chabrior, Fjord, Keurig, Starbucks, Beyond Burger, kantong kopi kraft, roti dalam plastik), thumbnail kecil latar putih, sebagian ber-noise (`Lainnya_00389, 00364, 00352`). Hampir tidak ada kardus coklat polos. Ada 2 yang bukan makanan: rak produk pembersih (`Lainnya_00402`), set kotak suku cadang (`Lainnya_00416`).
- `sheet_test_random96.png`: campuran yang sama persis dengan train — conveyor, GHTK, 224 ber-noise, stock photo rusak/utuh, render AI, plus sedikit Lainnya-like: Boursin (`test_00367.jpg` 193×200), beef jerky (`test_01428.jpg` 640×640, = `Aman_06277`/`Lainnya_00403` konflik), dark chocolate (`test_02432.jpg` 114×200), wadah makanan kraft (`test_00757.jpg`). Dari 96 acak ±4 tampak Lainnya → konsisten ~3-4%.
- `sheet_test_far_from_train.png` (48 test tanpa kembaran train): foto nyata — GHTK, paket oranye, orang memegang kotak, kotak sobek, label gizi Arab/Prancis (`test_02844.jpg`, Lainnya). Domain "asli" test ada di sini; inilah yang menentukan skor, bukan yang bocor.

##### 4. Lainnya vs Rusak — ciri pembeda dan jebakan

Pembeda yang kuat (dan yang akan dipelajari model):
- **Warna & teks**: Lainnya = kemasan ritel warna cerah dengan merek/teks besar, latar putih; Rusak = coklat kraft, tekstur sobek/lipatan, label pengiriman/barcode.
- **Ukuran file & resolusi**: Lainnya kecil (median 200 px, 12 KB, JPEG), Rusak 640×640/675×900. Ini shortcut berbahaya: resize ke 224/384 membuat Lainnya selalu tampak blur/upscaled → model bisa belajar "blur = Lainnya".
- **Rasio aspek**: Lainnya 83% non-persegi, Rusak 63% persegi.

Yang berpotensi membingungkan:
1. **Kemasan makanan kraft/coklat** (kotak nasi kraft, paper bag, cup carrier) ada di Aman (`Aman_06243`, `Aman_06279`) dan Rusak (`Rusak_01361`, `Rusak_01373`) padahal definisi Lainnya = food packaging/paper bag. Di test ada `test_00757.jpg` tipe ini. Tidak ada konsistensi label dari penyelenggara di area ini.
2. **Kemasan plastik oranye (Shopee flyer)** 92 test / 20 train: ini "kantong plastik" secara harfiah tapi di train 100% Aman. Ikuti train: Aman.
3. Augmentasi noise/rotasi ada di ketiga kelas tetapi **tidak seimbang**: proporsi gambar dengan sudut hitam (padding rotasi, `black_corner>0.5`): Aman 9.6%, Rusak 3.8%, Lainnya 3.8%, **Test 1.2%**. Rotasi-padding hampir tidak ada di test, jadi augmentasi itu artefak train, bukan sifat test.
4. **Conveyor 416×416**: Aman = versi ber-noise/rotasi, Rusak = render bersih (lihat `sheet_train_416_aman_vs_rusak.png`). Kotaknya sendiri tampak identik-utuh di keduanya. Model akan belajar "conveyor bersih = Rusak". Test 416×416 (193 file) 99% tanpa rotasi → akan diprediksi Rusak. Entah benar atau tidak, train mengatakan begitu.
5. **GHTK 675×900**: pola sama — Aman = versi ber-noise/rotasi (108), Rusak = foto bersih (391), padahal paket bersih itu kebanyakan tampak utuh (`sheet_train_675x900_aman_vs_rusak.png`). Test 280 file, 83% tanpa kembaran, semuanya foto bersih → kemungkinan besar diprediksi Rusak.
6. Lainnya yang ber-noise (`Lainnya_00389, 00364, 00352, 00344`) mirip Aman ber-noise pada ukuran kecil.

##### 5. Rekomendasi konkret untuk training/ensemble

1. **Loader**: PIL `Image.open(...).convert("RGB")`, jangan cv2/decode_jpeg (1.371 WEBP dan 1 AVIF berekstensi .jpg). Untuk RGBA komposit di atas putih/abu, bukan buang alpha (12 test punya alpha transparan nyata).
2. **Resize**: pakai letterbox/`contain` (pad), bukan center-crop, karena 53% test non-persegi dan objek sering kecil di tengah (224×224 noise, kotak kecil di latar abu). Untuk gambar ≥1500 px, downscale dulu dengan `Image.draft`/`reduce` supaya loader tidak lambat (92 file 3000×4000 + 40 PNG 7 MB).
3. **Val yang jujur**: hitung Macro F1 tambahan pada subset val yang **tidak** punya near-dup di train (1.327 dari 1.833; Rusak tinggal 246). Angka inilah yang mendekati test. Split resmi tetap dipakai, tapi laporkan dua angka.
4. **Dedup train** (opsional, split tetap): buang 653 file byte-redundan dan 26+ grup konflik Aman/Rusak, atau beri bobot kecil. Minimal: untuk pasangan konflik, set label ke Aman (visual: utuh) atau drop keduanya.
5. **Pseudo-label / override dari kebocoran**: untuk 467 test di `test_pseudo_labels_strong.csv`, tulis label langsung (override prediksi model) — 253 di antaranya byte-identik. Untuk 285 test yang kembar di dalam test, rata-ratakan probabilitas antar kembaran sebelum argmax.
6. **Kelompok test yang perlu perlakuan khusus**:
   - 224×224 (520), ≥1500 px / render AI (240), paket oranye 3000×4000 (92): di train semuanya Aman. Pastikan model tidak memberi Lainnya/Rusak di sini tanpa alasan kuat; prior Aman wajar.
   - Conveyor 416×416 (193) dan GHTK 675×900 (280): ikuti statistik train (Rusak dominan) — tapi ini 15% test dan berisiko jika ground truth test tidak mengikuti artefak augmentasi yang sama. Jangan menambah augmentasi noise/rotasi berat saat training; itu justru menghapus sinyal yang (sayangnya) dipakai labelnya.
7. **Lainnya**: karena cirinya ukuran kecil + warna + teks, model kecil pada resolusi rendah (224) sudah cukup; CLIP zero-shot ("retail food package" vs "cardboard shipping box") kemungkinan kuat untuk kelas ini dan bisa dipakai sebagai gate dalam ensemble. Kandidat Lainnya di test terbatas: 54 file dengan sisi 200 px + beberapa 640×640 (beef jerky). Threshold Lainnya sebaiknya di-tune di val tanpa dup (63 Lainnya val, hanya 3 yang bocor).
8. Augmentasi training: flip horizontal OK, rotasi dengan padding hitam dan salt-pepper **jangan** (test hampir tidak punya), color jitter ringan, grayscale 10% (test 1.7% grayscale, Rusak train 6.8%).

##### Catatan ketidakpastian
- Ground truth test tidak diketahui; klaim "test 675×900 akan diprediksi Rusak" adalah inferensi dari distribusi train, bukan verifikasi.
- Skor salt-pepper di `augscore.csv` terkontaminasi tekstur latar (conveyor) — hanya skor `black_corner` yang saya anggap valid.
- Near-dup memakai phash ≤ 6; gambar yang sama dengan crop berbeda tidak tertangkap, jadi angka kebocoran adalah batas bawah.


</details>

In [ ]:
%%simpan runs/eda/scan.py
"""Scan every image in Train_Set and Test_Set: md5, PIL metadata, phash/dhash. CPU only. Read-only."""
import os, sys, hashlib, csv
from multiprocessing import Pool
from PIL import Image, ImageFile
import imagehash
ImageFile.LOAD_TRUNCATED_IMAGES = False

ROOT = "/Users/yopiesuryadi/lomba-cv/data"
OUT = "/Users/yopiesuryadi/lomba-cv/runs/eda/meta.csv"

def scan(args):
    split, label, path = args
    rec = dict(split=split, label=label, path=path, filename=os.path.basename(path),
               ext=os.path.splitext(path)[1].lower(), bytes=os.path.getsize(path),
               md5="", fmt="", mode="", w=0, h=0, has_alpha=0, alpha_nontrivial=0,
               phash="", dhash="", error="")
    try:
        with open(path, "rb") as f:
            rec["md5"] = hashlib.md5(f.read()).hexdigest()
        im = Image.open(path)
        rec["fmt"] = im.format or ""
        rec["mode"] = im.mode
        rec["w"], rec["h"] = im.size
        im.load()  # force full decode to catch truncation
        if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
            rec["has_alpha"] = 1
            a = im.convert("RGBA").getchannel("A")
            lo, hi = a.getextrema()
            rec["alpha_nontrivial"] = int(lo < 255)
        rgb = im.convert("RGB")
        rec["phash"] = str(imagehash.phash(rgb))
        rec["dhash"] = str(imagehash.dhash(rgb))
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"[:200]
    return rec

def main():
    jobs = []
    for label in ("Aman", "Rusak", "Lainnya"):
        d = os.path.join(ROOT, "Train_Set", label)
        for fn in sorted(os.listdir(d)):
            if fn.startswith("."): continue
            jobs.append(("train", label, os.path.join(d, fn)))
    d = os.path.join(ROOT, "Test_Set")
    for fn in sorted(os.listdir(d)):
        if fn.startswith("."): continue
        jobs.append(("test", "", os.path.join(d, fn)))
    print("files:", len(jobs), flush=True)
    with Pool(4) as pool, open(OUT, "w", newline="") as f:
        wr = None
        for i, rec in enumerate(pool.imap_unordered(scan, jobs, chunksize=32)):
            if wr is None:
                wr = csv.DictWriter(f, fieldnames=list(rec.keys())); wr.writeheader()
            wr.writerow(rec)
            if i % 2000 == 0: print(i, flush=True)
    print("done", OUT)

if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/eda/dups.py
import pandas as pd, numpy as np, imagehash, itertools, collections
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 300)
m = pd.read_pickle("runs/eda/meta.pkl")
tr = m[m.split=="train"].reset_index(drop=True); te = m[m.split=="test"].reset_index(drop=True)

print("== EXACT md5 dup within train ==")
g = tr.groupby("md5")
dup_groups = [d for _, d in g if len(d) > 1]
n_files = sum(len(d) for d in dup_groups)
print("groups:", len(dup_groups), "files involved:", n_files, "redundant files:", n_files-len(dup_groups))
same, conflict = 0, []
for d in dup_groups:
    if d.label.nunique() == 1: same += 1
    else: conflict.append(d)
print("same-label groups:", same, "LABEL-CONFLICT groups:", len(conflict))
for d in conflict:
    print("  ", sorted(d.filename.tolist()), sorted(d.label.tolist()))
print("exact dup group size dist:", collections.Counter(len(d) for d in dup_groups))
print("by label (files in dup groups):", pd.concat(dup_groups).label.value_counts().to_dict() if dup_groups else {})
# train/val split overlap of exact dups
sp = pd.concat([pd.read_csv("split/train.csv").assign(fold="train"), pd.read_csv("split/val.csv").assign(fold="val")])
fold = dict(zip(sp.path, sp.fold))
tr["fold"] = tr.path.map(fold)
leak = sum(1 for d in dup_groups if d.path.map(fold).nunique() > 1)
print("exact dup groups spanning train/val split:", leak)

print("\n== EXACT md5 test<->train ==")
md5_tr = tr.groupby("md5").agg(paths=("path", list), labels=("label", lambda s: sorted(set(s))))
hit = te[te.md5.isin(md5_tr.index)]
print("test files with exact md5 match in train:", len(hit))
print("  label dist of matches:", collections.Counter(tuple(md5_tr.loc[h, "labels"]) for h in hit.md5))
print("\n== EXACT md5 dup within test ==")
tg = te.groupby("md5").size(); print("test md5 groups>1:", (tg>1).sum(), "files:", tg[tg>1].sum())

# perceptual
def H(s): return imagehash.hex_to_hash(s)
def arr(series): return np.array([H(s).hash.flatten() for s in series], dtype=bool)
P_tr, P_te = arr(tr.phash), arr(te.phash)
D_tr, D_te = arr(tr.dhash), arr(te.dhash)
def hamming_all(A, B):  # A (n,64) bool, B (m,64) -> (n,m) int
    A = A.astype(np.uint8); B = B.astype(np.uint8)
    return (A[:, None, :] != B[None, :, :]).sum(-1)

# test -> train nearest neighbor by phash (chunked)
nn_d, nn_i = [], []
for s in range(0, len(te), 256):
    d = hamming_all(P_te[s:s+256], P_tr)
    nn_i.append(d.argmin(1)); nn_d.append(d.min(1))
nn_d = np.concatenate(nn_d); nn_i = np.concatenate(nn_i)
te["nn_train_idx"] = nn_i; te["nn_phash_d"] = nn_d
te["nn_dhash_d"] = [int((D_te[i] != D_tr[j]).sum()) for i, j in enumerate(nn_i)]
print("\n== TEST->TRAIN phash nearest distance distribution ==")
print(pd.Series(nn_d).value_counts().sort_index().head(20).to_string())
for thr in (0, 2, 4, 6, 8, 10):
    print(f"  test files with phash NN dist <= {thr}: {(nn_d<=thr).sum()}")

# all train neighbors within threshold for matched tests -> label agreement
THR = 6
rows = []
for s in range(0, len(te), 256):
    d = hamming_all(P_te[s:s+256], P_tr)
    for ii, jj in zip(*np.where(d <= THR)):
        i = s + ii
        rows.append(dict(test_filename=te.filename[i], train_path=tr.path[jj], label=tr.label[jj],
                         hash_distance=int(d[ii, jj]), dhash_distance=int((D_te[i] != D_tr[jj]).sum()),
                         exact_md5=int(te.md5[i] == tr.md5[jj]),
                         test_res=f"{te.w[i]}x{te.h[i]}", train_res=f"{tr.w[jj]}x{tr.h[jj]}", train_fold=fold.get(tr.path[jj], "")))
mt = pd.DataFrame(rows).sort_values(["test_filename", "hash_distance"])
mt.to_csv("runs/eda/test_train_matches.csv", index=False)
print(f"\nwrote test_train_matches.csv: {len(mt)} pairs, {mt.test_filename.nunique()} unique test files (phash<={THR})")
agg = mt.groupby("test_filename").agg(n=("label", "size"), labels=("label", lambda s: tuple(sorted(set(s)))), mind=("hash_distance", "min"))
print("per-test label consistency:", collections.Counter(agg.labels))
print("by min distance:"); print(pd.crosstab(agg.mind, agg.labels.astype(str)))
# strict pseudo-label set: dist<=4 and consistent
strict = agg[(agg.mind <= 4) & (agg.labels.apply(len) == 1)]
strict = strict.assign(label=strict.labels.apply(lambda t: t[0]))
strict[["label", "mind", "n"]].to_csv("runs/eda/test_pseudo_labels_strict.csv")
print("strict pseudo-label set (phash<=4, consistent):", len(strict), strict.label.value_counts().to_dict())

print("\n== NEAR-DUP within train (phash<=THR, excluding exact md5) ==")
pairs = []
for s in range(0, len(tr), 512):
    d = hamming_all(P_tr[s:s+512], P_tr)
    for ii, jj in zip(*np.where(d <= THR)):
        i = s + ii
        if jj > i: pairs.append((i, jj, int(d[ii, jj])))
pp = pd.DataFrame(pairs, columns=["i", "j", "d"])
pp["li"] = tr.label.values[pp.i]; pp["lj"] = tr.label.values[pp.j]
pp["same_md5"] = tr.md5.values[pp.i] == tr.md5.values[pp.j]
pp["fi"] = tr.fold.values[pp.i]; pp["fj"] = tr.fold.values[pp.j]
print("pairs:", len(pp), "exact:", pp.same_md5.sum(), "near (non-exact):", (~pp.same_md5).sum())
print("distance dist (non-exact):", pp[~pp.same_md5].d.value_counts().sort_index().to_dict())
conf = pp[pp.li != pp.lj]
print("CROSS-LABEL pairs:", len(conf)); print(pd.crosstab(conf.li, conf.lj))
conf = conf.assign(fi_name=tr.filename.values[conf.i], fj_name=tr.filename.values[conf.j])
conf.to_csv("runs/eda/train_cross_label_pairs.csv", index=False)
print(conf[["fi_name", "li", "fj_name", "lj", "d", "same_md5"]].sort_values("d").head(60).to_string())
print("\nnear-dup pairs spanning train/val fold:", (pp.fi != pp.fj).sum(), "of", len(pp))
# connected components to count unique images
import scipy.sparse as sps, scipy.sparse.csgraph as csg
A = sps.coo_matrix((np.ones(len(pp)), (pp.i, pp.j)), shape=(len(tr), len(tr)))
ncomp, lab = csg.connected_components(A, directed=False)
sizes = pd.Series(lab).value_counts()
print("unique train images after merging near-dups (phash<=6):", ncomp, "of", len(tr), "; largest cluster sizes:", sizes.head(10).tolist())
tr["cluster"] = lab
big = sizes[sizes >= 5].index
for c in big[:8]:
    d = tr[tr.cluster == c]
    print(f"  cluster {c}: n={len(d)} labels={d.label.value_counts().to_dict()} e.g. {d.filename.head(4).tolist()}")
tr.to_pickle("runs/eda/train_meta.pkl"); te.to_pickle("runs/eda/test_meta.pkl")

In [ ]:
%%simpan runs/eda/augscore.py
"""Proxy for augmentation artifacts: black-corner fraction (rotation padding) and salt-pepper noise score. CPU."""
import pandas as pd, numpy as np
from multiprocessing import Pool
from PIL import Image
def score(path):
    try:
        im = Image.open(path).convert("L").resize((128,128))
        a = np.asarray(im, dtype=np.int16)
        c = 12
        corners = np.concatenate([a[:c,:c].ravel(), a[:c,-c:].ravel(), a[-c:,:c].ravel(), a[-c:,-c:].ravel()])
        black = float((corners <= 3).mean())
        # salt-pepper: pixels differing strongly from 3x3 median
        from scipy.ndimage import median_filter
        med = median_filter(a, size=3)
        sp = float((np.abs(a - med) > 60).mean())
        gray = Image.open(path).convert("RGB").resize((64,64)); g = np.asarray(gray, dtype=np.int16)
        sat = float(np.abs(g[...,0]-g[...,1]).mean() + np.abs(g[...,1]-g[...,2]).mean())  # ~0 => grayscale
        return black, sp, sat
    except Exception:
        return np.nan, np.nan, np.nan
if __name__ == "__main__":
    tr = pd.read_pickle("runs/eda/train_meta.pkl"); te = pd.read_pickle("runs/eda/test_meta.pkl")
    m = pd.concat([tr, te], ignore_index=True)
    with Pool(4) as p: res = p.map(score, m.path.tolist(), chunksize=64)
    m[["black_corner","saltpepper","colorfulness"]] = np.array(res)
    m["grp"] = np.where(m.split=="test", "TEST", m.label)
    m["rotated"] = m.black_corner > 0.5; m["noisy"] = m.saltpepper > 0.02; m["grayscale"] = m.colorfulness < 1.0
    pd.set_option("display.width", 200)
    print("== share of images with rotation black corners / salt-pepper noise / grayscale ==")
    print(m.groupby("grp")[["rotated","noisy","grayscale"]].mean().round(3))
    for r in ["416x416","675x900","640x640","224x224","512x512"]:
        d = m[m.res==r]
        print(f"\n-- res {r} --"); print(d.groupby("grp")[["rotated","noisy","grayscale"]].agg(["mean","size"]).round(3))
    m[["split","label","filename","res","black_corner","saltpepper","colorfulness"]].to_csv("runs/eda/augscore.csv", index=False)

In [ ]:
%%simpan runs/eda/sheets.py
import pandas as pd, numpy as np, random
from PIL import Image, ImageDraw, ImageFont, ImageOps
random.seed(42); np.random.seed(42)
tr = pd.read_pickle("runs/eda/train_meta.pkl"); te = pd.read_pickle("runs/eda/test_meta.pkl")
OUT = "runs/eda/"
font = ImageFont.load_default()

def thumb(path, size, caption=None):
    try:
        im = Image.open(path)
        if im.mode in ("RGBA","LA","P"):
            # expose alpha as checkerboard-ish grey so transparent regions are visible
            im = im.convert("RGBA"); bg = Image.new("RGBA", im.size, (200,200,200,255)); im = Image.alpha_composite(bg, im)
        im = im.convert("RGB")
        im = ImageOps.contain(im, (size, size-14))
    except Exception as e:
        im = Image.new("RGB", (size, size-14), (255,0,0))
    canvas = Image.new("RGB", (size, size), (30,30,30))
    canvas.paste(im, ((size-im.width)//2, 0))
    if caption:
        ImageDraw.Draw(canvas).text((2, size-13), caption[:40], fill=(255,255,0), font=font)
    return canvas

def sheet(paths, captions, cols, size, out, title=None):
    rows = (len(paths)+cols-1)//cols
    top = 18 if title else 0
    S = Image.new("RGB", (cols*size, rows*size+top), (0,0,0))
    if title: ImageDraw.Draw(S).text((4,3), title, fill=(255,255,255), font=font)
    for k,(p,c) in enumerate(zip(paths, captions)):
        S.paste(thumb(p, size, c), ((k%cols)*size, top+(k//cols)*size))
    S.save(out); print("saved", out, S.size)

# 1) 48 random per class
for lab in ("Aman","Rusak","Lainnya"):
    d = tr[tr.label==lab].sample(48, random_state=42)
    sheet(d.path.tolist(), [f"{f} {w}x{h}" for f,w,h in zip(d.filename,d.w,d.h)], 8, 200, OUT+f"sheet_train_{lab}.png", f"{lab} random 48")
# 2) 96 random test
d = te.sample(96, random_state=42)
sheet(d.path.tolist(), [f"{f} {w}x{h} {m}" for f,w,h,m in zip(d.filename,d.w,d.h,d["mode"])], 12, 200, OUT+"sheet_test_random96.png", "TEST random 96")
# 3) test special groups
grp = te[(te.w==2816)&(te.h==1536)]
sheet(grp.path.tolist()[:40], [f"{f} {m}" for f,m in zip(grp.filename, grp["mode"])], 8, 220, OUT+"sheet_test_2816x1536.png", f"TEST 2816x1536 (n={len(grp)})")
grp = te[((te.w==3000)&(te.h==4000))|((te.w==4000)&(te.h==3000))]
sheet(grp.path.tolist()[:48], [f"{f} {w}x{h}" for f,w,h in zip(grp.filename,grp.w,grp.h)], 8, 220, OUT+"sheet_test_3000x4000.png", f"TEST 3000x4000/4000x3000 (n={len(grp)})")
grp = te[(te.w.isin([1536,1792,2390]))&(te.h.isin([1024,1792]))]
sheet(grp.path.tolist()[:48], [f"{f} {w}x{h}" for f,w,h in zip(grp.filename,grp.w,grp.h)], 8, 220, OUT+"sheet_test_1536x1024_etc.png", f"TEST 1536x1024/1792x1024/2390x1792 (n={len(grp)})")
grp = te[(te.w==224)&(te.h==224)].sample(48, random_state=1)
sheet(grp.path.tolist(), grp.filename.tolist(), 8, 200, OUT+"sheet_test_224x224.png", "TEST 224x224 sample 48")
grp = te[te.nn_phash_d>=14].sample(48, random_state=1)
sheet(grp.path.tolist(), [f"{f} {w}x{h} d{d}" for f,w,h,d in zip(grp.filename,grp.w,grp.h,grp.nn_phash_d)], 8, 200, OUT+"sheet_test_far_from_train.png", "TEST with phash NN dist>=14 (no near train match), sample 48")
# 4) cross-label conflict pairs
cp = pd.read_csv(OUT+"train_cross_label_pairs.csv").sort_values("d")
cp = cp.drop_duplicates("fi_name").head(32)
paths, caps = [], []
for _, r in cp.iterrows():
    paths += [tr.path.values[r.i], tr.path.values[r.j]]; caps += [f"{r.fi_name} d{r.d}", f"{r.fj_name}"]
sheet(paths, caps, 8, 200, OUT+"sheet_train_crosslabel_pairs.png", "Cross-label near-dup pairs (left Aman, right Rusak/Lainnya)")
# 5) test<->train match examples (non-exact, d 2..6)
mt = pd.read_csv(OUT+"test_train_matches.csv")
ex = mt[(mt.exact_md5==0)].drop_duplicates("test_filename").sample(24, random_state=3)
paths, caps = [], []
for _, r in ex.iterrows():
    paths += [f"data/Test_Set/{r.test_filename}", r.train_path]; caps += [f"{r.test_filename} d{r.hash_distance} {r.test_res}", f"{r.label} {r.train_res}"]
sheet(paths, caps, 8, 200, OUT+"sheet_test_train_neardup_examples.png", "test (left) vs matched train (right), non-exact")
# 6) conflicting label test matches
ag = mt.groupby("test_filename").label.agg(lambda s: tuple(sorted(set(s))))
conf = ag[ag.apply(len)>1].index[:16]
paths, caps = [], []
for t in conf:
    sub = mt[mt.test_filename==t].sort_values("hash_distance")
    a = sub[sub.label=="Aman"].iloc[0]; b = sub[sub.label!="Aman"].iloc[0]
    paths += [f"data/Test_Set/{t}", a.train_path, b.train_path]; caps += [t, f"Aman d{a.hash_distance}", f"{b.label} d{b.hash_distance}"]
sheet(paths, caps, 9, 200, OUT+"sheet_test_conflicting_matches.png", "test | Aman match | Rusak/Lainnya match")

In [ ]:
!{PY} runs/eda/scan.py

**[jembatan]** `dups.py` membaca `runs/eda/meta.pkl`, sedangkan `scan.py` menulis `meta.csv`; `augscore.py`
juga butuh kolom `res` (`"WxH"`). Langkah konversi ini tidak ada di paket, jadi ditambahkan di sini.
Kolom hash dibaca sebagai string supaya nol di depan tidak hilang.

In [ ]:
import pandas as pd
meta = pd.read_csv("runs/eda/meta.csv", dtype={"md5": str, "phash": str, "dhash": str})
meta["res"] = meta.w.astype(str) + "x" + meta.h.astype(str)
meta.to_pickle("runs/eda/meta.pkl")
meta.groupby("split").size()

In [ ]:
!{PY} runs/eda/dups.py
!{PY} runs/eda/augscore.py
!{PY} runs/eda/sheets.py

In [ ]:
from IPython.display import Image, display
for f in ["sheet_train_crosslabel_pairs.png", "sheet_test_train_neardup_examples.png"]:
    print(f); display(Image(filename=f"runs/eda/{f}", width=900))

## 3. YOLO11s-cls (`runs/yolo-cls`)

Macro F1 val 0,812. Tidak dipakai di ensemble final.

<details><summary><b>REPORT.md asli: <code>runs/yolo-cls/REPORT.md</code></b> (klik untuk buka)</summary>

#### yolo-cls: Ultralytics YOLO11 classification

##### Pendekatan
- Model: `yolo11s-cls.pt` (pretrained ImageNet, fine-tune penuh) via ultralytics `YOLO.train(task=classify)`.
- Dataset: folder `ds/{train,val}/{Aman,Rusak,Lainnya}/` berisi **symlink** ke file asli sesuai `split/train.csv` dan `split/val.csv` (tidak ada copy). Ultralytics membaca symlink dengan benar (scan: 11471 train, 1833 val, 0 corrupt).
- Class imbalance: Lainnya di-oversample 4x di train lewat symlink bernama `*__dupN` ke file yang sama (train/Lainnya 1440 dari 360 asli). Val tidak diubah (1320/450/63).
- Augmentasi ultralytics default cls + fliplr 0.5, erasing 0.2, randaugment. RGB conversion ditangani ultralytics (cv2 BGR->RGB), prediksi memakai PIL `.convert('RGB')`.
- Prediksi: `predict.py` memprediksi file asli dari val.csv (path cocok untuk ensemble) dan semua file Test_Set, menulis probabilitas softmax 3 kelas.

##### Hyperparameter
- imgsz 320, batch 64, epochs max 6, patience 3, time cap 0.6 jam (ultralytics menyesuaikan jumlah epoch ke budget waktu), device mps, workers 4, optimizer auto (AdamW lr 0.001429), seed 42.
- Epoch yang benar-benar dijalankan: 7. Bobot dipakai: `ul/yolo11s_320/weights/best.pt` (best by top1 val).

##### Kurva training (results.csv)
|   epoch |      time |   train/loss |   metrics/accuracy_top1 |   metrics/accuracy_top5 |   val/loss |   lr/pg0 |   lr/pg1 |   lr/pg2 |
|--------:|----------:|-------------:|------------------------:|------------------------:|-----------:|---------:|---------:|---------:|
|  1.0000 |  432.9070 |       0.5881 |                  0.7714 |                  1.0000 |     0.5837 |   0.0005 |   0.0005 |   0.0005 |
|  2.0000 |  765.3470 |       0.5213 |                  0.7769 |                  1.0000 |     0.5824 |   0.0008 |   0.0008 |   0.0008 |
|  3.0000 | 1206.9700 |       0.5254 |                  0.7932 |                  1.0000 |     0.4943 |   0.0010 |   0.0010 |   0.0010 |
|  4.0000 | 1489.3800 |       0.4773 |                  0.8063 |                  1.0000 |     0.4630 |   0.0007 |   0.0007 |   0.0007 |
|  5.0000 | 1759.8800 |       0.4043 |                  0.8380 |                  1.0000 |     0.4406 |   0.0005 |   0.0005 |   0.0005 |
|  6.0000 | 2013.6400 |       0.3605 |                  0.8265 |                  1.0000 |     0.4243 |   0.0004 |   0.0004 |   0.0004 |
|  7.0000 | 2172.1700 |       0.3242 |                  0.8543 |                  1.0000 |     0.3902 |   0.0002 |   0.0002 |   0.0002 |

##### Hasil val (argmax, sklearn)
- **Macro F1 val: 0.8118**
- Per-class F1: Aman 0.9020, Rusak 0.7041, Lainnya 0.8293

Confusion matrix (baris = true, kolom = pred; urutan Aman/Rusak/Lainnya):

| true \ pred | Aman | Rusak | Lainnya |
|---|---|---|---|
| Aman | 1220 | 91 | 9 |
| Rusak | 155 | 295 | 0 |
| Lainnya | 10 | 2 | 51 |

##### Waktu
- Training: 36.5 menit (wall clock, GPU MPS dipakai bersama agen lain).
- Prediksi val: 19 s; prediksi test (3057 file): 27 s.

##### Output
- `submission.csv`: 3057 baris, semua filename cocok dengan sample_submission.csv: True. Distribusi prediksi test: {'Aman': 1756, 'Rusak': 1244, 'Lainnya': 57}.
- File korup/skip: val 0, test 0. File yang gagal dibaca diberi probabilitas uniform 1/3.
- `val_probs.csv`, `test_probs.csv`, `metrics.json`, `train.log`, `predict.log`.

##### Cara jalan ulang
```
PY=/Users/yopiesuryadi/.venvs/lomba-cv/bin/python
$PY train.py --model yolo11s-cls.pt --imgsz 320 --epochs 6 --patience 3 --time 0.6 --batch 64 --device mps --workers 4
$PY predict.py
$PY make_report.py
```

##### Catatan masalah
- Percobaan pertama (epochs 15, patience 5) dihentikan di tengah epoch 1: 2.1-2.4 s/it x 180 it ~ 7 menit/epoch karena GPU MPS dipakai beberapa agen bersamaan; 15 epoch tidak muat < 40 menit. Log: `train_attempt1_killed.log`.
- Jalan ulang dengan epochs 6 + `time=0.6` jam. Ultralytics menyesuaikan jumlah epoch ke budget waktu setelah epoch 1 dan akhirnya menjalankan 7 epoch (0.603 jam). Kurva val loss masih turun di epoch terakhir (0.39), top1 0.854: model belum konvergen, lebih banyak epoch kemungkinan menaikkan skor.
- Warmup default ultralytics 3 epoch memakan hampir separuh run pendek ini; LR puncak baru tercapai di epoch 3.
- Kelemahan utama: Rusak (F1 0.70, recall 0.66, 155 Rusak diprediksi Aman). Lainnya cukup baik (F1 0.83, 51/63 benar) berkat oversampling 4x.
- Symlink dibaca normal oleh ultralytics (torchvision ImageFolder + verify_image): 11471 train / 1833 val, 0 corrupt; hardlink tidak diperlukan. Tidak ada file korup di train/val/test (0 skip).
- Urutan kelas internal ultralytics alfabetis (Aman, Lainnya, Rusak); predict.py memetakan ulang ke urutan kolom Aman, Rusak, Lainnya.
- MPS berjalan tanpa masalah dengan ultralytics 8.4.172 / torch 2.14.1; peringatan "NMS time limit exceeded" di log hanya artefak warmup predictor, tidak relevan untuk klasifikasi.


</details>

In [ ]:
%%simpan runs/yolo-cls/train.py
"""yolo-cls: Ultralytics YOLO11 classification on Aman/Rusak/Lainnya.
1) Build ds/{train,val}/{class}/ with symlinks to original files (train.csv / val.csv),
   oversampling Lainnya in train by OVERSAMPLE x (extra symlinks with suffixed names).
2) Train yolo11*-cls with early stopping.
Usage: python train.py [--model yolo11s-cls.pt] [--imgsz 320] [--epochs 15] [--device mps]
"""
import argparse, os, time, json, sys
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/yopiesuryadi/lomba-cv")
RUN = ROOT / "runs" / "yolo-cls"
DS = RUN / "ds"
CLASSES = ["Aman", "Rusak", "Lainnya"]
OVERSAMPLE = 4  # Lainnya x4 in train


def build_ds(use_hardlink=False):
    if DS.exists():
        import shutil
        shutil.rmtree(DS)
    counts = {}
    for split in ["train", "val"]:
        df = pd.read_csv(ROOT / "split" / f"{split}.csv")
        for c in CLASSES:
            (DS / split / c).mkdir(parents=True, exist_ok=True)
        n = 0
        for p, lab in zip(df["path"], df["label"]):
            src = Path(p)
            if not src.exists():
                print("MISSING", src, file=sys.stderr)
                continue
            reps = OVERSAMPLE if (split == "train" and lab == "Lainnya") else 1
            for r in range(reps):
                name = src.name if r == 0 else f"{src.stem}__dup{r}{src.suffix}"
                dst = DS / split / lab / name
                if use_hardlink:
                    os.link(src, dst)
                else:
                    os.symlink(src, dst)
                n += 1
        counts[split] = n
    for split in ["train", "val"]:
        for c in CLASSES:
            counts[f"{split}/{c}"] = len(list((DS / split / c).iterdir()))
    print("dataset built:", json.dumps(counts, indent=1))
    (RUN / "ds_counts.json").write_text(json.dumps(counts, indent=1))
    return counts


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="yolo11s-cls.pt")
    ap.add_argument("--imgsz", type=int, default=320)
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--patience", type=int, default=5)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--device", default="mps")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--hardlink", action="store_true")
    ap.add_argument("--skip-build", action="store_true")
    ap.add_argument("--time", type=float, default=None, help="hours cap; ultralytics refits epochs to budget")
    ap.add_argument("--name", default="yolo11s_320")
    a = ap.parse_args()

    if not a.skip_build:
        build_ds(a.hardlink)

    from ultralytics import YOLO
    model = YOLO(a.model)
    t0 = time.time()
    model.train(
        data=str(DS), imgsz=a.imgsz, epochs=a.epochs, patience=a.patience,
        batch=a.batch, device=a.device, workers=a.workers, time=a.time,
        project=str(RUN / "ul"), name=a.name, exist_ok=True,
        pretrained=True, seed=42, deterministic=False, plots=True, verbose=True,
        # mild aug; keep geometry simple for packages
        fliplr=0.5, flipud=0.0, erasing=0.2, auto_augment="randaugment",
    )
    dt = time.time() - t0
    info = {"model": a.model, "imgsz": a.imgsz, "epochs": a.epochs, "patience": a.patience,
            "batch": a.batch, "device": a.device, "workers": a.workers, "time_cap_hours": a.time,
            "oversample_lainnya": OVERSAMPLE, "train_seconds": dt,
            "weights": str(RUN / "ul" / a.name / "weights" / "best.pt")}
    (RUN / "train_info.json").write_text(json.dumps(info, indent=1))
    print("TRAIN DONE", json.dumps(info))


if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/yolo-cls/predict.py
"""yolo-cls: predict val (original paths from val.csv) + Test_Set, write val_probs.csv,
test_probs.csv, submission.csv, metrics.json.
Usage: python predict.py [--weights ul/<name>/weights/best.pt] [--imgsz 320] [--device mps]
"""
import argparse, json, sys, time
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image

ROOT = Path("/Users/yopiesuryadi/lomba-cv")
RUN = ROOT / "runs" / "yolo-cls"
CLASSES = ["Aman", "Rusak", "Lainnya"]


def load_rgb(p):
    try:
        im = Image.open(p)
        im.load()
        return im.convert("RGB")
    except Exception as e:
        print(f"CORRUPT/SKIP {p}: {e}", file=sys.stderr)
        return None


def predict_paths(model, paths, imgsz, device, bs=64):
    """Return (N,3) probs ordered as CLASSES. Corrupt files -> uniform probs."""
    names = model.names  # idx -> name
    order = [list(names.values()).index(c) for c in CLASSES]
    out = np.full((len(paths), 3), 1 / 3, dtype=np.float64)
    bad = []
    for i in range(0, len(paths), bs):
        chunk = paths[i:i + bs]
        ims, idx = [], []
        for j, p in enumerate(chunk):
            im = load_rgb(p)
            if im is None:
                bad.append(str(p)); continue
            ims.append(im); idx.append(i + j)
        if not ims:
            continue
        res = model.predict(ims, imgsz=imgsz, device=device, verbose=False)
        for k, r in zip(idx, res):
            pr = r.probs.data.cpu().numpy()
            out[k] = pr[order]
        if (i // bs) % 10 == 0:
            print(f"  {i + len(chunk)}/{len(paths)}", flush=True)
    return out, bad


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", default=None)
    ap.add_argument("--imgsz", type=int, default=None)
    ap.add_argument("--device", default="mps")
    a = ap.parse_args()
    info = json.loads((RUN / "train_info.json").read_text()) if (RUN / "train_info.json").exists() else {}
    weights = a.weights or info.get("weights")
    imgsz = a.imgsz or info.get("imgsz", 320)

    from ultralytics import YOLO
    from sklearn.metrics import f1_score, confusion_matrix, classification_report
    model = YOLO(weights)
    print("names:", model.names)

    # ---- val
    val = pd.read_csv(ROOT / "split" / "val.csv")
    t0 = time.time()
    vp, vbad = predict_paths(model, [Path(p) for p in val["path"]], imgsz, a.device)
    vdf = val.copy()
    for i, c in enumerate(CLASSES):
        vdf[f"p_{c}"] = vp[:, i]
    vdf.to_csv(RUN / "val_probs.csv", index=False)
    pred = [CLASSES[i] for i in vp.argmax(1)]
    y = val["label"].tolist()
    macro = f1_score(y, pred, average="macro", labels=CLASSES)
    per = f1_score(y, pred, average=None, labels=CLASSES)
    cm = confusion_matrix(y, pred, labels=CLASSES)
    rep = classification_report(y, pred, labels=CLASSES, digits=4)
    metrics = {"macro_f1": float(macro), "per_class_f1": dict(zip(CLASSES, map(float, per))),
               "confusion_matrix_rows_true_cols_pred": cm.tolist(), "classes": CLASSES,
               "val_corrupt": vbad, "val_predict_seconds": time.time() - t0, "weights": weights, "imgsz": imgsz}
    print(rep); print("MACRO F1:", macro); print(cm)

    # ---- test
    test_dir = ROOT / "data" / "Test_Set"
    files = sorted([p for p in test_dir.iterdir() if p.is_file() and not p.name.startswith(".")])
    t0 = time.time()
    tp, tbad = predict_paths(model, files, imgsz, a.device)
    tdf = pd.DataFrame({"filename": [f.name for f in files]})
    for i, c in enumerate(CLASSES):
        tdf[f"p_{c}"] = tp[:, i]
    tdf.to_csv(RUN / "test_probs.csv", index=False)
    sub = pd.DataFrame({"filename": tdf["filename"], "label": [CLASSES[i] for i in tp.argmax(1)]})
    sub.to_csv(RUN / "submission.csv", index=False)
    metrics["test_corrupt"] = tbad
    metrics["test_predict_seconds"] = time.time() - t0
    metrics["test_pred_counts"] = sub["label"].value_counts().to_dict()

    # ---- sanity vs sample_submission
    samp = pd.read_csv(ROOT / "data" / "sample_submission.csv")
    assert len(sub) == 3057, len(sub)
    assert set(samp["filename"]) == set(sub["filename"]), "filename mismatch vs sample_submission"
    assert list(sub.columns) == ["filename", "label"]
    metrics["submission_rows"] = len(sub)
    metrics["submission_filenames_match_sample"] = True
    (RUN / "metrics.json").write_text(json.dumps(metrics, indent=1))
    print("PREDICT DONE", json.dumps({k: v for k, v in metrics.items() if "corrupt" not in k}))


if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/yolo-cls/make_report.py
"""Build REPORT.md from train_info.json, metrics.json, ds_counts.json, ul/<name>/results.csv."""
import json
from pathlib import Path
import pandas as pd

RUN = Path("/Users/yopiesuryadi/lomba-cv/runs/yolo-cls")
info = json.loads((RUN / "train_info.json").read_text())
m = json.loads((RUN / "metrics.json").read_text())
ds = json.loads((RUN / "ds_counts.json").read_text())
res = pd.read_csv(Path(info["weights"]).parent.parent / "results.csv")
res.columns = [c.strip() for c in res.columns]
C = m["classes"]
cm = m["confusion_matrix_rows_true_cols_pred"]
notes = (RUN / "NOTES.md").read_text() if (RUN / "NOTES.md").exists() else ""

L = []
L.append("# yolo-cls: Ultralytics YOLO11 classification\n")
L.append("## Pendekatan")
L.append(f"- Model: `{info['model']}` (pretrained ImageNet, fine-tune penuh) via ultralytics `YOLO.train(task=classify)`.")
L.append(f"- Dataset: folder `ds/{{train,val}}/{{Aman,Rusak,Lainnya}}/` berisi **symlink** ke file asli sesuai `split/train.csv` dan `split/val.csv` (tidak ada copy). Ultralytics membaca symlink dengan benar (scan: {ds['train']} train, {ds['val']} val, 0 corrupt).")
L.append(f"- Class imbalance: Lainnya di-oversample {info['oversample_lainnya']}x di train lewat symlink bernama `*__dupN` ke file yang sama (train/Lainnya {ds['train/Lainnya']} dari 360 asli). Val tidak diubah ({ds['val/Aman']}/{ds['val/Rusak']}/{ds['val/Lainnya']}).")
L.append("- Augmentasi ultralytics default cls + fliplr 0.5, erasing 0.2, randaugment. RGB conversion ditangani ultralytics (cv2 BGR->RGB), prediksi memakai PIL `.convert('RGB')`.")
L.append("- Prediksi: `predict.py` memprediksi file asli dari val.csv (path cocok untuk ensemble) dan semua file Test_Set, menulis probabilitas softmax 3 kelas.\n")
L.append("## Hyperparameter")
L.append(f"- imgsz {info['imgsz']}, batch {info['batch']}, epochs max {info['epochs']}, patience {info['patience']}, time cap {info.get('time_cap_hours')} jam (ultralytics menyesuaikan jumlah epoch ke budget waktu), device {info['device']}, workers {info['workers']}, optimizer auto (AdamW lr 0.001429), seed 42.")
L.append(f"- Epoch yang benar-benar dijalankan: {len(res)}. Bobot dipakai: `{Path(info['weights']).relative_to(RUN)}` (best by top1 val).\n")
L.append("## Kurva training (results.csv)")
L.append(res.to_markdown(index=False, floatfmt=".4f"))
L.append("")
L.append("## Hasil val (argmax, sklearn)")
L.append(f"- **Macro F1 val: {m['macro_f1']:.4f}**")
L.append("- Per-class F1: " + ", ".join(f"{c} {m['per_class_f1'][c]:.4f}" for c in C))
L.append("")
L.append("Confusion matrix (baris = true, kolom = pred; urutan " + "/".join(C) + "):\n")
L.append("| true \\ pred | " + " | ".join(C) + " |")
L.append("|---|" + "---|" * len(C))
for c, row in zip(C, cm):
    L.append(f"| {c} | " + " | ".join(str(v) for v in row) + " |")
L.append("")
L.append("## Waktu")
L.append(f"- Training: {info['train_seconds']/60:.1f} menit (wall clock, GPU MPS dipakai bersama agen lain).")
L.append(f"- Prediksi val: {m['val_predict_seconds']:.0f} s; prediksi test (3057 file): {m['test_predict_seconds']:.0f} s.\n")
L.append("## Output")
L.append(f"- `submission.csv`: {m['submission_rows']} baris, semua filename cocok dengan sample_submission.csv: {m['submission_filenames_match_sample']}. Distribusi prediksi test: {m['test_pred_counts']}.")
L.append(f"- File korup/skip: val {len(m['val_corrupt'])}, test {len(m['test_corrupt'])}" + (f" ({m['test_corrupt']})" if m['test_corrupt'] else "") + ". File yang gagal dibaca diberi probabilitas uniform 1/3.")
L.append("- `val_probs.csv`, `test_probs.csv`, `metrics.json`, `train.log`, `predict.log`.\n")
L.append("## Cara jalan ulang")
L.append("```\nPY=/Users/yopiesuryadi/.venvs/lomba-cv/bin/python\n$PY train.py --model yolo11s-cls.pt --imgsz 320 --epochs 6 --patience 3 --time 0.6 --batch 64 --device mps --workers 4\n$PY predict.py\n$PY make_report.py\n```\n")
L.append("## Catatan masalah")
L.append(notes)
(RUN / "REPORT.md").write_text("\n".join(L))
print("REPORT.md written")

In [ ]:
!{PY} runs/yolo-cls/train.py --model yolo11s-cls.pt --imgsz 320 --epochs 6 --patience 3 --time 0.6 --batch 64 --device {DEVICE} --workers 4
!{PY} runs/yolo-cls/predict.py --device {DEVICE}
!{PY} runs/yolo-cls/make_report.py

## 4. EfficientNetV2-b0 + focal loss (`runs/effnet-focal`)

Macro F1 val 0,827. Tidak dipakai di ensemble final. `predict.py` mengimpor dari `train.py`.

<details><summary><b>REPORT.md asli: <code>runs/effnet-focal/REPORT.md</code></b> (klik untuk buka)</summary>

#### effnet-focal — EfficientNetV2-B0 + focal loss + bias tuning

Agen: `effnet-focal`. Output di `/Users/yopiesuryadi/lomba-cv/runs/effnet-focal/`.
Catatan: `runs/codex-effnet/PROMPT.md` yang dirujuk tidak ada di disk (folder codex-effnet pun tidak ada), jadi tugas diinfer dari nama agen: EfficientNet (timm) + focal loss + post-hoc bias tuning untuk Macro F1. Semua aturan BRIEF.md dipatuhi (split tetap, tidak latih di val, semua angka dari eksekusi nyata).

##### Pendekatan
- Backbone: `tf_efficientnetv2_b0.in1k` (timm, 5.9M param), pretrained ImageNet-1k, head 3 kelas, drop_rate 0.2, drop_path 0.1.
  Dipilih setelah benchmark MPS saat kontensi: effnetv2_b0 fp32 0.58 s/step vs efficientnet_b0 0.82 s (fp32) / 0.98 s (fp16). Autocast fp16 di MPS justru lebih lambat untuk arsitektur ini, jadi training fp32.
- Loss: focal loss multi-kelas, gamma 2.0, alpha per kelas = count^-0.5 dinormalisasi mean 1 -> [Aman 0.413, Rusak 0.707, Lainnya 1.881].
- Sampling: WeightedRandomSampler ringan, bobot = count^-0.5 -> porsi per epoch Aman 0.555 / Rusak 0.324 / Lainnya 0.122.
- Augmentasi train: RandomResizedCrop(224, scale 0.5-1.0), HFlip, ColorJitter(0.3,0.3,0.2,0.02), RandomErasing p=0.2. Eval: Resize ke 224x224 (squash, tanpa crop supaya tepi paket tidak terpotong) + hflip TTA.
- Optimizer AdamW, lr body 3e-4 / head 3e-3, wd 0.02, warmup setengah epoch lalu cosine, grad clip 2.0, batch 48, 6 epoch, num_workers 4, seed 42. Checkpoint terbaik dipilih dari Macro F1 val (argmax, tanpa TTA).
- Bias tuning: setelah training, pada log-prob val (best ckpt + TTA) dicari bias aditif untuk Rusak dan Lainnya (Aman tetap 0), grid [-3, 3] langkah 0.1, memaksimalkan Macro F1. Hasil: bias = [0.0, -0.8, -0.7]. `val_probs.csv` dan `test_probs.csv` berisi probabilitas MENTAH (tanpa bias) untuk ensemble; bias disimpan di `bias.json` dan hanya dipakai saat argmax `submission.csv`.
- Gambar dibaca PIL -> convert RGB (RGBA/P ditangani), JPEG draft mode untuk file besar. File korup: train diganti sampel acak + log, eval tensor nol + flag. Kenyataan: 0 file korup di train/val/test (sesuai EDA runs/eda).

##### Hasil val (1833 gambar: Aman 1320, Rusak 450, Lainnya 63), best ckpt epoch 4, hflip TTA

| | Macro F1 | F1 Aman | F1 Rusak | F1 Lainnya |
|---|---|---|---|---|
| argmax mentah | **0.8269** | 0.8913 | 0.7438 | 0.8455 |
| bias tuned [0, -0.8, -0.7] | **0.8499** | 0.9201 | 0.7630 | 0.8667 |

Confusion matrix mentah (baris = asli, kolom = prediksi; urutan Aman, Rusak, Lainnya):
```
[[1132  180    8]
 [  77  373    0]
 [  11    0   52]]
```
Confusion matrix bias tuned:
```
[[1243   72    5]
 [ 128  322    0]
 [  11    0   52]]
```
Per epoch (Macro F1 val argmax tanpa TTA): 0.6760, 0.7393, 0.7810, **0.8275**, 0.8196, 0.8220. Epoch 4 dipakai.

##### Waktu
- Training 6 epoch: 29.4 menit (13:15-13:44 WIB; 4.1-6.6 menit/epoch tergantung kontensi MPS dari 3 agen lain). Val TTA + bias tuning < 1 menit. Prediksi test 3057 gambar dengan TTA ~1 menit.
- Smoke test awal efficientnet_b0 fp16: 3.9 s/step (est 14 menit/epoch) -> diganti ke effnetv2_b0 fp32 (steady 1.2-1.3 s/step).

##### Output
- `val_probs.csv` (1833 baris, path,label,p_Aman,p_Rusak,p_Lainnya; probabilitas mentah)
- `test_probs.csv` (3057 baris, filename,p_Aman,p_Rusak,p_Lainnya; mentah; jumlah baris = 1)
- `submission.csv` (3057 baris, filename,label; urutan sama dengan sample_submission.csv; argmax setelah bias). Distribusi: Aman 1843, Rusak 1162, Lainnya 52 (tanpa bias: 1592 / 1408 / 57).
- `best.pt`, `bias.json`, `metrics.json`, `history.json`, `train.log`, `predict.log`, `train.py`, `predict.py`.
- Jalankan ulang: `cd runs/effnet-focal && python train.py && python predict.py` (venv lomba-cv).

##### Catatan masalah / ketidakpastian
- Bias dituning di val yang sama dengan yang dipakai memilih checkpoint; Lainnya hanya 63 sampel sehingga kenaikan +0.023 bisa sebagian overfit ke val. Untuk ensemble, pakai probabilitas mentah dan tuning bias ulang di level ensemble.
- Kelas Rusak masih paling lemah (F1 0.74-0.76); kebingungan utama Aman <-> Rusak, bukan Lainnya.
- Distribusi prediksi di test (Rusak 38%) lebih tinggi dari porsi Rusak di train/val (24.5%). Bisa berarti test memang berbeda distribusi, atau model over-predict Rusak; belum bisa dipastikan tanpa label test.
- Resolusi 224 dan 6 epoch adalah hasil penyesuaian ke batas < 40 menit saat 3 agen lain berjalan; kurva val masih naik sampai epoch 4 dan datar setelahnya, jadi tambahan epoch kemungkinan hanya sedikit membantu, tapi resolusi lebih tinggi (256-288) belum dicoba.


</details>

In [ ]:
%%simpan runs/effnet-focal/train.py
"""effnet-focal: EfficientNet (timm) + focal loss + post-hoc logit bias tuning for Macro F1.
Outputs (in --out): train.log, best.pt, bias.json, metrics.json, val_probs.csv (raw softmax, for ensemble)."""
import argparse, time, math, logging, os, sys, json, random, itertools
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, timm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, confusion_matrix
from PIL import Image, ImageFile
import torchvision.transforms as T

ImageFile.LOAD_TRUNCATED_IMAGES = True
CLASSES = ["Aman", "Rusak", "Lainnya"]; CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
HERE = os.path.dirname(os.path.abspath(__file__))

def load_rgb(path, draft=None):
    img = Image.open(path)
    if draft and img.format == "JPEG": img.draft("RGB", (draft, draft))
    return img.convert("RGB")

def train_tf(s):
    return T.Compose([T.RandomResizedCrop(s, scale=(0.5, 1.0)), T.RandomHorizontalFlip(),
                      T.ColorJitter(0.3, 0.3, 0.2, 0.02), T.ToTensor(), T.Normalize(MEAN, STD),
                      T.RandomErasing(p=0.2, scale=(0.02, 0.12))])
def eval_tf(s):
    return T.Compose([T.Resize((s, s)), T.ToTensor(), T.Normalize(MEAN, STD)])

class CsvDS(Dataset):
    def __init__(self, paths, labels, tf, size, train, log):
        self.p, self.y, self.tf, self.size, self.train, self.log = list(paths), list(labels), tf, size, train, log
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        try:
            return self.tf(load_rgb(self.p[i], self.size * 2)), self.y[i], 1
        except Exception as e:
            self.log.warning(f"CORRUPT/SKIP {self.p[i]}: {e}")
            if self.train: return self[random.randrange(len(self))]
            return torch.zeros(3, self.size, self.size), self.y[i], 0

class FocalLoss(nn.Module):
    """Multi-class focal loss: -alpha_y (1-p_y)^gamma log p_y, alpha normalized to mean 1."""
    def __init__(self, alpha, gamma=2.0):
        super().__init__(); self.register_buffer("alpha", torch.as_tensor(alpha, dtype=torch.float32)); self.gamma = gamma
    def forward(self, logits, y):
        logp = F.log_softmax(logits.float(), 1); lp = logp.gather(1, y[:, None]).squeeze(1); p = lp.exp()
        return (self.alpha[y] * (1 - p).pow(self.gamma) * -lp).mean()

def tune_bias(logp, y, rng=(-3.0, 3.0), step=0.1):
    """Grid search additive bias on log-probs of Rusak & Lainnya (Aman fixed 0) maximizing macro F1."""
    grid = np.arange(rng[0], rng[1] + 1e-9, step); best = (-1, (0.0, 0.0))
    for b1, b2 in itertools.product(grid, grid):
        f = f1_score(y, (logp + np.array([0, b1, b2])).argmax(1), average="macro")
        if f > best[0] + 1e-12: best = (f, (float(b1), float(b2)))
    return best

@torch.no_grad()
def predict_logits(model, dl, dev, use_amp, tta=True):
    model.eval(); out = []; oks = []
    for x, _, ok in dl:
        x = x.to(dev)
        with torch.autocast("mps", dtype=torch.float16, enabled=use_amp):
            lo = model(x).float()
            if tta: lo = (lo + model(torch.flip(x, dims=[3])).float()) / 2
        out.append(lo.cpu()); oks.append(ok)
    return torch.cat(out).numpy(), torch.cat(oks).numpy()

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="tf_efficientnetv2_b0.in1k")
    ap.add_argument("--img", type=int, default=224)
    ap.add_argument("--epochs", type=int, default=6)
    ap.add_argument("--bs", type=int, default=48)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--head_lr", type=float, default=3e-3)
    ap.add_argument("--wd", type=float, default=0.02)
    ap.add_argument("--gamma", type=float, default=2.0)
    ap.add_argument("--alpha_power", type=float, default=0.5, help="focal alpha = count^-power (normalized mean 1)")
    ap.add_argument("--sampler_power", type=float, default=0.5, help="WeightedRandomSampler weight = count^-power; 0 = plain shuffle")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--amp", type=int, default=0, help="fp16 autocast is SLOWER on contended MPS for effnet; keep 0")
    ap.add_argument("--smoke", type=int, default=0)
    ap.add_argument("--out", default=HERE)
    args = ap.parse_args()
    OUT = args.out; os.makedirs(OUT, exist_ok=True)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(f"{OUT}/train.log")])
    log = logging.getLogger("effnet-focal"); log.info(f"args: {vars(args)}")
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    dev = torch.device("mps" if torch.backends.mps.is_available() else "cpu"); use_amp = bool(args.amp) and dev.type == "mps"

    tr = pd.read_csv("/Users/yopiesuryadi/lomba-cv/split/train.csv"); va = pd.read_csv("/Users/yopiesuryadi/lomba-cv/split/val.csv")
    ytr = tr.label.map(CLS2IDX).values; yva = va.label.map(CLS2IDX).values
    counts = np.bincount(ytr, minlength=3); log.info(f"train counts {dict(zip(CLASSES, counts.tolist()))} val counts {dict(zip(CLASSES, np.bincount(yva, minlength=3).tolist()))}")
    alpha = counts.astype(float) ** (-args.alpha_power); alpha = alpha / alpha.mean(); log.info(f"focal alpha {np.round(alpha,3).tolist()} gamma {args.gamma}")

    ds_tr = CsvDS(tr.path, ytr, train_tf(args.img), args.img, True, log); ds_va = CsvDS(va.path, yva, eval_tf(args.img), args.img, False, log)
    if args.sampler_power > 0:
        w = (counts.astype(float) ** (-args.sampler_power))[ytr]
        share = np.bincount(ytr, weights=w, minlength=3) / w.sum(); log.info(f"sampler power {args.sampler_power} -> expected share {np.round(share,3).tolist()}")
        sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), len(ytr), replacement=True); shuffle = False
    else: sampler = None; shuffle = True
    dl_tr = DataLoader(ds_tr, batch_size=args.bs, sampler=sampler, shuffle=shuffle, num_workers=args.workers, drop_last=True, persistent_workers=True, prefetch_factor=4)
    dl_va = DataLoader(ds_va, batch_size=96, shuffle=False, num_workers=args.workers)

    model = timm.create_model(args.model, pretrained=True, num_classes=3, drop_rate=0.2, drop_path_rate=0.1).to(dev)
    log.info(f"model {args.model} params {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
    head = list(model.get_classifier().parameters()); hid = {id(p) for p in head}
    opt = torch.optim.AdamW([{"params": [p for p in model.parameters() if id(p) not in hid], "lr": args.lr}, {"params": head, "lr": args.head_lr}], weight_decay=args.wd)
    spe = len(dl_tr); total = spe * args.epochs; warm = max(1, spe // 2)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    crit = FocalLoss(alpha, args.gamma).to(dev)

    def evaluate(tta=False):
        lo, ok = predict_logits(model, dl_va, dev, use_amp, tta=tta); pred = lo.argmax(1)
        return f1_score(yva, pred, average="macro"), f1_score(yva, pred, average=None), confusion_matrix(yva, pred), lo, ok

    best = -1; hist = []; t0 = time.time(); step = 0
    for ep in range(args.epochs):
        model.train(); te = time.time(); run = 0.0; n = 0
        for x, y, _ in dl_tr:
            x, y = x.to(dev, non_blocking=True), y.to(dev)
            with torch.autocast("mps", dtype=torch.float16, enabled=use_amp):
                loss = crit(model(x), y)
            if not torch.isfinite(loss):
                log.warning(f"non-finite loss step {step}, skip"); opt.zero_grad(set_to_none=True); step += 1; sched.step(); continue
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 2.0); opt.step(); sched.step(); step += 1
            run += loss.item(); n += 1
            if step % 50 == 0: log.info(f"ep{ep+1} step {step}/{total} loss {run/n:.4f} lr {sched.get_last_lr()[0]:.2e} {(time.time()-te)/n:.3f}s/step")
            if args.smoke and step >= args.smoke:
                el = time.time() - te; log.info(f"SMOKE {step} steps {el:.1f}s -> {el/step:.3f}s/step -> est {el/step*spe/60:.1f} min/epoch")
                tv = time.time(); f1, *_ = evaluate(); log.info(f"SMOKE val pass {time.time()-tv:.1f}s f1 {f1:.4f}"); sys.exit(0)
        f1, pc, cm, _, _ = evaluate(); ep_t = time.time() - te
        hist.append({"epoch": ep+1, "loss": run/max(n,1), "macro_f1": float(f1), "per_class_f1": [float(v) for v in pc], "cm": cm.tolist(), "sec": ep_t})
        log.info(f"EPOCH {ep+1}/{args.epochs} loss {run/max(n,1):.4f} macroF1 {f1:.4f} perclass {np.round(pc,4).tolist()} {ep_t/60:.1f}min\n{cm}")
        if f1 > best:
            best = f1; torch.save({"model": model.state_dict(), "args": vars(args), "epoch": ep+1, "macro_f1": float(f1)}, f"{OUT}/best.pt"); log.info(f"  saved best.pt ep{ep+1} F1 {f1:.4f}")
        json.dump({"history": hist, "best_macro_f1": best}, open(f"{OUT}/history.json", "w"), indent=1)
    train_sec = time.time() - t0; log.info(f"TRAIN DONE {train_sec/60:.1f} min best macroF1 {best:.4f}")

    # ---- final val pass with best checkpoint (+hflip TTA) and bias tuning
    model.load_state_dict(torch.load(f"{OUT}/best.pt", map_location=dev)["model"])
    f1_raw, pc_raw, cm_raw, lo, ok = evaluate(tta=True)
    logp = torch.log_softmax(torch.from_numpy(lo), 1).numpy(); P = np.exp(logp)
    f1_b, (b1, b2) = tune_bias(logp, yva); bias = np.array([0.0, b1, b2])
    pred_b = (logp + bias).argmax(1); pc_b = f1_score(yva, pred_b, average=None); cm_b = confusion_matrix(yva, pred_b)
    log.info(f"VAL (best ckpt, hflip TTA) raw argmax macroF1 {f1_raw:.4f} perclass {np.round(pc_raw,4).tolist()}\n{cm_raw}")
    log.info(f"VAL bias-tuned bias={bias.round(2).tolist()} macroF1 {f1_b:.4f} perclass {np.round(pc_b,4).tolist()}\n{cm_b}")
    log.info(f"val corrupt/unreadable: {(ok==0).sum()}")
    json.dump({"bias_logprob": bias.tolist(), "classes": CLASSES}, open(f"{OUT}/bias.json", "w"), indent=1)
    json.dump({"model": args.model, "img": args.img, "epochs": args.epochs, "best_epoch": int(torch.load(f"{OUT}/best.pt", map_location="cpu")["epoch"]),
               "train_sec": train_sec, "val_tta": "hflip",
               "raw": {"macro_f1": float(f1_raw), "per_class_f1": dict(zip(CLASSES, map(float, pc_raw))), "cm": cm_raw.tolist()},
               "bias_tuned": {"bias": bias.tolist(), "macro_f1": float(f1_b), "per_class_f1": dict(zip(CLASSES, map(float, pc_b))), "cm": cm_b.tolist()},
               "history": hist}, open(f"{OUT}/metrics.json", "w"), indent=1)
    pd.DataFrame({"path": va.path, "label": va.label, "p_Aman": P[:,0], "p_Rusak": P[:,1], "p_Lainnya": P[:,2]}).to_csv(f"{OUT}/val_probs.csv", index=False)
    log.info("wrote val_probs.csv bias.json metrics.json")

if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/effnet-focal/predict.py
"""Predict Test_Set with best.pt (+hflip TTA). Writes test_probs.csv (raw softmax) and submission.csv (bias-tuned argmax)."""
import os, sys, json, logging, numpy as np, pandas as pd, torch, timm
from torch.utils.data import DataLoader
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from train import CsvDS, eval_tf, predict_logits, CLASSES, HERE

def main():
    OUT = HERE; TEST = "/Users/yopiesuryadi/lomba-cv/data/Test_Set"
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(f"{OUT}/predict.log")])
    log = logging.getLogger("predict")
    dev = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    ck = torch.load(f"{OUT}/best.pt", map_location=dev); a = ck["args"]
    model = timm.create_model(a["model"], pretrained=False, num_classes=3).to(dev); model.load_state_dict(ck["model"])
    bias = np.array(json.load(open(f"{OUT}/bias.json"))["bias_logprob"])
    sample = pd.read_csv("/Users/yopiesuryadi/lomba-cv/data/sample_submission.csv")
    files = sorted(f for f in os.listdir(TEST) if not f.startswith("."))
    assert set(files) == set(sample.filename), f"test files {len(files)} vs sample {len(sample)} mismatch"
    ds = CsvDS([os.path.join(TEST, f) for f in files], [0]*len(files), eval_tf(a["img"]), a["img"], False, log)
    dl = DataLoader(ds, batch_size=96, shuffle=False, num_workers=4)
    lo, ok = predict_logits(model, dl, dev, dev.type == "mps", tta=True)
    logp = torch.log_softmax(torch.from_numpy(lo), 1).numpy(); P = np.exp(logp)
    log.info(f"test unreadable: {(ok==0).sum()} (predicted as-is from zero tensor)")
    pd.DataFrame({"filename": files, "p_Aman": P[:,0], "p_Rusak": P[:,1], "p_Lainnya": P[:,2]}).to_csv(f"{OUT}/test_probs.csv", index=False)
    pred = (logp + bias).argmax(1); raw = logp.argmax(1)
    sub = pd.DataFrame({"filename": files, "label": [CLASSES[i] for i in pred]})
    sub = sub.set_index("filename").loc[sample.filename].reset_index()  # same order as sample_submission
    sub.to_csv(f"{OUT}/submission.csv", index=False)
    log.info(f"bias {bias.round(2).tolist()} | pred dist biased {sub.label.value_counts().to_dict()} | raw {dict(zip(CLASSES, np.bincount(raw, minlength=3).tolist()))}")
    log.info(f"wrote test_probs.csv ({len(P)}) submission.csv ({len(sub)})")

if __name__ == "__main__":
    main()

In [ ]:
!{PY} runs/effnet-focal/train.py
!{PY} runs/effnet-focal/predict.py

## 5. ConvNeXt-tiny putaran 1 (`runs/timm-convnext`)

Macro F1 val 0,874 (bersih 0,844). **Dipakai di ensemble final.** `common.py` dan `prep_cache.py` diimpor
oleh `train.py`/`predict.py`; cache 256px dibangun otomatis oleh `train.py` bila belum ada.

<details><summary><b>REPORT.md asli: <code>runs/timm-convnext/REPORT.md</code></b> (klik untuk buka)</summary>

#### timm-convnext — fine-tune `convnext_tiny.fb_in22k_ft_in1k` (PyTorch, MPS)

Semua angka di bawah berasal dari eksekusi nyata pada 2026-10-04 (log: `train.log`, `predict.log`, `history.json`, `val_metrics.json`).

##### Pendekatan
- Model: timm `convnext_tiny.fb_in22k_ft_in1k`, head baru 3 kelas (Aman, Rusak, Lainnya), drop_path 0.1. Fine-tune hampir penuh: stem + 2 stage pertama dibekukan (26.59M dari 27.82M parameter dilatih). Pembekuan dipakai untuk memangkas waktu backward karena GPU dipakai bersama agen lain (lihat Masalah).
- Data: `split/train.csv` (10.391) / `split/val.csv` (1.833, kolom val 1 baris header). Split tidak diubah, val tidak dipakai untuk latih.
- Pra-proses: semua gambar (train/val/test) di-cache satu kali ke JPEG RGB sisi pendek 256 px (`prep_cache.py` -> `cache_256/`, 28 detik, 15.281 file). Konversi RGB dilakukan saat cache (menangani RGBA/P). Data asli tidak disentuh. File korup: 0 (`corrupt_files.log` kosong); kalau ada, train mengganti dengan sampel acak + log, eval memberi probabilitas uniform + log.
- Augmentasi: RandomResizedCrop(192, scale 0.6-1.0), HorizontalFlip, ColorJitter(0.3,0.3,0.2), RandomGrayscale(0.05), RandomErasing(p=0.15). Eval: Resize(218) + CenterCrop(192).
- Ketidakseimbangan kelas: **WeightedRandomSampler** (bukan class-weighted CE). Bobot sampel = count_kelas^-0.75, sehingga porsi per epoch kira-kira Aman 44.8% / Rusak 34.2% / Lainnya 21.0% (vs asli 72/25/3.5). Tidak dibuat 33/33/33 penuh karena Lainnya hanya 360 gambar train; dengan power 0.75 tiap gambar Lainnya tampil ~6x per epoch.
- Loss: CrossEntropy label smoothing 0.1. Optimizer AdamW wd 0.05, lr body 1e-4, head 1e-3, warmup linear 1 epoch lalu cosine. Batch 48, num_workers 4. Mixed precision fp16 autocast + GradScaler di MPS: stabil, tidak ada loss non-finite. Grad clipping dilepas (benchmark: menambah ~25% waktu per step di MPS).
- Checkpoint terbaik dipilih berdasarkan Macro F1 val (argmax) tiap epoch -> `best.pt` (epoch 5).
- Prediksi (`predict.py`): TTA ringan = rata-rata softmax orig + hflip, untuk val dan seluruh Test_Set.

##### Hyperparameter final
| item | nilai |
|---|---|
| resolusi | 192 (diturunkan dari 224, lihat Masalah) |
| epoch | 5 (diturunkan dari 8) |
| batch | 48 |
| lr body / head | 1e-4 / 1e-3, AdamW wd 0.05, warmup 1 epoch + cosine |
| label smoothing | 0.1 |
| sampler | WeightedRandomSampler, bobot count^-0.75 |
| freeze | stem + stages[0:2] |
| AMP | fp16 autocast + GradScaler (stabil) |
| seed | 42 |

##### Hasil val (1.833 gambar)
Per epoch (argmax, tanpa TTA):

| epoch | train loss | Macro F1 | F1 Aman | F1 Rusak | F1 Lainnya | menit |
|---|---|---|---|---|---|---|
| 1 | 0.7958 | 0.7329 | 0.8455 | 0.6590 | 0.6941 | 3.7 |
| 2 | 0.5710 | 0.8081 | 0.8757 | 0.7234 | 0.8252 | 4.0 |
| 3 | 0.5122 | 0.8349 | 0.9002 | 0.7641 | 0.8406 | 3.6 |
| 4 | 0.4550 | 0.8660 | 0.9225 | 0.8033 | 0.8722 | 3.2 |
| 5 | 0.4406 | **0.8680** | 0.9215 | 0.7971 | 0.8855 | 3.2 |

Confusion matrix epoch 5 tanpa TTA (baris = true, kolom = pred; urutan Aman, Rusak, Lainnya):
```
[[1186  127    7]
 [  64  383    3]
 [   4    1   58]]
```

**Final (best.pt epoch 5, TTA orig+hflip, argmax) — ini yang ada di `val_probs.csv`:**
- Macro F1 val = **0.8740**
- Per-class F1: Aman 0.9243, Rusak 0.8054, Lainnya 0.8923
- Precision/recall: Aman 0.948/0.902, Rusak 0.757/0.860, Lainnya 0.866/0.921; accuracy 0.892
- Confusion matrix:
```
[[1190  124    6]
 [  60  387    3]
 [   5    0   58]]
```
TTA menambah +0.006 Macro F1. Kurva masih naik di epoch 5 (loss turun, F1 naik tipis) -> model belum jenuh; epoch lebih banyak kemungkinan masih membantu.

##### Test
- `test_probs.csv` dan `submission.csv`: 3.057 baris data (+ header), set filename identik dengan `sample_submission.csv` (di-assert di `predict.py`), 0 file tidak terbaca.
- Distribusi prediksi test: Aman 1673 / Rusak 1319 / Lainnya 65. Porsi Rusak (43%) jauh lebih tinggi dari porsi train (25%). Di val model juga condong ke Rusak (124 Aman -> Rusak, precision Rusak 0.757), efek sampler yang menaikkan porsi Rusak. Untuk ensemble, ambang/prior bisa dikalibrasi memakai `val_probs.csv`.

##### Waktu
- Cache gambar: 28 s. Training 5 epoch termasuk eval tiap epoch: **17.7 menit** (3.2-4.0 menit/epoch, ~1.0 s/step steady-state). Prediksi val + test dengan TTA: 94 s. Total wall-clock < 25 menit (di luar smoke test/benchmark awal).

##### Masalah dan keputusan
1. **GPU dipakai bersama**: saat smoke test, 1 step batch 48 @224 memakan 3.7-4.1 s (est. 13-15 menit/epoch; 8 epoch tidak mungkin < 40 menit). Benchmark sintetis step yang sama berayun 1.5-2.7 s tergantung beban agen lain (load average 5-12). Mitigasi: resolusi 192, 5 epoch, bekukan stem+2 stage, lepas grad-clip, dan guard otomatis `--max_minutes 34` (memotong jumlah epoch kalau epoch-1 terlalu lama; tidak terpicu karena epoch-1 hanya 3.7 menit). Saat training sebenarnya berjalan beban ternyata turun (1.0 s/step), sehingga sebenarnya masih ada ruang untuk 8 epoch; tidak diulang demi batas waktu.
2. **Dataloader**: gambar asli sampai 2816 px membuat decode lambat; diselesaikan dengan cache 256 px (`cache_256/`, 395 MB, bisa dihapus; `train.py` membangunnya ulang otomatis jika tidak ada).
3. macOS memakai `spawn` untuk worker DataLoader -> seluruh skrip harus di bawah `if __name__ == "__main__"` (sudah).
4. `efficientnet_b2` (fallback) di-benchmark: tidak lebih cepat dari convnext_tiny di MPS (depthwise conv lambat), jadi tidak dipakai.
5. Tidak ada file korup ditemukan; mekanisme skip+log tetap terpasang.

##### Cara menjalankan ulang
```
cd /Users/yopiesuryadi/lomba-cv/runs/timm-convnext
/Users/yopiesuryadi/.venvs/lomba-cv/bin/python prep_cache.py 6      # opsional, dibangun otomatis oleh train.py
/Users/yopiesuryadi/.venvs/lomba-cv/bin/python train.py             # default = konfigurasi di atas
/Users/yopiesuryadi/.venvs/lomba-cv/bin/python predict.py           # val_probs.csv, test_probs.csv, submission.csv
```
File: `train.py`, `predict.py`, `common.py` (dataset/transform), `prep_cache.py`, `best.pt`, `history.json`, `val_metrics.json`, `train.log`, `predict.log`.


</details>

In [ ]:
%%simpan runs/timm-convnext/common.py
import logging, random
import numpy as np, pandas as pd, torch
from PIL import Image, ImageFile
from torch.utils.data import Dataset
import torchvision.transforms as T

ImageFile.LOAD_TRUNCATED_IMAGES = True
CLASSES = ["Aman", "Rusak", "Lainnya"]
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def load_rgb(path, draft_size=None):
    """Open image, convert to RGB. JPEG draft mode speeds up decoding of huge files."""
    img = Image.open(path)
    if draft_size and img.format == "JPEG":
        img.draft("RGB", (draft_size, draft_size))
    return img.convert("RGB")

def train_tf(size):
    return T.Compose([
        T.RandomResizedCrop(size, scale=(0.6, 1.0)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(0.3, 0.3, 0.2),
        T.RandomGrayscale(p=0.05),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
        T.RandomErasing(p=0.15, scale=(0.02, 0.15)),
    ])

def eval_tf(size):
    return T.Compose([
        T.Resize(int(size * 1.14)),
        T.CenterCrop(size),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])

class CsvDataset(Dataset):
    """paths + labels; corrupt files are logged and replaced by a random other sample (train)
    or by a zero tensor flagged via ok=0 (eval)."""
    def __init__(self, paths, labels, tf, size, train=True, logger=None, use_cache=True):
        self.paths, self.labels, self.tf, self.size, self.train = list(paths), list(labels), tf, size, train
        if use_cache:  # use pre-resized 256px copies when available (see prep_cache.py)
            from prep_cache import cache_path
            import os
            self.paths = [cache_path(p) if os.path.exists(cache_path(p)) else p for p in self.paths]
            self.n_cached = sum(1 for p in self.paths if p.startswith(os.path.dirname(cache_path(self.paths[0]))))
        self.log = logger or logging.getLogger("data")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]
        try:
            img = load_rgb(p, draft_size=self.size * 2)
            return self.tf(img), self.labels[i], 1
        except Exception as e:
            self.log.warning(f"CORRUPT/SKIP {p}: {e}")
            if self.train:
                return self[random.randrange(len(self))]
            return torch.zeros(3, self.size, self.size), self.labels[i], 0

In [ ]:
%%simpan runs/timm-convnext/prep_cache.py
"""One-time pre-resize cache: shorter side -> 256 px, RGB JPEG q92. Original data untouched.
Reason: raw images go up to 2816 px; decoding them each epoch made the dataloader the bottleneck (~30 img/s)."""
import os, sys, time, pandas as pd
from multiprocessing import Pool
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
ROOT = "/Users/yopiesuryadi/lomba-cv"
CACHE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "cache_256")
SIDE = 256

def cache_path(src):
    rel = os.path.relpath(src, os.path.join(ROOT, "data"))
    return os.path.join(CACHE, os.path.splitext(rel)[0] + ".jpg")

def work(src):
    dst = cache_path(src)
    if os.path.exists(dst): return (src, "ok")
    try:
        img = Image.open(src)
        if img.format == "JPEG": img.draft("RGB", (SIDE*2, SIDE*2))
        img = img.convert("RGB")
        w, h = img.size; s = SIDE / min(w, h)
        if s < 1: img = img.resize((max(1, round(w*s)), max(1, round(h*s))), Image.BICUBIC)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        img.save(dst, "JPEG", quality=92)
        return (src, "ok")
    except Exception as e:
        return (src, f"CORRUPT {e!r}")

if __name__ == "__main__":
    tr = pd.read_csv(f"{ROOT}/split/train.csv"); va = pd.read_csv(f"{ROOT}/split/val.csv")
    test = sorted(os.listdir(f"{ROOT}/data/Test_Set"))
    srcs = list(tr.path) + list(va.path) + [f"{ROOT}/data/Test_Set/{f}" for f in test if not f.startswith(".")]
    t = time.time(); bad = []
    with Pool(int(sys.argv[1]) if len(sys.argv) > 1 else 6) as p:
        for i, (s, st) in enumerate(p.imap_unordered(work, srcs, chunksize=16)):
            if st != "ok": bad.append((s, st))
            if i % 2000 == 0: print(i, f"{time.time()-t:.0f}s", flush=True)
    print(f"done {len(srcs)} files in {time.time()-t:.0f}s, corrupt: {len(bad)}")
    with open(os.path.join(os.path.dirname(CACHE), "corrupt_files.log"), "w") as f:
        for s, st in bad: f.write(f"{s}\t{st}\n")
    for b in bad: print(b)

In [ ]:
%%simpan runs/timm-convnext/train.py
"""Fine-tune timm convnext_tiny.fb_in22k_ft_in1k (3 classes) on MPS. See REPORT.md.
Run: python train.py  (defaults = the run reported). Needs cache_256/ from prep_cache.py (built automatically if missing)."""
import argparse, time, math, logging, os, sys, json, random
import numpy as np, pandas as pd, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, confusion_matrix
from common import *

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="convnext_tiny.fb_in22k_ft_in1k")
    ap.add_argument("--img", type=int, default=192)
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--max_minutes", type=float, default=34.0, help="after epoch 1, cut epochs so est. train time fits")
    ap.add_argument("--freeze_stages", type=int, default=2, help="freeze stem + first N convnext stages")
    ap.add_argument("--bs", type=int, default=48)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--head_lr", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--amp", type=int, default=1)
    ap.add_argument("--sampler_power", type=float, default=0.75, help="sample weight = count^-power (1=fully balanced)")
    ap.add_argument("--cache", type=int, default=1)
    ap.add_argument("--smoke", type=int, default=0, help="run N steps then exit (timing)")
    ap.add_argument("--out", default=os.path.dirname(os.path.abspath(__file__)))
    args = ap.parse_args()

    OUT = args.out; os.makedirs(OUT, exist_ok=True)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(f"{OUT}/train.log")])
    log = logging.getLogger("train")
    log.info(f"args: {vars(args)}")
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    dev = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    tr = pd.read_csv("/Users/yopiesuryadi/lomba-cv/split/train.csv")
    va = pd.read_csv("/Users/yopiesuryadi/lomba-cv/split/val.csv")
    ytr = tr.label.map(CLS2IDX).values; yva = va.label.map(CLS2IDX).values
    counts = np.bincount(ytr, minlength=3); log.info(f"train counts {dict(zip(CLASSES, counts.tolist()))}")

    if args.cache and not os.path.isdir(os.path.join(OUT, "cache_256")):
        log.info("building cache_256 ..."); os.system(f"{sys.executable} {os.path.join(OUT, 'prep_cache.py')} 6")

    # --- sampling choice: WeightedRandomSampler (NOT class-weighted CE). weight_i = count[y_i]^-power
    cls_w = counts.astype(float) ** (-args.sampler_power)
    samp_w = cls_w[ytr]
    eff = (cls_w * counts) / (cls_w * counts).sum()
    log.info(f"sampler power {args.sampler_power} -> expected class share per epoch {dict(zip(CLASSES, np.round(eff,3).tolist()))}")
    sampler = WeightedRandomSampler(torch.as_tensor(samp_w, dtype=torch.double), num_samples=len(ytr), replacement=True)

    ds_tr = CsvDataset(tr.path, ytr, train_tf(args.img), args.img, True, use_cache=bool(args.cache))
    ds_va = CsvDataset(va.path, yva, eval_tf(args.img), args.img, False, use_cache=bool(args.cache))
    log.info(f"image source example: {ds_tr.paths[0]}")
    dl_tr = DataLoader(ds_tr, batch_size=args.bs, sampler=sampler, num_workers=args.workers, drop_last=True, persistent_workers=True, prefetch_factor=4)
    dl_va = DataLoader(ds_va, batch_size=96, shuffle=False, num_workers=args.workers)

    model = timm.create_model(args.model, pretrained=True, num_classes=3, drop_path_rate=0.1).to(dev)
    if args.freeze_stages > 0 and hasattr(model, "stages"):
        for mod in [model.stem] + list(model.stages[:args.freeze_stages]):
            for p in mod.parameters(): p.requires_grad = False
    head_params = list(model.get_classifier().parameters()); head_ids = {id(p) for p in head_params}
    body_params = [p for p in model.parameters() if id(p) not in head_ids and p.requires_grad]
    log.info(f"trainable params: {sum(p.numel() for p in body_params+head_params)/1e6:.2f}M / {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
    opt = torch.optim.AdamW([{"params": body_params, "lr": args.lr}, {"params": head_params, "lr": args.head_lr}], weight_decay=args.wd)
    steps_per_epoch = len(dl_tr); S = {"total": steps_per_epoch * args.epochs}; warm = steps_per_epoch
    def lr_lambda(s):
        if s < warm: return (s + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, S["total"] - warm))))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    use_amp = bool(args.amp) and dev.type == "mps"
    scaler = torch.amp.GradScaler("mps", enabled=use_amp)

    @torch.no_grad()
    def evaluate():
        model.eval(); probs = []
        for x, y, ok in dl_va:
            with torch.autocast("mps", dtype=torch.float16, enabled=use_amp):
                lo = model(x.to(dev))
            probs.append(lo.float().softmax(1).cpu())
        P = torch.cat(probs).numpy(); pred = P.argmax(1)
        f1 = f1_score(yva, pred, average="macro"); pc = f1_score(yva, pred, average=None)
        return f1, pc, confusion_matrix(yva, pred), P

    best = -1; hist = []; t0 = time.time(); step = 0
    n_epochs = args.epochs; ep = -1
    while ep + 1 < n_epochs:
        ep += 1
        model.train(); te = time.time(); run = 0.0; n = 0; nan_hit = False
        for x, y, _ in dl_tr:
            x, y = x.to(dev, non_blocking=True), y.to(dev)
            with torch.autocast("mps", dtype=torch.float16, enabled=use_amp):
                loss = crit(model(x), y)
            if not torch.isfinite(loss):
                nan_hit = True; log.warning(f"non-finite loss at step {step}, skipping batch")
                opt.zero_grad(set_to_none=True); step += 1; sched.step(); continue
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step(); step += 1
            run += loss.item(); n += 1
            if step % 50 == 0:
                log.info(f"ep{ep+1} step {step}/{S['total']} loss {run/n:.4f} lr {sched.get_last_lr()[0]:.2e} {(time.time()-te)/n:.3f}s/step")
            if args.smoke and step >= args.smoke:
                el = time.time() - te
                log.info(f"SMOKE: {step} steps in {el:.1f}s -> {el/step:.3f}s/step -> est {el/step*steps_per_epoch/60:.1f} min/epoch train")
                tv = time.time(); f1, pc, cm, _ = evaluate(); log.info(f"SMOKE val pass {time.time()-tv:.1f}s f1 {f1:.4f}")
                sys.exit(0)
        f1, pc, cm, P = evaluate()
        ep_t = time.time() - te
        if ep == 0:
            fit = int(args.max_minutes * 60 // ep_t)
            if fit < n_epochs:
                log.warning(f"epoch took {ep_t/60:.1f} min; cutting epochs {n_epochs} -> {max(2, fit)} to fit {args.max_minutes} min")
                n_epochs = max(2, fit); S["total"] = steps_per_epoch * n_epochs
        rec = {"epoch": ep+1, "train_loss": run/max(n,1), "macro_f1": float(f1), "per_class_f1": [float(v) for v in pc], "cm": cm.tolist(), "epoch_sec": ep_t, "nan": nan_hit, "n_epochs_planned": n_epochs}
        hist.append(rec)
        log.info(f"EPOCH {ep+1}/{n_epochs} loss {rec['train_loss']:.4f} macroF1 {f1:.4f} perclass {np.round(pc,4).tolist()} time {ep_t/60:.1f}min\nCM:\n{cm}")
        if f1 > best:
            best = f1; torch.save({"model": model.state_dict(), "args": vars(args), "epoch": ep+1, "macro_f1": float(f1)}, f"{OUT}/best.pt")
            log.info(f"  saved best.pt (epoch {ep+1}, F1 {f1:.4f})")
        json.dump({"history": hist, "best_macro_f1": best, "total_train_sec": time.time()-t0}, open(f"{OUT}/history.json", "w"), indent=1)
    log.info(f"DONE total {(time.time()-t0)/60:.1f} min, best macroF1 {best:.4f}")

if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/timm-convnext/predict.py
"""Predict val + Test_Set with best.pt, TTA = orig + hflip (mean softmax).
Writes val_probs.csv, test_probs.csv, submission.csv."""
import os, sys, time, logging, numpy as np, pandas as pd, torch, timm
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from common import *

ROOT = "/Users/yopiesuryadi/lomba-cv"
OUT = os.path.dirname(os.path.abspath(__file__))

@torch.no_grad()
def predict(model, dl, dev, use_amp=True):
    probs, oks = [], []
    for x, _, ok in dl:
        x = x.to(dev)
        with torch.autocast("mps", dtype=torch.float16, enabled=use_amp):
            p = model(x).float().softmax(1) + model(torch.flip(x, dims=[3])).float().softmax(1)
        probs.append((p / 2).cpu()); oks.append(ok)
    P = torch.cat(probs).numpy(); ok = torch.cat(oks).numpy()
    P[ok == 0] = 1.0 / 3  # unreadable file -> uniform
    return P, ok

def main():
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(f"{OUT}/predict.log")])
    log = logging.getLogger("predict")
    dev = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    ck = torch.load(f"{OUT}/best.pt", map_location="cpu", weights_only=False)
    args = ck["args"]; size = args["img"]
    log.info(f"best.pt from epoch {ck['epoch']} val macroF1 {ck['macro_f1']:.4f}; img {size}")
    model = timm.create_model(args["model"], pretrained=False, num_classes=3)
    model.load_state_dict(ck["model"]); model.to(dev).eval()
    t0 = time.time()

    # --- val
    va = pd.read_csv(f"{ROOT}/split/val.csv"); yva = va.label.map(CLS2IDX).values
    ds = CsvDataset(va.path, yva, eval_tf(size), size, False, use_cache=bool(args.get("cache", 1)))
    P, ok = predict(model, DataLoader(ds, batch_size=96, num_workers=4), dev)
    pred = P.argmax(1)
    f1 = f1_score(yva, pred, average="macro"); pc = f1_score(yva, pred, average=None); cm = confusion_matrix(yva, pred)
    log.info(f"VAL (TTA) macroF1 {f1:.4f} per-class {dict(zip(CLASSES, np.round(pc,4).tolist()))}\nCM (rows=true, cols=pred, order {CLASSES}):\n{cm}")
    log.info("\n" + classification_report(yva, pred, target_names=CLASSES, digits=4))
    pd.DataFrame({"path": va.path, "label": va.label, "p_Aman": P[:,0], "p_Rusak": P[:,1], "p_Lainnya": P[:,2]}).to_csv(f"{OUT}/val_probs.csv", index=False)
    json_metrics = {"val_macro_f1_tta": float(f1), "per_class_f1": dict(zip(CLASSES, [float(v) for v in pc])), "cm": cm.tolist(), "unreadable_val": int((ok==0).sum())}

    # --- test
    files = sorted(f for f in os.listdir(f"{ROOT}/data/Test_Set") if not f.startswith("."))
    paths = [f"{ROOT}/data/Test_Set/{f}" for f in files]
    ds = CsvDataset(paths, [0]*len(paths), eval_tf(size), size, False, use_cache=bool(args.get("cache", 1)))
    P, ok = predict(model, DataLoader(ds, batch_size=96, num_workers=4), dev)
    log.info(f"TEST: {len(files)} files, unreadable {(ok==0).sum()}, pred dist {dict(zip(CLASSES, np.bincount(P.argmax(1), minlength=3).tolist()))}")
    pd.DataFrame({"filename": files, "p_Aman": P[:,0], "p_Rusak": P[:,1], "p_Lainnya": P[:,2]}).to_csv(f"{OUT}/test_probs.csv", index=False)
    sub = pd.DataFrame({"filename": files, "label": [CLASSES[i] for i in P.argmax(1)]})
    sub.to_csv(f"{OUT}/submission.csv", index=False)
    # sanity vs sample_submission
    ss = pd.read_csv(f"{ROOT}/data/sample_submission.csv")
    assert len(sub) == 3057 and set(sub.filename) == set(ss.filename), (len(sub), len(set(sub.filename) ^ set(ss.filename)))
    json_metrics["test_pred_dist"] = dict(zip(CLASSES, np.bincount(P.argmax(1), minlength=3).tolist()))
    json_metrics["predict_sec"] = time.time() - t0
    import json; json.dump(json_metrics, open(f"{OUT}/val_metrics.json", "w"), indent=1)
    log.info(f"wrote val_probs.csv test_probs.csv submission.csv in {time.time()-t0:.0f}s")

if __name__ == "__main__":
    main()

In [ ]:
!{PY} runs/timm-convnext/train.py
!{PY} runs/timm-convnext/predict.py

## 6. Embedding foundation model + logistic regression (`runs/fm-probe`)

Ekstraksi embedding CLIP ViT-B/16 (laion2b) + DINOv2 ViT-S/14 sekali, lalu classifier di CPU.
Macro F1 val 0,841. Embedding dan `paths_{train,val,test}.json` dari sini **dipakai ulang** oleh `emb-heads` dan `emb-r2`.

<details><summary><b>REPORT.md asli: <code>runs/fm-probe/REPORT.md</code></b> (klik untuk buka)</summary>

#### fm-probe: frozen foundation-model features + Logistic Regression

Agen: `fm-probe`. Tanpa fine-tune backbone; GPU (mps) hanya dipakai untuk ekstraksi embedding sekali, lalu semua classifier jalan di CPU dalam hitungan detik. Embedding disimpan sebagai `.npy` di folder ini dan bisa dipakai ulang agen lain.

##### Pendekatan
1. **Ekstraksi embedding** (`extract.py`, device mps, batch 64, num_workers 4, gambar di-convert ke RGB, file gagal baca -> vektor nol + dicatat di `bad_files_*.log`; hasil: 0 file korup di train/val/test):
   - **OpenCLIP ViT-B-16** pretrained `laion2b_s34b_b88k`, image embedding 512-d, L2-normalized, preprocess bawaan open_clip (224).
   - **DINOv2 ViT-S/14** `vit_small_patch14_dinov2.lvd142m` (timm, `num_classes=0`, `img_size=224`), pooled/CLS feature 384-d, lalu L2-normalized.
2. **Zero-shot CLIP** sebagai pembanding: 2 prompt per kelas (lihat `zeroshot_prompts.json`; termasuk prompt yang diminta: "a photo of an intact cardboard shipping box", "a photo of a damaged, dented, torn or wet cardboard box", "a photo of a plastic shopping bag, paper bag or food packaging"), text embedding dirata-rata per kelas, logit = 100 * cos-sim, argmax. Softmax dari logit zero-shot (3 angka) juga dicoba sebagai **fitur ekstra** (`+zs`).
3. **Classifier** (`train.py`): `StandardScaler` + `LogisticRegression(class_weight='balanced', max_iter=3000)`, grid C = 0.01 / 0.03 / 0.1 / 1 / 10 (grid 0.1/1/10 dari brief diperluas ke bawah karena tren membaik di C kecil). Fitur: clip, dino, clip+dino (concat), masing-masing dengan/tanpa fitur zero-shot. Pemilihan model berdasarkan **Macro F1 val (argmax)**. Train hanya dari `split/train.csv`; val tidak pernah dipakai untuk fit.

##### Hasil Macro F1 val (argmax), semua varian

| Fitur | C=0.01 | C=0.03 | C=0.1 | C=1 | C=10 |
|---|---|---|---|---|---|
| clip | 0.8056 | 0.8039 | 0.8044 | 0.7926 | 0.7879 |
| dino | 0.7207 | 0.7277 | 0.7412 | 0.7521 | 0.7541 |
| **clip+dino** | 0.8371 | **0.8411** | 0.8349 | 0.8345 | 0.8328 |
| clip+zs | 0.8046 | 0.8051 | 0.8074 | 0.7870 | 0.7827 |
| dino+zs | 0.7254 | 0.7328 | 0.7428 | 0.7527 | 0.7587 |
| clip+dino+zs | 0.8392 | 0.8389 | 0.8345 | 0.8298 | 0.8297 |

Zero-shot CLIP murni (tanpa training): **Macro F1 0.3265** (per-class F1 Aman 0.5729, Rusak 0.2832, Lainnya 0.1236). Prompt teks tidak mampu membedakan Aman vs Rusak dan malah banyak menyeret gambar ke Lainnya; zero-shot hanya berguna sebagai pembanding. Menambahkan skor zero-shot sebagai fitur ekstra memberi perubahan kecil (+0.003 pada clip saja, -0.002 pada clip+dino), tidak signifikan.

##### Varian terbaik: LR[clip+dino] C=0.03 (dipakai untuk val_probs/test_probs/submission)

Macro F1 val = **0.8411**, accuracy 0.8434.

| Kelas | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Aman | 0.9259 | 0.8515 | 0.8871 | 1320 |
| Rusak | 0.6547 | 0.8133 | 0.7255 | 450 |
| Lainnya | 0.9333 | 0.8889 | 0.9106 | 63 |

Confusion matrix (baris = true, kolom = pred; urutan Aman, Rusak, Lainnya):

```
            pred Aman  pred Rusak  pred Lainnya
true Aman        1124         193             3
true Rusak         83         366             1
true Lainnya        7           0            56
```

Distribusi prediksi test (3057): Aman 1559, Rusak 1443, Lainnya 55. Catatan: proporsi Rusak di test jauh lebih besar daripada di train (24%), kemungkinan distribusi test berbeda, atau model terlalu agresif ke Rusak karena `class_weight='balanced'`.

##### Waktu (Apple M5, mps, berbagi GPU dengan agen lain)
- Ekstraksi CLIP: train 265 s, val 70 s, test 107 s (~7,4 menit).
- Ekstraksi DINOv2: train 133 s, val 25 s, test 41 s (~3,3 menit).
- Semua fit LR (30 kombinasi) + tulis output: 34,5 s di CPU.
- Total end-to-end kira-kira 12 menit, tanpa training backbone.

##### Catatan masalah
- Zero-shot CLIP sangat lemah di tugas ini (0.33); prompt bahasa alami tidak menangkap "rusak" pada kardus. Jangan dipakai sendiri.
- Kelemahan utama: Aman <-> Rusak (193 Aman diprediksi Rusak, 83 sebaliknya). Fitur frozen global (224 px) kehilangan detail kerusakan kecil (penyok/sobek) — ini batas pendekatan tanpa fine-tune. Lainnya justru paling bersih (F1 0.91) karena beda semantik jelas.
- DINOv2 sendiri lebih lemah daripada CLIP (0.75 vs 0.81), tetapi concat keduanya naik +0.035; fitur saling melengkapi.
- Peringatan PIL "Palette images with Transparency" muncul untuk beberapa file P-mode, tetap terbaca dan di-convert ke RGB; tidak ada file yang dilewati.
- Tidak ada augmentasi / TTA. Kandidat perbaikan: TTA (flip), resolusi lebih tinggi untuk DINOv2, atau ensemble probabilitas dengan agen yang fine-tune.

##### File
- `extract.py` — ekstraksi embedding (jalankan: `python extract.py --models clip,dino`; melewati split yang sudah ada .npy-nya).
- `train.py` — sweep classifier, pilih terbaik, tulis `val_probs.csv`, `test_probs.csv`, `submission.csv`, `results.json`.
- `emb_{clip,dino}_{train,val,test}.npy`, `paths_{train,val,test}.json`, `clip_text_emb.npy`, `zeroshot_prompts.json` — embedding yang bisa dipakai ulang.
- `extract.log`, `train.log`, `train_clip_only.log` (pass awal sebelum DINO selesai), `results.json` — log eksekusi nyata sumber semua angka di atas.


</details>

In [ ]:
%%simpan runs/fm-probe/extract.py
"""Extract frozen foundation-model embeddings (OpenCLIP ViT-B-16 + DINOv2 ViT-S/14) for train/val/test.
Saves .npy embeddings + path lists in the run folder so they can be reused."""
import os, sys, json, time, argparse
import numpy as np, pandas as pd, torch
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
ImageFile.LOAD_TRUNCATED_IMAGES = True

RUN = "/Users/yopiesuryadi/lomba-cv/runs/fm-probe"
SPLIT = "/Users/yopiesuryadi/lomba-cv/split"
TEST_DIR = "/Users/yopiesuryadi/lomba-cv/data/Test_Set"
DEV = "mps" if torch.backends.mps.is_available() else "cpu"

class ImgDS(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]
        try:
            img = Image.open(p); img.load(); img = img.convert("RGB")
            return self.tf(img), i, 1
        except Exception as e:
            return torch.zeros(3, 224, 224), i, 0

def load_lists():
    tr = pd.read_csv(f"{SPLIT}/train.csv"); va = pd.read_csv(f"{SPLIT}/val.csv")
    te = sorted(os.listdir(TEST_DIR))
    te_paths = [os.path.join(TEST_DIR, f) for f in te]
    return {"train": (tr.path.tolist(), tr.label.tolist()),
            "val": (va.path.tolist(), va.label.tolist()),
            "test": (te_paths, None)}

@torch.no_grad()
def run(name, fwd, tf, lists, bs=64, nw=4):
    bad_log = open(f"{RUN}/bad_files_{name}.log", "w")
    for split, (paths, labels) in lists.items():
        out = f"{RUN}/emb_{name}_{split}.npy"
        if os.path.exists(out):
            print(f"[{name}] {split} exists, skip"); continue
        dl = DataLoader(ImgDS(paths, tf), batch_size=bs, num_workers=nw, shuffle=False)
        feats = [None]*len(paths); t0 = time.time(); nbad = 0
        for b, (x, idx, ok) in enumerate(dl):
            f = fwd(x.to(DEV)).float().cpu().numpy()
            for j, i in enumerate(idx.tolist()):
                feats[i] = f[j]
                if ok[j].item() == 0:
                    nbad += 1; bad_log.write(f"{split}\t{paths[i]}\n")
            if b % 20 == 0:
                print(f"[{name}] {split} {b*bs}/{len(paths)} {time.time()-t0:.0f}s", flush=True)
        feats = np.stack(feats).astype(np.float32)
        np.save(out, feats)
        with open(f"{RUN}/paths_{split}.json", "w") as fh:
            json.dump({"paths": paths, "labels": labels}, fh)
        print(f"[{name}] {split} done {feats.shape} in {time.time()-t0:.0f}s, bad={nbad}", flush=True)
    bad_log.close()

def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--models", default="clip,dino"); a = ap.parse_args()
    lists = load_lists()
    if "clip" in a.models:
        import open_clip
        model, _, tf = open_clip.create_model_and_transforms("ViT-B-16", pretrained="laion2b_s34b_b88k")
        model = model.to(DEV).eval()
        def fwd(x):
            f = model.encode_image(x); return f / f.norm(dim=-1, keepdim=True)
        run("clip", fwd, tf, lists)
        # zero-shot text embeddings
        tok = open_clip.get_tokenizer("ViT-B-16")
        prompts = {"Aman": ["a photo of an intact cardboard shipping box",
                            "a photo of an undamaged cardboard parcel box in good condition"],
                   "Rusak": ["a photo of a damaged, dented, torn or wet cardboard box",
                             "a photo of a crushed, broken or ripped cardboard parcel"],
                   "Lainnya": ["a photo of a plastic shopping bag, paper bag or food packaging",
                               "a photo of a plastic bag, paper bag, pouch or food package, not a cardboard box"]}
        with torch.no_grad():
            T = {}
            for k, ps in prompts.items():
                t = model.encode_text(tok(ps).to(DEV)); t = t / t.norm(dim=-1, keepdim=True)
                t = t.mean(0); T[k] = (t / t.norm()).float().cpu().numpy()
        np.save(f"{RUN}/clip_text_emb.npy", np.stack([T["Aman"], T["Rusak"], T["Lainnya"]]))
        json.dump(prompts, open(f"{RUN}/zeroshot_prompts.json", "w"), indent=1)
        del model; torch.mps.empty_cache() if DEV == "mps" else None
    if "dino" in a.models:
        import timm
        model = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, num_classes=0, img_size=224).to(DEV).eval()
        cfg = timm.data.resolve_data_config({}, model=model); cfg["input_size"] = (3, 224, 224)
        tf = timm.data.create_transform(**cfg)
        run("dino", lambda x: model(x), tf, lists)

if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/fm-probe/train.py
"""Fit light classifiers on frozen embeddings (from extract.py), pick best by val Macro F1,
write val_probs.csv, test_probs.csv, submission.csv, results.json."""
import os, json, time, itertools
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, classification_report

RUN = "/Users/yopiesuryadi/lomba-cv/runs/fm-probe"
CLASSES = ["Aman", "Rusak", "Lainnya"]; C2I = {c: i for i, c in enumerate(CLASSES)}

def load(name):
    return {s: np.load(f"{RUN}/emb_{name}_{s}.npy") for s in ["train", "val", "test"]}

def norm(x): return x / np.linalg.norm(x, axis=1, keepdims=True).clip(1e-8)

def main():
    t0 = time.time()
    P = {s: json.load(open(f"{RUN}/paths_{s}.json")) for s in ["train", "val", "test"]}
    ytr = np.array([C2I[l] for l in P["train"]["labels"]]); yva = np.array([C2I[l] for l in P["val"]["labels"]])
    have_dino = os.path.exists(f"{RUN}/emb_dino_test.npy")
    clip = load("clip"); dino = load("dino") if have_dino else None
    if dino: dino = {s: norm(v) for s, v in dino.items()}
    T = np.load(f"{RUN}/clip_text_emb.npy")  # 3 x 512

    # zero-shot (logit scale 100 as in CLIP)
    zs = {s: 100.0 * clip[s] @ T.T for s in clip}
    zs_pred = zs["val"].argmax(1)
    results = {}
    def report(name, pred):
        mf1 = f1_score(yva, pred, average="macro"); pc = f1_score(yva, pred, average=None)
        results[name] = {"macro_f1": float(mf1), "per_class_f1": dict(zip(CLASSES, map(float, pc))),
                         "cm": confusion_matrix(yva, pred).tolist()}
        print(f"{name:40s} MacroF1={mf1:.4f}  per-class={np.round(pc,4)}", flush=True)
    report("zeroshot_clip", zs_pred)

    def softmax(z): z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
    zsp = {s: softmax(zs[s]) for s in zs}

    feats = {"clip": clip}
    if dino:
        feats["dino"] = dino
        feats["clip+dino"] = {s: np.concatenate([clip[s], dino[s]], 1) for s in clip}
    # add zero-shot probs as extra features
    for k in list(feats):
        feats[k + "+zs"] = {s: np.concatenate([feats[k][s], zsp[s]], 1) for s in clip}

    best = None; fitted = {}
    for fname, F in feats.items():
        sc = StandardScaler().fit(F["train"])
        Xtr, Xva = sc.transform(F["train"]), sc.transform(F["val"])
        for C in [0.01, 0.03, 0.1, 1, 10]:
            clf = LogisticRegression(C=C, class_weight="balanced", max_iter=3000, n_jobs=4)
            clf.fit(Xtr, ytr)
            pred = clf.predict(Xva); key = f"LR[{fname}] C={C}"
            report(key, pred)
            if best is None or results[key]["macro_f1"] > results[best]["macro_f1"]:
                best = key; fitted = {"clf": clf, "sc": sc, "F": F}
    print(f"\nBEST: {best}  MacroF1={results[best]['macro_f1']:.4f}")
    print(classification_report(yva, fitted["clf"].predict(fitted["sc"].transform(fitted["F"]["val"])), target_names=CLASSES, digits=4))
    print("confusion (rows=true Aman,Rusak,Lainnya):\n", np.array(results[best]["cm"]))

    clf, sc, F = fitted["clf"], fitted["sc"], fitted["F"]
    pva = clf.predict_proba(sc.transform(F["val"])); pte = clf.predict_proba(sc.transform(F["test"]))
    # class order of clf is sorted ints 0,1,2 == CLASSES order
    assert list(clf.classes_) == [0, 1, 2]
    pd.DataFrame({"path": P["val"]["paths"], "label": P["val"]["labels"],
                  "p_Aman": pva[:, 0], "p_Rusak": pva[:, 1], "p_Lainnya": pva[:, 2]}).to_csv(f"{RUN}/val_probs.csv", index=False)
    fn = [os.path.basename(p) for p in P["test"]["paths"]]
    pd.DataFrame({"filename": fn, "p_Aman": pte[:, 0], "p_Rusak": pte[:, 1], "p_Lainnya": pte[:, 2]}).to_csv(f"{RUN}/test_probs.csv", index=False)
    sub = pd.DataFrame({"filename": fn, "label": [CLASSES[i] for i in pte.argmax(1)]})
    # match sample_submission order
    samp = pd.read_csv("/Users/yopiesuryadi/lomba-cv/data/sample_submission.csv")
    sub = samp[["filename"]].merge(sub, on="filename", how="left")
    assert sub.label.notna().all() and len(sub) == 3057, (sub.label.isna().sum(), len(sub))
    sub.to_csv(f"{RUN}/submission.csv", index=False)
    print("test pred distribution:", sub.label.value_counts().to_dict())
    results["_best"] = best; results["_fit_seconds"] = time.time() - t0
    json.dump(results, open(f"{RUN}/results.json", "w"), indent=1)

if __name__ == "__main__":
    main()

In [ ]:
!{PY} runs/fm-probe/extract.py --models clip,dino
!{PY} runs/fm-probe/train.py

## 7. Head SVM / MLP / kNN di atas embedding putaran 1 (`runs/emb-heads`)

SVM RBF 0,912 dan MLP 3 seed 0,909. **MLP (`*_mlp.csv`) dipakai di ensemble final** sebagai "MLP p1".
`extract_multicrop.py` di folder ini tidak selesai saat lomba (lihat REPORT); ekstraksi multi-crop yang dipakai
ada di `emb-r2`, jadi sel run-nya dibiarkan sebagai komentar.

<details><summary><b>REPORT.md asli: <code>runs/emb-heads/REPORT.md</code></b> (klik untuk buka)</summary>

#### emb-heads: classifier head alternatif di atas embedding beku CLIP+DINO (CPU)

Agen: `emb-heads`. Memakai ulang embedding dari `runs/fm-probe/` (OpenCLIP ViT-B-16 512-d + DINOv2 ViT-S/14 384-d, concat 896-d, DINO di-L2-normalisasi dulu seperti fm-probe), urutan baris mengikuti `paths_{train,val,test}.json`. Semua classifier dilatih di CPU (torch 4 thread). Tujuan: head yang BERBEDA dari LogisticRegression fm-probe (0.8411) untuk diversitas ensemble.

##### Varian yang dicoba (semua pada fitur yang sama: StandardScaler(train) -> 896-d; kNN memakai cosine pada concat L2-normed tanpa scaler)

| Varian | Macro F1 val | F1 Aman | F1 Rusak | F1 Lainnya | Waktu fit |
|---|---|---|---|---|---|
| **SVM RBF C=10, class_weight balanced, probability=True (argmax prob)** | **0.9123** | 0.9507 | 0.8528 | 0.9333 | 467 s |
| MLP seed 2 | 0.9115 | 0.9519 | 0.8607 | 0.9219 | 7 s |
| SVM RBF C=3 | 0.9096 | 0.9489 | 0.8467 | 0.9333 | 407 s |
| MLP rata-rata softmax 3 seed | 0.9087 | 0.9528 | 0.8684 | 0.9048 | 19 s total |
| MLP seed 0 | 0.9055 | 0.9473 | 0.8559 | 0.9134 | 7 s |
| MLP seed 1 | 0.9015 | 0.9461 | 0.8465 | 0.9120 | 6 s |
| SVM RBF C=1 | 0.8956 | 0.9422 | 0.8278 | 0.9167 | 419 s |
| kNN k=5 cosine distance-weighted | 0.8762 | 0.9385 | 0.8207 | 0.8696 | 1 s |
| kNN k=15 cosine distance-weighted | 0.8544 | 0.9321 | 0.7918 | 0.8393 | 1 s |
| (pembanding) fm-probe LR clip+dino C=0.03 | 0.8411 | 0.8871 | 0.7255 | 0.9106 | - |

SVM dengan argmax pada decision function (bukan probabilitas Platt): C=1 0.8909, C=3 0.9014, C=10 0.9071, sedikit di bawah argmax probabilitas. Waktu SVM besar karena `probability=True` menjalankan 5-fold CV internal, dan 3 nilai C dijalankan paralel (joblib, 3 proses) sambil berbagi CPU dengan agen lain.

###### Hyperparameter MLP (`train.py`)
Linear 896->512, GELU, Dropout 0.3 (plus Dropout 0.1 di input), Linear 512->3. CrossEntropy label_smoothing 0.1 dengan bobot kelas sqrt-balanced (sqrt(N/(3*n_c)); bobot "balanced" penuh terlalu agresif ke Rusak, lihat catatan fm-probe). AdamW lr 1e-3, weight_decay 5e-2, OneCycleLR, batch 256, maks 60 epoch, early stop patience 12 pada Macro F1 val (bobot epoch terbaik yang dipakai). 3 seed (0/1/2): berhenti di epoch 29/25/36, epoch terbaik 17/13/24. Probabilitas = softmax.

###### Hyperparameter SVM / kNN
`SVC(kernel='rbf', gamma='scale', class_weight='balanced', probability=True)`, C 1/3/10. `KNeighborsClassifier(weights='distance', metric='cosine')`, k 5/15.

##### Varian terpilih: SVM RBF C=10 (dipakai untuk val_probs.csv / test_probs.csv / submission.csv)

Macro F1 val = **0.9123**, accuracy 0.9274.

| Kelas | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Aman | 0.9310 | 0.9712 | 0.9507 | 1320 |
| Rusak | 0.9073 | 0.8044 | 0.8528 | 450 |
| Lainnya | 0.9825 | 0.8889 | 0.9333 | 63 |

Confusion matrix (baris = true, kolom = pred; urutan Aman, Rusak, Lainnya):

```
            pred Aman  pred Rusak  pred Lainnya
true Aman        1282          37             1
true Rusak         88         362             0
true Lainnya        7           0            56
```

Distribusi prediksi test (3057): Aman 1829, Rusak 1172, Lainnya 56. Kesepakatan label test dengan fm-probe LR: 89.8%; dengan MLP avg3: 96.9%.

###### Alternatif untuk ensemble: MLP rata-rata 3 seed (file `val_probs_mlp.csv`, `test_probs_mlp.csv`, `submission_mlp.csv`)
Macro F1 val 0.9087, accuracy 0.9307. Karakternya berbeda dari SVM: recall Rusak lebih tinggi (0.8578 vs 0.8044), precision Rusak lebih rendah (0.8793 vs 0.9073); SVM C=10 lebih konservatif ke Aman. Confusion matrix MLP avg3: [[1263,53,4],[62,386,2],[6,0,57]]. Distribusi test: Aman 1752, Rusak 1246, Lainnya 59. Probabilitas MLP adalah softmax langsung (lebih wajar untuk dirata-rata dengan agen lain daripada Platt-scaling SVM). Selisih 0.9123 vs 0.9087 masih di dalam sebaran antar-seed MLP (0.9015-0.9115), jadi keduanya layak masuk ensemble.

##### Poin 3 (CLIP multi-crop): DIBATALKAN karena waktu
`extract_multicrop.py` (resize sisi pendek 448 + center crop 448, 4 kuadran 224 + center 224, 5 embedding per gambar, normalisasi open_clip, batch 64 mps) dijalankan 13:20:40. Laju terukur: 1280 gambar train dalam ~204 s (6,3 gambar/s; proses utama idle menunggu GPU, worker loader sudah penuh -> bottleneck GPU yang dibagi 3 training lain). Proyeksi 15.281 gambar x 5 crop = ~40 menit, melebihi batas "< 30 menit", dan ikut memperlambat agen lain. Proses dihentikan 13:25 saat 1280/10391, tidak ada .npy yang tersimpan. `train.py --feat mc|mc+base|mcmax+base` sudah siap kalau ekstraksi dijalankan ulang saat GPU kosong (perkiraan ~8-10 menit bila GPU tidak dibagi, mengacu laju fm-probe 224 global).

##### Waktu (Apple M5, CPU, berbagi mesin dengan 3 training GPU)
- MLP 3 seed: 19 s total. kNN: 2 s. SVM 3 nilai C paralel: 467 s wall (per model 407-467 s).
- `train.py --feat base` end-to-end: 13:21:28 -> 13:29:40 (8,2 menit). Finalize + verifikasi < 10 s.
- Total agen termasuk percobaan multi-crop yang dibatalkan: ~15 menit.

##### Catatan
- Lonjakan dari LR 0.8411 ke 0.91 datang dari head non-linear (RBF / MLP) pada fitur yang sama persis; kelemahan utama fm-probe (Rusak F1 0.73) naik ke 0.85-0.87. Aman<->Rusak tetap sumber error terbesar (SVM: 37 Aman->Rusak, 88 Rusak->Aman), konsisten dengan hilangnya detail kerusakan pada 224 px global.
- Early stopping MLP dan pemilihan seed/C memakai val, sehingga angka val sedikit optimistis (bias seleksi); val tidak pernah dipakai untuk fit bobot/scaler. Rentang antar-seed MLP 0.9015-0.9115 memberi ukuran noise ~0.01.
- sklearn 1.9 memberi FutureWarning: `SVC(probability=True)` deprecated, pengganti `CalibratedClassifierCV`; masih berjalan normal.
- Tidak ada file korup (mengikuti log fm-probe; embedding dipakai apa adanya).

##### File
- `train.py` — semua varian (`--feat base --models mlp,svm,knn`), simpan probabilitas per varian ke `probs/<nama>.npz` dan metrik ke `results.json`; log di `train_base.log` / `train_base.out`.
- `finalize.py` — pilih varian terbaik (atau `--name`), tulis `val_probs.csv`, `test_probs.csv`, `submission.csv` (urutan sama dengan sample_submission, 3057 baris, dicek); `--suffix _mlp` untuk file alternatif. Log: `finalize.log`, `finalize_mlp.log`.
- `extract_multicrop.py` — ekstraksi 5-crop (tidak selesai, lihat `extract_mc.log`).
- `results.json` — semua angka di atas (`_selected` = varian yang dipakai, `_alt_mlp` = alternatif).


</details>

In [ ]:
%%simpan runs/emb-heads/extract_multicrop.py
"""CLIP ViT-B-16 (laion2b_s34b_b88k) multi-crop embeddings: shortest side -> 448, center crop 448,
then 4 quadrants 224 + center 224 = 5 crops. Each crop encoded + L2-normalized; saves per-crop
embeddings (N,5,512) so mean/max pooling can be chosen later. Same split lists/order as fm-probe."""
import os, json, time, argparse
import numpy as np, pandas as pd, torch
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
ImageFile.LOAD_TRUNCATED_IMAGES = True

RUN = "/Users/yopiesuryadi/lomba-cv/runs/emb-heads"
FM = "/Users/yopiesuryadi/lomba-cv/runs/fm-probe"
DEV = "mps" if torch.backends.mps.is_available() else "cpu"
MEAN = (0.48145466, 0.4578275, 0.40821073); STD = (0.26862954, 0.26130258, 0.27577711)  # open_clip ViT-B-16 defaults

class DS(Dataset):
    def __init__(self, paths):
        self.paths = paths
        self.pre = T.Compose([T.Resize(448, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(448)])
        self.post = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]); img.load(); img = img.convert("RGB")
            img = self.pre(img)
            boxes = [(0,0,224,224),(224,0,448,224),(0,224,224,448),(224,224,448,448),(112,112,336,336)]
            x = torch.stack([self.post(img.crop(b)) for b in boxes])  # 5,3,224,224
            return x, i, 1
        except Exception:
            return torch.zeros(5,3,224,224), i, 0

@torch.no_grad()
def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--splits", default="train,val,test"); ap.add_argument("--bs", type=int, default=64)
    a = ap.parse_args()
    import open_clip
    model, _, _ = open_clip.create_model_and_transforms("ViT-B-16", pretrained="laion2b_s34b_b88k")
    model = model.to(DEV).eval()
    bad = open(f"{RUN}/bad_files_mc.log", "a")
    for split in a.splits.split(","):
        out = f"{RUN}/emb_clipmc_{split}.npy"
        if os.path.exists(out): print(f"{split} exists, skip", flush=True); continue
        paths = json.load(open(f"{FM}/paths_{split}.json"))["paths"]
        dl = DataLoader(DS(paths), batch_size=a.bs, num_workers=4, shuffle=False)
        feats = np.zeros((len(paths), 5, 512), np.float32); t0 = time.time(); nbad = 0
        for b, (x, idx, ok) in enumerate(dl):
            B = x.shape[0]
            f = model.encode_image(x.view(-1, 3, 224, 224).to(DEV))
            f = (f / f.norm(dim=-1, keepdim=True)).float().cpu().numpy().reshape(B, 5, 512)
            feats[idx.numpy()] = f
            for j, i in enumerate(idx.tolist()):
                if ok[j].item() == 0: nbad += 1; bad.write(f"{split}\t{paths[i]}\n")
            if b % 20 == 0: print(f"{split} {b*a.bs}/{len(paths)} {time.time()-t0:.0f}s", flush=True)
        np.save(out, feats)
        print(f"{split} done {feats.shape} in {time.time()-t0:.0f}s bad={nbad}", flush=True)
    bad.close()

if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/emb-heads/train.py
"""emb-heads: alternative classifier heads on frozen CLIP+DINO embeddings (from runs/fm-probe) for
ensemble diversity. CPU only. Variants: PyTorch MLP (896->512->3, dropout, label smoothing, class-weighted
CE, AdamW, early stop on val Macro F1), SVM RBF (C grid), kNN cosine (k grid).
Each variant's val/test softmax probs saved to probs/<name>.npz; metrics appended to results.json.
Usage: python train.py --feat base|mc|mc+base [--models mlp,svm,knn]"""
import os, json, time, argparse
import numpy as np, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, confusion_matrix
from joblib import Parallel, delayed

RUN = "/Users/yopiesuryadi/lomba-cv/runs/emb-heads"; FM = "/Users/yopiesuryadi/lomba-cv/runs/fm-probe"
CLASSES = ["Aman", "Rusak", "Lainnya"]; C2I = {c: i for i, c in enumerate(CLASSES)}
SPLITS = ["train", "val", "test"]
torch.set_num_threads(4)

def l2(x): return x / np.linalg.norm(x, axis=1, keepdims=True).clip(1e-8)

def load_feats(feat):
    clip = {s: np.load(f"{FM}/emb_clip_{s}.npy") for s in SPLITS}
    dino = {s: l2(np.load(f"{FM}/emb_dino_{s}.npy")) for s in SPLITS}
    base = {s: np.concatenate([clip[s], dino[s]], 1) for s in SPLITS}
    if feat == "base": return base
    mc = {s: np.load(f"{RUN}/emb_clipmc_{s}.npy") for s in SPLITS}            # N,5,512 per-crop, each L2-normed
    mcmean = {s: l2(mc[s].mean(1)) for s in SPLITS}
    if feat == "mc": return {s: np.concatenate([mcmean[s], dino[s]], 1) for s in SPLITS}        # 896-d: multicrop clip + dino
    if feat == "mc+base": return {s: np.concatenate([mcmean[s], base[s]], 1) for s in SPLITS}   # 1408-d
    if feat == "mcmax+base": return {s: np.concatenate([l2(mc[s].max(1)), mcmean[s], base[s]], 1) for s in SPLITS}  # 1920-d
    raise ValueError(feat)

def metrics(y, p):
    pred = p.argmax(1)
    return {"macro_f1": float(f1_score(y, pred, average="macro")),
            "per_class_f1": dict(zip(CLASSES, map(float, f1_score(y, pred, average=None)))),
            "cm": confusion_matrix(y, pred).tolist()}

class MLP(nn.Module):
    def __init__(self, d, h=512, p=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(0.1), nn.Linear(d, h), nn.GELU(), nn.Dropout(p), nn.Linear(h, 3))
    def forward(self, x): return self.net(x)

def train_mlp(Xtr, ytr, Xva, yva, Xte, seed, epochs=60, patience=12, lr=1e-3, wd=5e-2, bs=256, ls=0.1, log=print):
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr_t, ytr_t = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr)
    Xva_t, Xte_t = torch.tensor(Xva, dtype=torch.float32), torch.tensor(Xte, dtype=torch.float32)
    cnt = np.bincount(ytr, minlength=3); w = torch.tensor((len(ytr) / (3 * cnt)) ** 0.5, dtype=torch.float32)  # sqrt-balanced
    model = MLP(Xtr.shape[1]); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * int(np.ceil(len(ytr) / bs)), pct_start=0.1)
    lossf = nn.CrossEntropyLoss(weight=w, label_smoothing=ls)
    best = (-1, None, 0); bad = 0; hist = []
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(ytr)); tl = 0
        for i in range(0, len(ytr), bs):
            idx = perm[i:i + bs]; opt.zero_grad()
            loss = lossf(model(Xtr_t[idx]), ytr_t[idx]); loss.backward(); opt.step(); sched.step(); tl += loss.item() * len(idx)
        model.eval()
        with torch.no_grad(): pva = torch.softmax(model(Xva_t), 1).numpy()
        f1 = f1_score(yva, pva.argmax(1), average="macro"); hist.append(round(f1, 4))
        if f1 > best[0]: best = (f1, {k: v.clone() for k, v in model.state_dict().items()}, ep); bad = 0
        else: bad += 1
        if ep % 5 == 0 or bad == 0: log(f"  seed{seed} ep{ep:02d} loss={tl/len(ytr):.4f} valF1={f1:.4f} best={best[0]:.4f}@{best[2]}")
        if bad >= patience: log(f"  seed{seed} early stop ep{ep}"); break
    model.load_state_dict(best[1]); model.eval()
    with torch.no_grad():
        return torch.softmax(model(Xva_t), 1).numpy(), torch.softmax(model(Xte_t), 1).numpy(), best[2], hist

def fit_svm(C, Xtr, ytr, Xva, Xte):
    t = time.time(); clf = SVC(C=C, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=0).fit(Xtr, ytr)
    assert list(clf.classes_) == [0, 1, 2]
    return C, clf.predict_proba(Xva), clf.predict_proba(Xte), clf.predict(Xva), time.time() - t

def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--feat", default="base"); ap.add_argument("--models", default="mlp,svm,knn")
    ap.add_argument("--seeds", type=int, default=3); a = ap.parse_args()
    logf = open(f"{RUN}/train_{a.feat}.log", "a")
    def log(s): print(s, flush=True); logf.write(s + "\n"); logf.flush()
    P = {s: json.load(open(f"{FM}/paths_{s}.json")) for s in SPLITS}
    ytr = np.array([C2I[l] for l in P["train"]["labels"]]); yva = np.array([C2I[l] for l in P["val"]["labels"]])
    F = load_feats(a.feat); log(f"=== feat={a.feat} dims={F['train'].shape} models={a.models} {time.strftime('%H:%M:%S')}")
    sc = StandardScaler().fit(F["train"]); Xtr, Xva, Xte = (sc.transform(F[s]).astype(np.float32) for s in SPLITS)
    resf = f"{RUN}/results.json"; results = json.load(open(resf)) if os.path.exists(resf) else {}
    def save(name, pva, pte, extra):
        m = metrics(yva, pva); m.update(extra); m["feat"] = a.feat; results[name] = m
        np.savez(f"{RUN}/probs/{name}.npz", val=pva.astype(np.float32), test=pte.astype(np.float32))
        json.dump(results, open(resf, "w"), indent=1)
        log(f"{name:45s} MacroF1={m['macro_f1']:.4f} perclass={ {k: round(v,4) for k,v in m['per_class_f1'].items()} } t={extra.get('seconds',0):.0f}s")

    if "mlp" in a.models:
        t0 = time.time(); pvas, ptes = [], []
        for seed in range(a.seeds):
            t1 = time.time(); pva, pte, bep, hist = train_mlp(Xtr, ytr, Xva, yva, Xte, seed, log=log)
            save(f"MLP[{a.feat}] seed={seed}", pva, pte, {"seconds": time.time() - t1, "best_epoch": bep, "val_f1_history": hist})
            pvas.append(pva); ptes.append(pte)
        if a.seeds > 1: save(f"MLP[{a.feat}] avg{a.seeds}seeds", np.mean(pvas, 0), np.mean(ptes, 0), {"seconds": time.time() - t0})
    if "knn" in a.models:
        Ntr, Nva, Nte = l2(F["train"]), l2(F["val"]), l2(F["test"])  # cosine on raw (unscaled) L2-normed concat
        for k in [5, 15]:
            t1 = time.time(); knn = KNeighborsClassifier(n_neighbors=k, weights="distance", metric="cosine", n_jobs=4).fit(Ntr, ytr)
            save(f"kNN[{a.feat}] k={k} cos dist-weighted", knn.predict_proba(Nva), knn.predict_proba(Nte), {"seconds": time.time() - t1})
    if "svm" in a.models:
        Cs = [1, 3, 10]
        outs = Parallel(n_jobs=3)(delayed(fit_svm)(C, Xtr, ytr, Xva, Xte) for C in Cs)
        for C, pva, pte, pred_dec, sec in outs:
            m_dec = f1_score(yva, pred_dec, average="macro")  # decision-function argmax (differs from Platt-prob argmax)
            save(f"SVM-RBF[{a.feat}] C={C} balanced (prob argmax)", pva, pte, {"seconds": sec, "macro_f1_decision_argmax": float(m_dec)})
    log(f"=== done feat={a.feat} {time.strftime('%H:%M:%S')}")

if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/emb-heads/finalize.py
"""Pick best variant (val Macro F1, argmax) from results.json, write val_probs.csv, test_probs.csv, submission.csv.
Usage: python finalize.py [--name "<variant name>"] (default: best by macro_f1)"""
import os, json, argparse
import numpy as np, pandas as pd
from sklearn.metrics import classification_report
RUN = "/Users/yopiesuryadi/lomba-cv/runs/emb-heads"; FM = "/Users/yopiesuryadi/lomba-cv/runs/fm-probe"
CLASSES = ["Aman", "Rusak", "Lainnya"]
ap = argparse.ArgumentParser(); ap.add_argument("--name", default=None); ap.add_argument("--suffix", default=""); a = ap.parse_args()
R = json.load(open(f"{RUN}/results.json")); R = {k: v for k, v in R.items() if not k.startswith("_")}
name = a.name or max(R, key=lambda k: R[k]["macro_f1"])
print(f"SELECTED: {name}  MacroF1={R[name]['macro_f1']:.4f}")
P = {s: json.load(open(f"{FM}/paths_{s}.json")) for s in ["val", "test"]}
z = np.load(f"{RUN}/probs/{name}.npz"); pva, pte = z["val"], z["test"]
yva = np.array([CLASSES.index(l) for l in P["val"]["labels"]])
print(classification_report(yva, pva.argmax(1), target_names=CLASSES, digits=4))
print("confusion (rows=true):\n", np.array(R[name]["cm"]))
pd.DataFrame({"path": P["val"]["paths"], "label": P["val"]["labels"], "p_Aman": pva[:, 0], "p_Rusak": pva[:, 1], "p_Lainnya": pva[:, 2]}).to_csv(f"{RUN}/val_probs{a.suffix}.csv", index=False)
fn = [os.path.basename(p) for p in P["test"]["paths"]]
pd.DataFrame({"filename": fn, "p_Aman": pte[:, 0], "p_Rusak": pte[:, 1], "p_Lainnya": pte[:, 2]}).to_csv(f"{RUN}/test_probs{a.suffix}.csv", index=False)
samp = pd.read_csv("/Users/yopiesuryadi/lomba-cv/data/sample_submission.csv")
sub = samp[["filename"]].merge(pd.DataFrame({"filename": fn, "label": [CLASSES[i] for i in pte.argmax(1)]}), on="filename", how="left")
assert len(sub) == 3057 and sub.label.notna().all(), (len(sub), sub.label.isna().sum())
sub.to_csv(f"{RUN}/submission{a.suffix}.csv", index=False)
print("test pred distribution:", sub.label.value_counts().to_dict())
(R.__setitem__("_selected", name) if not a.suffix else R.__setitem__("_alt"+a.suffix, name)); json.dump(R, open(f"{RUN}/results.json", "w"), indent=1)

In [ ]:
os.makedirs("runs/emb-heads/probs", exist_ok=True)   # train.py menyimpan probs/<varian>.npz tanpa membuat foldernya
# !{PY} runs/emb-heads/extract_multicrop.py          # tidak dipakai, lihat emb-r2
!{PY} runs/emb-heads/train.py --feat base --models mlp,svm,knn
!{PY} runs/emb-heads/finalize.py                                            # terbaik: SVM-RBF[base] C=10
!{PY} runs/emb-heads/finalize.py --name "MLP[base] avg3seeds" --suffix _mlp  # alternatif MLP untuk ensemble

## 8. ConvNeXt-tiny putaran 2 (`runs/convnext-r2`)

224px, 8 epoch (run asli dihentikan manual di epoch 8, checkpoint terbaik epoch 7), sampler lebih lembut,
letterbox, tanpa augmentasi noise/rotasi. Macro F1 val 0,868. **Dipakai di ensemble final.**

<details><summary><b>REPORT.md asli: <code>runs/convnext-r2/REPORT.md</code></b> (klik untuk buka)</summary>

#### convnext-r2 — putaran 2 fine-tune `convnext_tiny.fb_in22k_ft_in1k` (PyTorch, MPS)

Semua angka berasal dari eksekusi nyata pada 2026-10-04 (log: `train.log` / `train_stdout.log`, `predict.log`, `history.json`, `val_metrics.json`). Kode disalin dari `runs/timm-convnext/` lalu diadaptasi.

##### Perubahan dibanding putaran 1 (`runs/timm-convnext`, Macro F1 val TTA 0.8740)
| item | putaran 1 | putaran 2 (ini) |
|---|---|---|
| cache gambar | sisi pendek 256 (`cache_256/`) | **sisi pendek 320 baru** (`cache_320/`, 534 MB, 15.281 file, 11 s, 0 korup); cache lama tidak disentuh |
| resolusi | 192 | **224** (rencana 256 dibatalkan, lihat Waktu) |
| epoch | 5 | rencana 8 (cosine dijadwalkan 8), **7 epoch selesai**, dihentikan di tengah epoch 8 (batas waktu) |
| freeze | stem + 2 stage | **tidak ada, semua 27.82M parameter dilatih** |
| batch | 48 | 48 |
| augmentasi train | RRC scale 0.6-1, hflip, ColorJitter 0.3/0.3/0.2, RandomGrayscale 0.05, RandomErasing 0.15 | **RRC scale 0.7-1, hflip, ColorJitter 0.2/0.2/0.1 saja** (tanpa noise/rotasi/grayscale/erasing, karena audit data: Aman berisi salinan ber-noise/rotasi = jalan pintas) |
| eval transform | Resize 1.14x + CenterCrop | **Letterbox**: sisi panjang -> 224, pad ke 224x224 (fill abu-abu mean), seluruh gambar terlihat |
| sampler | WeightedRandomSampler count^-0.75 (Aman/Rusak/Lainnya ~45/34/21%) | **count^-0.5** (porsi per epoch ~55.5/32.4/12.2%) |
| sisanya | sama: AdamW wd 0.05, lr body 1e-4 / head 1e-3, warmup 1 epoch + cosine, CE label smoothing 0.1, drop_path 0.1, fp16 autocast + GradScaler, seed 42, num_workers 4, TTA orig+hflip |

Split `split/train.csv` / `split/val.csv` dipakai apa adanya; val tidak dipakai untuk latih.

##### Hasil val (1.833 gambar)
Per epoch, argmax tanpa TTA (`history.json`):

| epoch | train loss | Macro F1 | F1 Aman | F1 Rusak | F1 Lainnya | menit |
|---|---|---|---|---|---|---|
| 1 | 0.7682 | 0.7748 | 0.8506 | 0.6629 | 0.8108 | 4.0 |
| 2 | 0.5718 | 0.8028 | 0.8891 | 0.7266 | 0.7928 | 3.7 |
| 3 | 0.4981 | 0.8489 | 0.9305 | 0.7949 | 0.8214 | 5.9 |
| 4 | 0.4461 | 0.8418 | 0.9317 | 0.8085 | 0.7850 | 6.0 |
| 5 | 0.4146 | 0.8517 | 0.9336 | 0.8252 | 0.7963 | 6.9 |
| 6 | 0.3930 | 0.8615 | 0.9398 | 0.8410 | 0.8037 | 5.1 |
| 7 | 0.3842 | **0.8724** | 0.9453 | 0.8536 | 0.8182 | 4.0 |

Epoch 8 dihentikan pada step 1600/1728 (LR sudah ~2e-6, hampir nol); tidak ada evaluasi epoch 8. Confusion matrix epoch 7 tanpa TTA: `[[1261 57 2] [71 379 0] [16 2 45]]`.

**Final (best.pt epoch 7, TTA orig+hflip, argmax) — isi `val_probs.csv`:**
- Macro F1 val = **0.8677** (putaran 1: 0.8740)
- Per-class F1: Aman **0.9457** (p1 0.9243), Rusak **0.8536** (p1 0.8054), Lainnya **0.8037** (p1 0.8923)
- Precision/recall: Aman 0.935/0.957, Rusak 0.865/0.842, Lainnya **0.977/0.683**; accuracy 0.919 (p1 0.892)
- Confusion matrix (baris = true, kolom = pred; Aman, Rusak, Lainnya):
```
[[1263   56    1]
 [  71  379    0]
 [  17    3   43]]
```
TTA di sini malah menurunkan Macro F1 tipis (0.8724 -> 0.8677) karena 2 Lainnya tambahan jatuh ke Aman; pada Aman/Rusak TTA sedikit membantu.

Bacaan: Aman dan Rusak membaik jelas (kebocoran Aman->Rusak turun dari 124 ke 56; over-predict Rusak putaran 1 terkoreksi oleh sampler yang lebih lembut; accuracy 0.892 -> 0.919). Yang turun adalah **recall Lainnya** (43/63; 17 Lainnya diprediksi Aman) sementara precision Lainnya 0.977 (1 false positive). Penyebab paling mungkin: porsi Lainnya per epoch turun dari 21% ke 12% dan augmentasi kuat (erasing/grayscale) dilepas, jadi model lebih konservatif pada kelas minoritas. Model ini kurang percaya diri pada Lainnya, bukan salah arah.

Catatan untuk ensemble (bukan angka utama, karena di-tuning di val sendiri): override argmax dengan `p_Lainnya >= 0.30` menaikkan Macro F1 val menjadi 0.8812 (Aman 0.9460, Rusak 0.8555, Lainnya 0.8421); ambang 0.25 -> 0.8801, 0.20 -> 0.8776. `submission.csv` tetap memakai argmax murni.

##### Test
- `test_probs.csv` dan `submission.csv`: 3.057 baris data + header, set filename identik dengan `sample_submission.csv` (di-assert di `predict.py` dan dicek ulang terpisah), kolom `filename,label`, 0 file tidak terbaca, tanpa NaN.
- Distribusi prediksi test: Aman 1868 / Rusak 1136 / Lainnya 53 (putaran 1: 1673 / 1319 / 65). Porsi Rusak turun dari 43% ke 37%, konsisten dengan koreksi over-predict Rusak. Lainnya 1.7% (train 3.5%), konsisten dengan recall Lainnya yang rendah di val.

##### Waktu
- Cache 320: 11 s. Smoke test 30 step @256 bs40: 1.22 s/step -> 5.3 menit latih/epoch + 0.55 menit eval ≈ 5.8 menit/epoch -> 10 epoch ≈ 58 menit, jauh di atas 35 menit, jadi diturunkan ke **224 px, 8 epoch** (estimasi 4.2 menit/epoch ≈ 34 menit).
- Training mulai 13:56:14. Epoch 1-2: 0.95-1.04 s/step (3.7-4.0 menit/epoch) sesuai estimasi, lalu beban GPU dari agen lain naik: epoch 3-5 memakan 5.9-6.9 menit (~1.8 s/step); epoch 6-7 kembali 5.1 dan 4.0 menit; epoch 8 berjalan 0.86 s/step. 7 epoch selesai pada 14:31:48 (**35.6 menit** termasuk eval tiap epoch); trainer dihentikan 14:33:06 di step 1600/1728 epoch 8.
- Prediksi val + test dengan TTA: 81 s (selesai 14:34:57). Total training + prediksi final: **~38.7 menit** wall-clock dari mulai training (di luar cache 11 s, smoke ~1.2 menit, dan satu prediksi perantara yang dibuang, lihat Masalah 2).

##### Masalah dan keputusan
1. Target 256 px / 10 epoch tidak terkejar dalam 35-40 menit walau GPU lebih lega di awal; dipilih 224/8. Guard `--max_minutes` bawaan hanya menilai dari epoch 1 (saat beban masih ringan) sehingga tidak terpicu ketika beban naik di epoch 3-5.
2. **Kesalahan proses watchdog**: saat proyeksi 8 epoch menembus 40 menit, saya pasang watchdog `pkill -f "lomba-cv/bin/python train.py"` untuk berhenti setelah epoch 6, lalu menjalankan `predict.py` dengan best.pt epoch 6 (Macro F1 TTA 0.8663). Pola pkill itu ternyata tidak cocok (ps menampilkan path binary Python yang sudah di-resolve, bukan path venv), sehingga trainer terus berjalan dan menimpa `best.pt` dengan epoch 7. Setelah ketahuan, trainer dimatikan lewat PID, dan `predict.py` dijalankan ulang; **semua artefak (`val_probs.csv`, `test_probs.csv`, `submission.csv`, `val_metrics.json`) sekarang konsisten dengan `best.pt` epoch 7** (dicek: `predict.log` memuat "best.pt from epoch 7"). Hasil epoch 6 (0.8663) hanya tercatat di sini sebagai riwayat.
3. Epoch 8 dihentikan 128 step sebelum selesai demi batas 40 menit; LR saat itu sudah ~2e-6 sehingga bobot epoch 8 hampir pasti sangat dekat dengan epoch 7. Kurva masih landai naik (0.8615 -> 0.8724 di epoch 6 -> 7).
4. Sampler ^-0.5 memperbaiki Aman/Rusak tapi menekan recall Lainnya; nilai tengah (^-0.6) atau ambang Lainnya ~0.3 di tahap ensemble layak dicoba.
5. Tidak ada file korup; mekanisme skip+log tetap terpasang (`corrupt_files.log` kosong).

##### Cara menjalankan ulang
```
cd /Users/yopiesuryadi/lomba-cv/runs/convnext-r2
/Users/yopiesuryadi/.venvs/lomba-cv/bin/python prep_cache.py 6   # opsional, dibangun otomatis oleh train.py
/Users/yopiesuryadi/.venvs/lomba-cv/bin/python train.py          # default = 224px, 8 epoch, bs 48, sampler ^-0.5, tanpa freeze (~36-40 menit tergantung beban GPU)
/Users/yopiesuryadi/.venvs/lomba-cv/bin/python predict.py        # val_probs.csv, test_probs.csv, submission.csv
```
Catatan: run yang dilaporkan dihentikan manual di epoch 8; menjalankan ulang penuh akan menambah evaluasi epoch 8 dan bisa memilih checkpoint yang berbeda.
File: `train.py`, `predict.py`, `common.py` (dataset/transform, Letterbox), `prep_cache.py`, `best.pt` (epoch 7), `history.json`, `val_metrics.json`, `train.log`, `train_stdout.log`, `predict.log` (berisi dua run: epoch 6 lalu epoch 7), `cache_320/` (bisa dihapus).


</details>

In [ ]:
%%simpan runs/convnext-r2/common.py
import logging, random
import numpy as np, pandas as pd, torch
from PIL import Image, ImageFile
from torch.utils.data import Dataset
import torchvision.transforms as T

ImageFile.LOAD_TRUNCATED_IMAGES = True
CLASSES = ["Aman", "Rusak", "Lainnya"]
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def load_rgb(path, draft_size=None):
    """Open image, convert to RGB. JPEG draft mode speeds up decoding of huge files."""
    img = Image.open(path)
    if draft_size and img.format == "JPEG":
        img.draft("RGB", (draft_size, draft_size))
    return img.convert("RGB")

def train_tf(size):
    # r2: NO noise / heavy rotation / grayscale / erasing (data audit: "Aman" contains noisy/rotated copies -> shortcut).
    return T.Compose([
        T.RandomResizedCrop(size, scale=(0.7, 1.0)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(0.2, 0.2, 0.1),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])

class Letterbox:
    """Resize so the LONGER side == size, then pad to size x size (keeps whole image, no center-crop)."""
    def __init__(self, size, fill=(124, 116, 104)):
        self.size, self.fill = size, fill
    def __call__(self, img):
        w, h = img.size; s = self.size / max(w, h)
        nw, nh = max(1, round(w * s)), max(1, round(h * s))
        img = img.resize((nw, nh), Image.BICUBIC)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        canvas.paste(img, ((self.size - nw) // 2, (self.size - nh) // 2))
        return canvas

def eval_tf(size):
    return T.Compose([
        Letterbox(size),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])

class CsvDataset(Dataset):
    """paths + labels; corrupt files are logged and replaced by a random other sample (train)
    or by a zero tensor flagged via ok=0 (eval)."""
    def __init__(self, paths, labels, tf, size, train=True, logger=None, use_cache=True):
        self.paths, self.labels, self.tf, self.size, self.train = list(paths), list(labels), tf, size, train
        if use_cache:  # use pre-resized 320px copies when available (see prep_cache.py)
            from prep_cache import cache_path
            import os
            self.paths = [cache_path(p) if os.path.exists(cache_path(p)) else p for p in self.paths]
            self.n_cached = sum(1 for p in self.paths if p.startswith(os.path.dirname(cache_path(self.paths[0]))))
        self.log = logger or logging.getLogger("data")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]
        try:
            img = load_rgb(p, draft_size=self.size * 2)
            return self.tf(img), self.labels[i], 1
        except Exception as e:
            self.log.warning(f"CORRUPT/SKIP {p}: {e}")
            if self.train:
                return self[random.randrange(len(self))]
            return torch.zeros(3, self.size, self.size), self.labels[i], 0

In [ ]:
%%simpan runs/convnext-r2/prep_cache.py
"""One-time pre-resize cache: shorter side -> 320 px, RGB JPEG q92. Original data untouched.
Reason: raw images go up to 2816 px; decoding them each epoch made the dataloader the bottleneck (~30 img/s)."""
import os, sys, time, pandas as pd
from multiprocessing import Pool
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
ROOT = "/Users/yopiesuryadi/lomba-cv"
CACHE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "cache_320")
SIDE = 320

def cache_path(src):
    rel = os.path.relpath(src, os.path.join(ROOT, "data"))
    return os.path.join(CACHE, os.path.splitext(rel)[0] + ".jpg")

def work(src):
    dst = cache_path(src)
    if os.path.exists(dst): return (src, "ok")
    try:
        img = Image.open(src)
        if img.format == "JPEG": img.draft("RGB", (SIDE*2, SIDE*2))
        img = img.convert("RGB")
        w, h = img.size; s = SIDE / min(w, h)
        if s < 1: img = img.resize((max(1, round(w*s)), max(1, round(h*s))), Image.BICUBIC)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        img.save(dst, "JPEG", quality=92)
        return (src, "ok")
    except Exception as e:
        return (src, f"CORRUPT {e!r}")

if __name__ == "__main__":
    tr = pd.read_csv(f"{ROOT}/split/train.csv"); va = pd.read_csv(f"{ROOT}/split/val.csv")
    test = sorted(os.listdir(f"{ROOT}/data/Test_Set"))
    srcs = list(tr.path) + list(va.path) + [f"{ROOT}/data/Test_Set/{f}" for f in test if not f.startswith(".")]
    t = time.time(); bad = []
    with Pool(int(sys.argv[1]) if len(sys.argv) > 1 else 6) as p:
        for i, (s, st) in enumerate(p.imap_unordered(work, srcs, chunksize=16)):
            if st != "ok": bad.append((s, st))
            if i % 2000 == 0: print(i, f"{time.time()-t:.0f}s", flush=True)
    print(f"done {len(srcs)} files in {time.time()-t:.0f}s, corrupt: {len(bad)}")
    with open(os.path.join(os.path.dirname(CACHE), "corrupt_files.log"), "w") as f:
        for s, st in bad: f.write(f"{s}\t{st}\n")
    for b in bad: print(b)

In [ ]:
%%simpan runs/convnext-r2/train.py
"""convnext-r2: fine-tune timm convnext_tiny.fb_in22k_ft_in1k (3 classes) on MPS, 256px, all params trained. See REPORT.md.
Run: python train.py  (defaults = the run reported). Needs cache_320/ from prep_cache.py (built automatically if missing)."""
import argparse, time, math, logging, os, sys, json, random
import numpy as np, pandas as pd, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, confusion_matrix
from common import *

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="convnext_tiny.fb_in22k_ft_in1k")
    ap.add_argument("--img", type=int, default=224)
    ap.add_argument("--epochs", type=int, default=8)
    ap.add_argument("--max_minutes", type=float, default=36.0, help="after epoch 1, cut epochs so est. train time fits")
    ap.add_argument("--freeze_stages", type=int, default=0, help="freeze stem + first N convnext stages")
    ap.add_argument("--bs", type=int, default=48)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--head_lr", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--amp", type=int, default=1)
    ap.add_argument("--sampler_power", type=float, default=0.5, help="sample weight = count^-power (1=fully balanced)")
    ap.add_argument("--cache", type=int, default=1)
    ap.add_argument("--smoke", type=int, default=0, help="run N steps then exit (timing)")
    ap.add_argument("--out", default=os.path.dirname(os.path.abspath(__file__)))
    args = ap.parse_args()

    OUT = args.out; os.makedirs(OUT, exist_ok=True)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(f"{OUT}/train.log")])
    log = logging.getLogger("train")
    log.info(f"args: {vars(args)}")
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    dev = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    tr = pd.read_csv("/Users/yopiesuryadi/lomba-cv/split/train.csv")
    va = pd.read_csv("/Users/yopiesuryadi/lomba-cv/split/val.csv")
    ytr = tr.label.map(CLS2IDX).values; yva = va.label.map(CLS2IDX).values
    counts = np.bincount(ytr, minlength=3); log.info(f"train counts {dict(zip(CLASSES, counts.tolist()))}")

    if args.cache and not os.path.isdir(os.path.join(OUT, "cache_320")):
        log.info("building cache_320 ..."); os.system(f"{sys.executable} {os.path.join(OUT, 'prep_cache.py')} 6")

    # --- sampling choice: WeightedRandomSampler (NOT class-weighted CE). weight_i = count[y_i]^-power
    cls_w = counts.astype(float) ** (-args.sampler_power)
    samp_w = cls_w[ytr]
    eff = (cls_w * counts) / (cls_w * counts).sum()
    log.info(f"sampler power {args.sampler_power} -> expected class share per epoch {dict(zip(CLASSES, np.round(eff,3).tolist()))}")
    sampler = WeightedRandomSampler(torch.as_tensor(samp_w, dtype=torch.double), num_samples=len(ytr), replacement=True)

    ds_tr = CsvDataset(tr.path, ytr, train_tf(args.img), args.img, True, use_cache=bool(args.cache))
    ds_va = CsvDataset(va.path, yva, eval_tf(args.img), args.img, False, use_cache=bool(args.cache))
    log.info(f"image source example: {ds_tr.paths[0]}")
    dl_tr = DataLoader(ds_tr, batch_size=args.bs, sampler=sampler, num_workers=args.workers, drop_last=True, persistent_workers=True, prefetch_factor=4)
    dl_va = DataLoader(ds_va, batch_size=96, shuffle=False, num_workers=args.workers)

    model = timm.create_model(args.model, pretrained=True, num_classes=3, drop_path_rate=0.1).to(dev)
    if args.freeze_stages > 0 and hasattr(model, "stages"):
        for mod in [model.stem] + list(model.stages[:args.freeze_stages]):
            for p in mod.parameters(): p.requires_grad = False
    head_params = list(model.get_classifier().parameters()); head_ids = {id(p) for p in head_params}
    body_params = [p for p in model.parameters() if id(p) not in head_ids and p.requires_grad]
    log.info(f"trainable params: {sum(p.numel() for p in body_params+head_params)/1e6:.2f}M / {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
    opt = torch.optim.AdamW([{"params": body_params, "lr": args.lr}, {"params": head_params, "lr": args.head_lr}], weight_decay=args.wd)
    steps_per_epoch = len(dl_tr); S = {"total": steps_per_epoch * args.epochs}; warm = steps_per_epoch
    def lr_lambda(s):
        if s < warm: return (s + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, S["total"] - warm))))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    use_amp = bool(args.amp) and dev.type == "mps"
    scaler = torch.amp.GradScaler("mps", enabled=use_amp)

    @torch.no_grad()
    def evaluate():
        model.eval(); probs = []
        for x, y, ok in dl_va:
            with torch.autocast("mps", dtype=torch.float16, enabled=use_amp):
                lo = model(x.to(dev))
            probs.append(lo.float().softmax(1).cpu())
        P = torch.cat(probs).numpy(); pred = P.argmax(1)
        f1 = f1_score(yva, pred, average="macro"); pc = f1_score(yva, pred, average=None)
        return f1, pc, confusion_matrix(yva, pred), P

    best = -1; hist = []; t0 = time.time(); step = 0
    n_epochs = args.epochs; ep = -1
    while ep + 1 < n_epochs:
        ep += 1
        model.train(); te = time.time(); run = 0.0; n = 0; nan_hit = False
        for x, y, _ in dl_tr:
            x, y = x.to(dev, non_blocking=True), y.to(dev)
            with torch.autocast("mps", dtype=torch.float16, enabled=use_amp):
                loss = crit(model(x), y)
            if not torch.isfinite(loss):
                nan_hit = True; log.warning(f"non-finite loss at step {step}, skipping batch")
                opt.zero_grad(set_to_none=True); step += 1; sched.step(); continue
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step(); step += 1
            run += loss.item(); n += 1
            if step % 50 == 0:
                log.info(f"ep{ep+1} step {step}/{S['total']} loss {run/n:.4f} lr {sched.get_last_lr()[0]:.2e} {(time.time()-te)/n:.3f}s/step")
            if args.smoke and step >= args.smoke:
                el = time.time() - te
                log.info(f"SMOKE: {step} steps in {el:.1f}s -> {el/step:.3f}s/step -> est {el/step*steps_per_epoch/60:.1f} min/epoch train")
                tv = time.time(); f1, pc, cm, _ = evaluate(); log.info(f"SMOKE val pass {time.time()-tv:.1f}s f1 {f1:.4f}")
                sys.exit(0)
        f1, pc, cm, P = evaluate()
        ep_t = time.time() - te
        if ep == 0:
            fit = int(args.max_minutes * 60 // ep_t)
            if fit < n_epochs:
                log.warning(f"epoch took {ep_t/60:.1f} min; cutting epochs {n_epochs} -> {max(2, fit)} to fit {args.max_minutes} min")
                n_epochs = max(2, fit); S["total"] = steps_per_epoch * n_epochs
        rec = {"epoch": ep+1, "train_loss": run/max(n,1), "macro_f1": float(f1), "per_class_f1": [float(v) for v in pc], "cm": cm.tolist(), "epoch_sec": ep_t, "nan": nan_hit, "n_epochs_planned": n_epochs}
        hist.append(rec)
        log.info(f"EPOCH {ep+1}/{n_epochs} loss {rec['train_loss']:.4f} macroF1 {f1:.4f} perclass {np.round(pc,4).tolist()} time {ep_t/60:.1f}min\nCM:\n{cm}")
        if f1 > best:
            best = f1; torch.save({"model": model.state_dict(), "args": vars(args), "epoch": ep+1, "macro_f1": float(f1)}, f"{OUT}/best.pt")
            log.info(f"  saved best.pt (epoch {ep+1}, F1 {f1:.4f})")
        json.dump({"history": hist, "best_macro_f1": best, "total_train_sec": time.time()-t0}, open(f"{OUT}/history.json", "w"), indent=1)
    log.info(f"DONE total {(time.time()-t0)/60:.1f} min, best macroF1 {best:.4f}")

if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/convnext-r2/predict.py
"""Predict val + Test_Set with best.pt, TTA = orig + hflip (mean softmax).
Writes val_probs.csv, test_probs.csv, submission.csv."""
import os, sys, time, logging, numpy as np, pandas as pd, torch, timm
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from common import *

ROOT = "/Users/yopiesuryadi/lomba-cv"
OUT = os.path.dirname(os.path.abspath(__file__))

@torch.no_grad()
def predict(model, dl, dev, use_amp=True):
    probs, oks = [], []
    for x, _, ok in dl:
        x = x.to(dev)
        with torch.autocast("mps", dtype=torch.float16, enabled=use_amp):
            p = model(x).float().softmax(1) + model(torch.flip(x, dims=[3])).float().softmax(1)
        probs.append((p / 2).cpu()); oks.append(ok)
    P = torch.cat(probs).numpy(); ok = torch.cat(oks).numpy()
    P[ok == 0] = 1.0 / 3  # unreadable file -> uniform
    return P, ok

def main():
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(f"{OUT}/predict.log")])
    log = logging.getLogger("predict")
    dev = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    ck = torch.load(f"{OUT}/best.pt", map_location="cpu", weights_only=False)
    args = ck["args"]; size = args["img"]
    log.info(f"best.pt from epoch {ck['epoch']} val macroF1 {ck['macro_f1']:.4f}; img {size}")
    model = timm.create_model(args["model"], pretrained=False, num_classes=3)
    model.load_state_dict(ck["model"]); model.to(dev).eval()
    t0 = time.time()

    # --- val
    va = pd.read_csv(f"{ROOT}/split/val.csv"); yva = va.label.map(CLS2IDX).values
    ds = CsvDataset(va.path, yva, eval_tf(size), size, False, use_cache=bool(args.get("cache", 1)))
    P, ok = predict(model, DataLoader(ds, batch_size=96, num_workers=4), dev)
    pred = P.argmax(1)
    f1 = f1_score(yva, pred, average="macro"); pc = f1_score(yva, pred, average=None); cm = confusion_matrix(yva, pred)
    log.info(f"VAL (TTA) macroF1 {f1:.4f} per-class {dict(zip(CLASSES, np.round(pc,4).tolist()))}\nCM (rows=true, cols=pred, order {CLASSES}):\n{cm}")
    log.info("\n" + classification_report(yva, pred, target_names=CLASSES, digits=4))
    pd.DataFrame({"path": va.path, "label": va.label, "p_Aman": P[:,0], "p_Rusak": P[:,1], "p_Lainnya": P[:,2]}).to_csv(f"{OUT}/val_probs.csv", index=False)
    json_metrics = {"val_macro_f1_tta": float(f1), "per_class_f1": dict(zip(CLASSES, [float(v) for v in pc])), "cm": cm.tolist(), "unreadable_val": int((ok==0).sum())}

    # --- test
    files = sorted(f for f in os.listdir(f"{ROOT}/data/Test_Set") if not f.startswith("."))
    paths = [f"{ROOT}/data/Test_Set/{f}" for f in files]
    ds = CsvDataset(paths, [0]*len(paths), eval_tf(size), size, False, use_cache=bool(args.get("cache", 1)))
    P, ok = predict(model, DataLoader(ds, batch_size=96, num_workers=4), dev)
    log.info(f"TEST: {len(files)} files, unreadable {(ok==0).sum()}, pred dist {dict(zip(CLASSES, np.bincount(P.argmax(1), minlength=3).tolist()))}")
    pd.DataFrame({"filename": files, "p_Aman": P[:,0], "p_Rusak": P[:,1], "p_Lainnya": P[:,2]}).to_csv(f"{OUT}/test_probs.csv", index=False)
    sub = pd.DataFrame({"filename": files, "label": [CLASSES[i] for i in P.argmax(1)]})
    sub.to_csv(f"{OUT}/submission.csv", index=False)
    # sanity vs sample_submission
    ss = pd.read_csv(f"{ROOT}/data/sample_submission.csv")
    assert len(sub) == 3057 and set(sub.filename) == set(ss.filename), (len(sub), len(set(sub.filename) ^ set(ss.filename)))
    json_metrics["test_pred_dist"] = dict(zip(CLASSES, np.bincount(P.argmax(1), minlength=3).tolist()))
    json_metrics["predict_sec"] = time.time() - t0
    import json; json.dump(json_metrics, open(f"{OUT}/val_metrics.json", "w"), indent=1)
    log.info(f"wrote val_probs.csv test_probs.csv submission.csv in {time.time()-t0:.0f}s")

if __name__ == "__main__":
    main()

In [ ]:
!{PY} runs/convnext-r2/train.py
!{PY} runs/convnext-r2/predict.py

## 9. CLIP ViT-L/14 + multi-crop 448 (`runs/emb-r2`)

- `extract_l14.py --half --nw 0`: embedding ViT-L/14 (fp16, decode in-process; worker DataLoader lambat di macOS).
- `extract_multicrop.py --half --nw 0`: 5 crop 224 dari 448px, CLIP ViT-B/16.
- `extract_dino448.py`: saat lomba hanya dipakai untuk mengukur laju (`--probe`), tidak ada embedding yang dipakai.
- `train.py` per kombinasi fitur, lalu `--finalize`. Varian terpilih `SVM-RBF[l14+base] C=30` (0,922);
  alternatif `_mlp` = **MLP[mc+base] avg3 (0,921, bersih 0,894), yang dipakai di ensemble final** sebagai "MLP multi-crop".

`--half` ditujukan untuk GPU (MPS/CUDA); di CPU hapus flag itu.

<details><summary><b>REPORT.md asli: <code>runs/emb-r2/REPORT.md</code></b> (klik untuk buka)</summary>

#### emb-r2: embedding backbone lebih besar (OpenCLIP ViT-L-14) + head SVM/MLP, dengan dan tanpa fitur putaran 1

Agen: `emb-r2`. Lanjutan `fm-probe` (ekstraksi CLIP ViT-B-16 + DINOv2-S/14, 896-d) dan `emb-heads` (SVM RBF 0.9123 / MLP 0.9087 di val). Hipotesis yang diuji: fitur global dari backbone lebih besar (ViT-L-14, 768-d) dan/atau fitur multi-crop 448 px menolong kelas Rusak. Semua baris mengikuti urutan `runs/fm-probe/paths_{train,val,test}.json`, jadi concat dengan 896-d putaran 1 langsung cocok (paths json disalin ke folder ini).

##### Kronologi dan keputusan (waktu nyata, Apple M5 mps, GPU dibagi dengan satu agen training lain)
- 13:53 mulai; unduh bobot `ViT-L-14 laion2b_s32b_b82k` 88 s.
- Probe 256 gambar train (`extract_l14.py --probe 256`, DataLoader 4 worker): fp32 **6,4 gambar/s -> proyeksi 39,9 menit**; fp16 8,2 gambar/s -> 30,9 menit. Probe multi-crop ViT-B-16 fp16 (5 crop 224 dari 448): 8,4 gambar/s -> 30,5 menit. Probe DINOv2-S/14 448 px fp16: 10,0 gambar/s -> 25,4 menit. Semua lewat batas 20 menit, dan fallback multi-crop ternyata tidak lebih murah (5 x 17,6 GFLOPs ~ 88 GFLOPs/gambar vs ViT-L-14 ~81 GFLOPs).
- Sebelum menyerah, dicek penyebabnya: loader SAJA tanpa model 10,8 gambar/s (8 worker malah 7,4), CPU 77% idle; decode+transform in-process satu thread 242 gambar/s; GPU murni ViT-L-14 fp16 (tensor sintetis, batch 48) **38,5 gambar/s** (fp32 13,6). Jadi bottleneck probe adalah worker DataLoader (spawn di macOS), bukan GPU. Diperbaiki dengan `--nw 0` (decode in-process).
- 14:05-14:12 ekstraksi ViT-L-14 fp16, batch 48, nw 0: train 10.391 gambar 303 s, val 1.833 gambar 59 s, test 3.057 gambar 100 s (~33 gambar/s, total 7,7 menit). 0 file korup (`bad_files_l14.log` kosong). Embedding 768-d L2-normed, dicek: tanpa NaN, tanpa baris nol.
- 14:13 classifier L-14 dan L-14+896 berjalan paralel di CPU; bersamaan ekstraksi multi-crop fp16 (`extract_multicrop.py --half --nw 0`, ~19 gambar/s karena 5 crop per gambar) dijalankan di GPU sebagai poin 2.

Catatan fp16: ViT dijalankan half precision di mps, output di-cast ke float32 lalu L2-normalisasi. Probe fp32 vs fp16 tidak dibandingkan akurasinya (waktu); untuk fitur beku yang dinormalisasi perbedaan numeriknya kecil, tetapi ini belum diverifikasi angka.

##### Setup classifier (`train.py`)
- Fitur: (a) `l14` 768-d; (b) `l14+base` 1664-d = L-14 concat [CLIP B-16 512 + DINOv2-S 384 L2-normed] dari fm-probe. `StandardScaler` fit di train. Train hanya `split/train.csv`; val tidak pernah dipakai untuk fit bobot/scaler.
- SVM: `SVC(kernel='rbf', gamma='scale', class_weight='balanced', probability=True)`, C 3/10/30, 3 nilai C paralel (joblib). Probabilitas Platt (CV internal 5-fold), argmax prob.
- MLP: sama persis dengan emb-heads (Dropout 0.1 -> Linear d->512 -> GELU -> Dropout 0.3 -> Linear 512->3; CE label smoothing 0.1 bobot kelas sqrt-balanced; AdamW lr 1e-3 wd 5e-2 OneCycle; batch 256; maks 60 epoch, early stop patience 12 pada Macro F1 val, bobot epoch terbaik), 3 seed, softmax dirata-rata.
- Val bersih: val tanpa pasangan md5 atau phash<=6 ke train (aturan `ensemble.py`, dari `runs/eda/meta.csv`): **1327/1833** (Aman 1021, Rusak 246, Lainnya 60). Pemilihan varian memakai Macro F1 val penuh (sesuai tugas); val bersih dilaporkan di sampingnya.
- Pembanding putaran 1 pada val bersih yang sama (`baseline_clean_val.json`, dihitung dari val_probs.csv masing-masing): emb-heads SVM C=10 full 0.9123 / **bersih 0.8840** (Rusak 0.758); emb-heads MLP avg3 0.9087 / bersih 0.8787 (Rusak 0.782); fm-probe LR 0.8411 / bersih 0.8264.

##### Hasil semua varian (val penuh 1833 / val bersih 1327; argmax; angka dari `results_*.json`)

| Varian | Macro F1 val | Macro F1 bersih | F1 Aman | F1 Rusak | F1 Lainnya | Rusak bersih | Lainnya bersih | Waktu fit |
|---|---|---|---|---|---|---|---|---|
| **SVM-RBF[l14+base] C=30 balanced** | **0.9219** | 0.8893 | 0.9554 | 0.8681 | 0.9421 | 0.7795 | 0.9391 | 574 s |
| MLP[l14+base] seed=1 | 0.9187 | 0.8912 | 0.9556 | 0.8714 | 0.9291 | 0.7897 | 0.9333 | 16 s |
| MLP[l14+base] avg3 | 0.9161 | 0.8872 | 0.9532 | 0.8658 | 0.9291 | 0.7806 | 0.9333 | 43 s |
| SVM-RBF[l14+base] C=10 balanced | 0.9158 | 0.8855 | 0.9528 | 0.8602 | 0.9344 | 0.7692 | 0.9391 | 559 s |
| MLP[l14+base] seed=0 | 0.9146 | 0.8857 | 0.9535 | 0.8696 | 0.9206 | 0.7860 | 0.9244 | 11 s |
| SVM-RBF[l14] C=30 balanced | 0.9144 | 0.8804 | 0.9508 | 0.8504 | 0.9421 | 0.7563 | 0.9391 | 464 s |
| MLP[l14] seed=0 | 0.9137 | 0.8887 | 0.9507 | 0.8614 | 0.9291 | 0.7787 | 0.9412 | 9 s |
| MLP[l14] seed=2 | 0.9131 | 0.8828 | 0.9489 | 0.8549 | 0.9355 | 0.7653 | 0.9402 | 7 s |
| MLP[l14+base] seed=2 | 0.9127 | 0.8859 | 0.9538 | 0.8696 | 0.9147 | 0.7909 | 0.9180 | 16 s |
| SVM-RBF[l14+base] C=3 balanced | 0.9119 | 0.8822 | 0.9498 | 0.8515 | 0.9344 | 0.7613 | 0.9391 | 492 s |
| SVM-RBF[l14] C=3 balanced | 0.9099 | 0.8828 | 0.9483 | 0.8471 | 0.9344 | 0.7638 | 0.9391 | 353 s |
| SVM-RBF[l14] C=10 balanced | 0.9094 | 0.8792 | 0.9485 | 0.8453 | 0.9344 | 0.7540 | 0.9391 | 434 s |
| MLP[l14] avg3 | 0.9081 | 0.8777 | 0.9486 | 0.8552 | 0.9206 | 0.7656 | 0.9244 | 24 s |
| MLP[l14] seed=1 | 0.9060 | 0.8769 | 0.9480 | 0.8565 | 0.9134 | 0.7712 | 0.9167 | 8 s |
| (pembanding) emb-heads SVM-RBF[base 896] C=10 | 0.9123 | 0.8840 | 0.9507 | 0.8528 | 0.9333 | 0.7580 | 0.9474 | - |
| (pembanding) emb-heads MLP[base 896] avg3 | 0.9087 | 0.8787 | 0.9528 | 0.8684 | 0.9048 | 0.7824 | 0.9076 | - |

##### Varian terpilih: SVM-RBF[l14+base] C=30 balanced (dipakai untuk `val_probs.csv`, `test_probs.csv`, `submission.csv`)

Macro F1 val = **0.9219** (accuracy 0.9345); Macro F1 val bersih = **0.8893**. Dibanding emb-heads SVM C=10 pada 896-d: +0.0096 val penuh, +0.0053 val bersih; F1 Rusak 0.8528 -> 0.8681 (bersih 0.758 -> 0.780).

| Kelas | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Aman | 0.9386 | 0.9727 | 0.9554 | 1320 |
| Rusak | 0.9140 | 0.8267 | 0.8681 | 450 |
| Lainnya | 0.9828 | 0.9048 | 0.9421 | 63 |

Confusion matrix val penuh (baris = true, kolom = pred; Aman, Rusak, Lainnya):

```
            pred Aman  pred Rusak  pred Lainnya
true Aman        1284          35             1
true Rusak         78         372             0
true Lainnya        6           0            57
```

Confusion matrix val bersih (1327):

```
true Aman         992          28             1
true Rusak         71         175             0
true Lainnya        6           0            54
```

Distribusi prediksi test (3057): Aman 1845, Rusak 1157, Lainnya 55. `submission.csv` dicek: 3057 baris, nama file dan urutan sama persis dengan `sample_submission.csv`. Kesepakatan label test dengan emb-heads SVM: 96,5%; dengan MLP[l14+base] avg3 run ini: 97,2%.

###### Alternatif: MLP[l14+base] avg3 (`val_probs_mlp.csv`, `test_probs_mlp.csv`, `submission_mlp.csv`)
Macro F1 val 0.9161, bersih 0.8872 (F1 Rusak 0.8658; bersih 0.7806). Softmax langsung (bukan Platt), karakter sama seperti di emb-heads: recall Rusak lebih tinggi, precision lebih rendah. Distribusi test: Aman 1772, Rusak 1226, Lainnya 59.

##### Catatan
- **Perbaikan nyata tetapi kecil.** Backbone 4,6x lebih besar (ViT-L-14) sendiri (`l14`, 768-d) kurang lebih setara 896-d putaran 1 (SVM 0.9099-0.9144 vs 0.9123; bersih 0.879-0.883 vs 0.884). Nilainya muncul saat di-concat: `l14+base` menang di semua head (SVM C=30 0.9219, MLP avg3 0.9161). Fitur ViT-L-14 dan B-16+DINO saling melengkapi, bukan saling menggantikan.
- Kelemahan Rusak **tidak terpecahkan** oleh fitur global yang lebih besar: F1 Rusak bersih 0.780 (dari 0.758); sumber error tetap Rusak -> Aman (71 dari 246 Rusak bersih, recall ~0.71). Ini mendukung dugaan bahwa kerusakan kecil hilang pada representasi global 224 px, bukan karena kapasitas backbone. Jalur yang lebih menjanjikan untuk Rusak: fine-tune resolusi lebih tinggi atau fitur lokal (multi-crop/patch), bukan embedding global yang lebih besar.
- Val bersih konsisten ~0.03 di bawah val penuh untuk semua varian (near-duplicate ke train menaikkan angka val penuh). Selisih antar varian teratas (0.9219 vs 0.9187 vs 0.9161) lebih kecil daripada sebaran antar-seed MLP (0.9127-0.9187), jadi urutan di puncak tidak signifikan; pilihan C=30 dan seed memakai val (bias seleksi kecil, konsisten dengan emb-heads).
- C=30 lebih baik dari C=10 di kedua set fitur; C lebih besar (100) belum dicoba karena waktu.
- sklearn `SVC(probability=True)` FutureWarning (deprecated) muncul, berjalan normal. SVM 1664-d: 492-574 s per C (3 C paralel, bersamaan dengan 3 SVM 768-d dan ekstraksi multi-crop).

##### Waktu
- Unduh bobot 88 s; probe/diagnosis loader ~9 menit (13:55-14:04); ekstraksi ViT-L-14 7,7 menit (14:05-14:12); classifier L-14 dan L-14+base paralel 14:13-14:24 (MLP 3 seed 24-43 s; SVM 353-574 s per C); finalize + verifikasi < 10 s. Total sampai output utama: 13:53 -> 14:25 (~32 menit).

##### File
- `extract_l14.py` — ekstraksi ViT-L-14 (`--probe N`, `--half`, `--nw`); `probe.log`, `extract_l14.log`, `bad_files_l14.log` (kosong).
- `emb_l14_{train,val,test}.npy` (768-d, L2-normed), `paths_{train,val,test}.json` (salinan fm-probe, urutan baris).
- `train.py` — classifier (`--feat l14|l14+base|mc+base|l14+mc+base --models svm,mlp`), `--finalize [--name X --suffix _mlp]`; log `train_<feat>.log`/`.out`, `train_finalize.log`; metrik `results_<feat>.json` + `results.json` (`_selected`, `_selected_mlp`); probabilitas tiap varian `probs/<nama>.npz`.
- `baseline_clean_val.json` — Macro F1 bersih pembanding putaran 1.
- `extract_multicrop.py` — salinan emb-heads + `--probe/--half/--nw`; `extract_mc.log`, `emb_clipmc_*.npy` (lihat bagian multi-crop di bawah).

##### Poin 2: CLIP ViT-B-16 multi-crop (selesai; hanya MLP karena waktu)

Ekstraksi `extract_multicrop.py --half --nw 0` (resize sisi pendek 448, center crop 448, 4 kuadran 224 + center 224, 5 embedding 512-d per gambar, L2-normed; fitur `mc` = rata-rata 5 crop lalu L2-normed): train 537 s, val 81 s, test 137 s (12,6 menit, ~20 gambar/s, bersamaan dengan 6 SVM di CPU), 0 file korup. `emb_clipmc_{train,val,test}.npy` (N,5,512) tersimpan di folder ini dan bisa dipakai agen lain. SVM pada fitur ini TIDAK dijalankan (butuh ~10 menit lagi, lewat batas 40 menit).

| Varian | Macro F1 val | Macro F1 bersih | F1 Aman | F1 Rusak | F1 Lainnya | Rusak bersih | Lainnya bersih | Waktu fit |
|---|---|---|---|---|---|---|---|---|
| MLP[l14+mc+base] seed=1 | 0.9227 | 0.9011 | 0.9605 | 0.8884 | 0.9194 | 0.8235 | 0.9231 | 9 s |
| MLP[l14+mc+base] seed=2 | 0.9210 | 0.8964 | 0.9547 | 0.8719 | 0.9365 | 0.7975 | 0.9412 | 7 s |
| MLP[mc+base] avg3 | 0.9207 | 0.8937 | 0.9570 | 0.8771 | 0.9280 | 0.8042 | 0.9244 | 17 s |
| MLP[l14+mc+base] seed=0 | 0.9205 | 0.8954 | 0.9563 | 0.8774 | 0.9280 | 0.8025 | 0.9322 | 8 s |
| MLP[l14+mc+base] avg3 | 0.9191 | 0.8951 | 0.9574 | 0.8791 | 0.9206 | 0.8076 | 0.9244 | 24 s |
| MLP[mc+base] seed=1 | 0.9183 | 0.8944 | 0.9574 | 0.8768 | 0.9206 | 0.8059 | 0.9244 | 5 s |
| MLP[mc+base] seed=2 | 0.9177 | 0.8899 | 0.9548 | 0.8703 | 0.9280 | 0.7949 | 0.9244 | 6 s |
| MLP[mc+base] seed=0 | 0.9145 | 0.8881 | 0.9558 | 0.8742 | 0.9134 | 0.7966 | 0.9167 | 6 s |

Multi-crop adalah fitur baru yang paling menolong Rusak: MLP[mc+base] avg3 F1 Rusak 0.8771 (bersih **0.8042**, dari 0.758 emb-heads SVM / 0.780 SVM-RBF[l14+base] C=30), Macro F1 bersih 0.8937 (> 0.8893 varian terpilih), meski Macro F1 val penuh 0.9207 sedikit di bawah 0.9219 (selisih dalam noise antar-seed). Menambah L-14 di atasnya (`l14+mc+base`, 2176-d) tidak menambah apa-apa yang konsisten (avg3 0.9191 / bersih 0.8951). Ini menguatkan kesimpulan: detail lokal 448 px lebih berharga untuk Rusak daripada backbone global lebih besar.
Confusion matrix MLP[mc+base] avg3 val penuh (baris true Aman/Rusak/Lainnya): [[1270, 48, 2], [59, 389, 2], [5, 0, 58]]; val bersih: [[981, 38, 2], [53, 191, 2], [5, 0, 55]].

**Keputusan output**: sesuai aturan tugas (pilih berdasarkan Macro F1 val penuh) `submission.csv` tetap SVM-RBF[l14+base] C=30 (0.9219). File alternatif `_mlp` DIGANTI ke **MLP[mc+base] avg3** (`val_probs_mlp.csv`, `test_probs_mlp.csv`, `submission_mlp.csv`; `_selected_mlp` di results.json), menggantikan MLP[l14+base] avg3 yang disebut di bagian sebelumnya, karena lebih baik di val penuh (0.9207 vs 0.9161) dan val bersih. Distribusi test MLP[mc+base] avg3: {'Aman': 1749, 'Rusak': 1250, 'Lainnya': 58}; 3057 baris, urutan sama dengan sample_submission; kesepakatan label dengan submission.csv 95.7%. Untuk ensemble, `val_probs_mlp.csv` ini layak diberi bobot setara atau lebih besar daripada `val_probs.csv`.

Total waktu agen: 13:53 -> 14:29 (~36 menit). results.json: `_selected` = SVM-RBF[l14+base] C=30 balanced, `_selected_mlp` = MLP[mc+base] avg3.

- `extract_dino448.py` — hanya dipakai untuk probe laju DINOv2-S/14 448 px (10,0 gambar/s, lihat `probe.log`); tidak ada embedding yang diekstrak dengannya.


</details>

In [ ]:
%%simpan runs/emb-r2/extract_l14.py
"""emb-r2: OpenCLIP ViT-L-14 (laion2b_s32b_b82k) image embeddings, 768-d L2-normed, 224 px, same
row order as runs/fm-probe/paths_{split}.json. --probe N: time first N train images and print the
projection for all 15281 images (exit 2 if projection > --budget minutes). Without --probe: extract all. --half runs the ViT in fp16 on mps. --nw 0 (default) decodes in-process:
spawned DataLoader workers were the bottleneck on this Mac (10 img/s loader-only vs 242 img/s in-process decode)."""
import os, json, time, argparse, shutil
import numpy as np, torch
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
ImageFile.LOAD_TRUNCATED_IMAGES = True
RUN = "/Users/yopiesuryadi/lomba-cv/runs/emb-r2"; FM = "/Users/yopiesuryadi/lomba-cv/runs/fm-probe"
DEV = "mps" if torch.backends.mps.is_available() else "cpu"

class DS(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]); img.load(); return self.tf(img.convert("RGB")), i, 1
        except Exception:
            return torch.zeros(3, 224, 224), i, 0

@torch.no_grad()
def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--probe", type=int, default=0); ap.add_argument("--budget", type=float, default=20)
    ap.add_argument("--bs", type=int, default=48); ap.add_argument("--half", action="store_true"); ap.add_argument("--nw", type=int, default=0); ap.add_argument("--splits", default="train,val,test"); a = ap.parse_args()
    import open_clip
    t = time.time(); model, _, tf = open_clip.create_model_and_transforms("ViT-L-14", pretrained="laion2b_s32b_b82k")
    model = model.to(DEV).eval()
    if a.half: model = model.half()
    dt_ = torch.float16 if a.half else torch.float32
    print(f"model loaded {time.time()-t:.0f}s dev={DEV}", flush=True)
    P = {s: json.load(open(f"{FM}/paths_{s}.json"))["paths"] for s in ["train", "val", "test"]}
    ntotal = sum(len(v) for v in P.values())
    if a.probe:
        paths = P["train"][:a.probe]; dl = DataLoader(DS(paths, tf), batch_size=a.bs, num_workers=a.nw)
        x, _, _ = next(iter(dl)); model.encode_image(x.to(DEV, dt_)); torch.mps.synchronize()  # warm-up
        t0 = time.time(); n = 0
        for x, idx, ok in dl: f = model.encode_image(x.to(DEV, dt_)); f.float().cpu(); n += len(idx)
        dt = time.time() - t0; rate = n / dt; proj = ntotal / rate / 60
        print(f"PROBE {n} imgs {dt:.1f}s -> {rate:.1f} img/s; projection {ntotal} imgs = {proj:.1f} min (budget {a.budget})", flush=True)
        raise SystemExit(0 if proj <= a.budget else 2)
    bad = open(f"{RUN}/bad_files_l14.log", "a")
    for split in a.splits.split(","):
        out = f"{RUN}/emb_l14_{split}.npy"
        if os.path.exists(out): print(f"{split} exists, skip", flush=True); continue
        paths = P[split]; dl = DataLoader(DS(paths, tf), batch_size=a.bs, num_workers=a.nw, shuffle=False)
        feats = np.zeros((len(paths), 768), np.float32); t0 = time.time(); nbad = 0
        for b, (x, idx, ok) in enumerate(dl):
            f = model.encode_image(x.to(DEV, dt_)).float(); f = (f / f.norm(dim=-1, keepdim=True)).cpu().numpy()
            feats[idx.numpy()] = f
            for j, i in enumerate(idx.tolist()):
                if ok[j].item() == 0: nbad += 1; bad.write(f"{split}\t{paths[i]}\n")
            if b % 25 == 0: print(f"{split} {b*a.bs}/{len(paths)} {time.time()-t0:.0f}s", flush=True)
        np.save(out, feats); shutil.copy(f"{FM}/paths_{split}.json", f"{RUN}/paths_{split}.json")
        print(f"{split} done {feats.shape} in {time.time()-t0:.0f}s bad={nbad}", flush=True)
    bad.close()

if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/emb-r2/extract_multicrop.py
"""emb-r2 copy of emb-heads/extract_multicrop.py (+ --probe, --half fp16, output in runs/emb-r2).
CLIP ViT-B-16 (laion2b_s34b_b88k) multi-crop embeddings: shortest side -> 448, center crop 448,
then 4 quadrants 224 + center 224 = 5 crops. Each crop encoded + L2-normalized; saves per-crop
embeddings (N,5,512) so mean/max pooling can be chosen later. Same split lists/order as fm-probe."""
import os, json, time, argparse
import numpy as np, pandas as pd, torch
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
ImageFile.LOAD_TRUNCATED_IMAGES = True

RUN = "/Users/yopiesuryadi/lomba-cv/runs/emb-r2"
FM = "/Users/yopiesuryadi/lomba-cv/runs/fm-probe"
DEV = "mps" if torch.backends.mps.is_available() else "cpu"
MEAN = (0.48145466, 0.4578275, 0.40821073); STD = (0.26862954, 0.26130258, 0.27577711)  # open_clip ViT-B-16 defaults

class DS(Dataset):
    def __init__(self, paths):
        self.paths = paths
        self.pre = T.Compose([T.Resize(448, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(448)])
        self.post = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]); img.load(); img = img.convert("RGB")
            img = self.pre(img)
            boxes = [(0,0,224,224),(224,0,448,224),(0,224,224,448),(224,224,448,448),(112,112,336,336)]
            x = torch.stack([self.post(img.crop(b)) for b in boxes])  # 5,3,224,224
            return x, i, 1
        except Exception:
            return torch.zeros(5,3,224,224), i, 0

@torch.no_grad()
def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--splits", default="train,val,test"); ap.add_argument("--bs", type=int, default=16); ap.add_argument("--half", action="store_true"); ap.add_argument("--nw", type=int, default=0); ap.add_argument("--probe", type=int, default=0)
    a = ap.parse_args()
    import open_clip
    model, _, _ = open_clip.create_model_and_transforms("ViT-B-16", pretrained="laion2b_s34b_b88k")
    model = model.to(DEV).eval()
    if a.half: model = model.half()
    dt_ = torch.float16 if a.half else torch.float32
    if a.probe:
        paths = json.load(open(f"{FM}/paths_train.json"))["paths"][:a.probe]; dl = DataLoader(DS(paths), batch_size=a.bs, num_workers=a.nw)
        x,_,_ = next(iter(dl)); model.encode_image(x.view(-1,3,224,224).to(DEV, dt_)); torch.mps.synchronize()
        t0 = time.time(); n = 0
        for x, idx, ok in dl: f = model.encode_image(x.view(-1,3,224,224).to(DEV, dt_)); f.float().cpu(); n += len(idx)
        dt = time.time()-t0; print(f"PROBE mc {n} imgs {dt:.1f}s -> {n/dt:.1f} img/s; projection 15281 = {15281/(n/dt)/60:.1f} min", flush=True); return
    bad = open(f"{RUN}/bad_files_mc.log", "a")
    for split in a.splits.split(","):
        out = f"{RUN}/emb_clipmc_{split}.npy"
        if os.path.exists(out): print(f"{split} exists, skip", flush=True); continue
        paths = json.load(open(f"{FM}/paths_{split}.json"))["paths"]
        dl = DataLoader(DS(paths), batch_size=a.bs, num_workers=a.nw, shuffle=False)
        feats = np.zeros((len(paths), 5, 512), np.float32); t0 = time.time(); nbad = 0
        for b, (x, idx, ok) in enumerate(dl):
            B = x.shape[0]
            f = model.encode_image(x.view(-1, 3, 224, 224).to(DEV, dt_)).float()
            f = (f / f.norm(dim=-1, keepdim=True)).cpu().numpy().reshape(B, 5, 512)
            feats[idx.numpy()] = f
            for j, i in enumerate(idx.tolist()):
                if ok[j].item() == 0: nbad += 1; bad.write(f"{split}\t{paths[i]}\n")
            if b % 20 == 0: print(f"{split} {b*a.bs}/{len(paths)} {time.time()-t0:.0f}s", flush=True)
        np.save(out, feats)
        print(f"{split} done {feats.shape} in {time.time()-t0:.0f}s bad={nbad}", flush=True)
    bad.close()

if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/emb-r2/extract_dino448.py
"""emb-r2: DINOv2 ViT-S/14 (timm vit_small_patch14_dinov2.lvd142m) at 448 px (pos-embed interpolated by
timm img_size=448), pooled 384-d feature, L2-normed. Fallback chosen because ViT-L-14 and CLIP 5-crop both
projected ~31 min on the shared GPU. Same row order as runs/fm-probe/paths_{split}.json.
--probe N measures the first N train images; --half runs the ViT in fp16 on mps."""
import os, json, time, argparse, shutil
import numpy as np, torch
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
ImageFile.LOAD_TRUNCATED_IMAGES = True
RUN = "/Users/yopiesuryadi/lomba-cv/runs/emb-r2"; FM = "/Users/yopiesuryadi/lomba-cv/runs/fm-probe"
DEV = "mps" if torch.backends.mps.is_available() else "cpu"; RES = 448

class DS(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]); img.load(); return self.tf(img.convert("RGB")), i, 1
        except Exception:
            return torch.zeros(3, RES, RES), i, 0

@torch.no_grad()
def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--probe", type=int, default=0); ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--half", action="store_true"); ap.add_argument("--splits", default="train,val,test"); a = ap.parse_args()
    import timm
    model = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, num_classes=0, img_size=RES).to(DEV).eval()
    if a.half: model = model.half()
    dt_ = torch.float16 if a.half else torch.float32
    cfg = timm.data.resolve_data_config({}, model=model)
    tf = T.Compose([T.Resize(RES, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(RES), T.ToTensor(), T.Normalize(cfg["mean"], cfg["std"])])
    print(f"model loaded dev={DEV} half={a.half} mean={cfg['mean']}", flush=True)
    P = {s: json.load(open(f"{FM}/paths_{s}.json"))["paths"] for s in ["train", "val", "test"]}; ntotal = sum(len(v) for v in P.values())
    if a.probe:
        dl = DataLoader(DS(P["train"][:a.probe], tf), batch_size=a.bs, num_workers=4)
        x, _, _ = next(iter(dl)); model(x.to(DEV, dt_)); torch.mps.synchronize()
        t0 = time.time(); n = 0
        for x, idx, ok in dl: model(x.to(DEV, dt_)).float().cpu(); n += len(idx)
        dt = time.time() - t0; print(f"PROBE dino448 {n} imgs {dt:.1f}s -> {n/dt:.1f} img/s; projection {ntotal} = {ntotal/(n/dt)/60:.1f} min", flush=True); return
    bad = open(f"{RUN}/bad_files_dino448.log", "a")
    for split in a.splits.split(","):
        out = f"{RUN}/emb_dino448_{split}.npy"
        if os.path.exists(out): print(f"{split} exists, skip", flush=True); continue
        paths = P[split]; dl = DataLoader(DS(paths, tf), batch_size=a.bs, num_workers=4, shuffle=False)
        feats = np.zeros((len(paths), 384), np.float32); t0 = time.time(); nbad = 0
        for b, (x, idx, ok) in enumerate(dl):
            f = model(x.to(DEV, dt_)).float(); f = (f / f.norm(dim=-1, keepdim=True)).cpu().numpy(); feats[idx.numpy()] = f
            for j, i in enumerate(idx.tolist()):
                if ok[j].item() == 0: nbad += 1; bad.write(f"{split}\t{paths[i]}\n")
            if b % 25 == 0: print(f"{split} {b*a.bs}/{len(paths)} {time.time()-t0:.0f}s", flush=True)
        np.save(out, feats); shutil.copy(f"{FM}/paths_{split}.json", f"{RUN}/paths_{split}.json")
        print(f"{split} done {feats.shape} in {time.time()-t0:.0f}s bad={nbad}", flush=True)
    bad.close()

if __name__ == "__main__":
    main()

In [ ]:
%%simpan runs/emb-r2/train.py
"""emb-r2: SVM-RBF (C 3/10/30 balanced, Platt prob) + MLP (3 seeds, avg) on new frozen embeddings
(OpenCLIP ViT-L-14 768-d and/or CLIP ViT-B-16 5-crop mean), alone or concatenated with the 896-d
round-1 features (fm-probe CLIP ViT-B-16 512 + DINOv2 ViT-S/14 384, DINO L2-normed). Row order is
runs/fm-probe/paths_{split}.json for everything. Also reports Macro F1 on the CLEAN val subset (val
images with no md5 / phash<=6 near-duplicate in train, computed like ensemble.py from runs/eda/meta.csv).
Usage: python train.py --feat l14 [--models svm,mlp] ; python train.py --finalize [--name X] [--suffix _mlp]"""
import os, json, time, argparse
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from joblib import Parallel, delayed

RUN = "/Users/yopiesuryadi/lomba-cv/runs/emb-r2"; FM = "/Users/yopiesuryadi/lomba-cv/runs/fm-probe"
MC = "/Users/yopiesuryadi/lomba-cv/runs/emb-heads"; ROOT = "/Users/yopiesuryadi/lomba-cv"
CLASSES = ["Aman", "Rusak", "Lainnya"]; C2I = {c: i for i, c in enumerate(CLASSES)}; SPLITS = ["train", "val", "test"]
torch.set_num_threads(4)

def l2(x): return x / np.linalg.norm(x, axis=1, keepdims=True).clip(1e-8)

def load_feats(feat):
    parts = []
    for name in feat.split("+"):
        if name == "base":
            parts.append({s: np.concatenate([np.load(f"{FM}/emb_clip_{s}.npy"), l2(np.load(f"{FM}/emb_dino_{s}.npy"))], 1) for s in SPLITS})
        elif name == "l14":
            parts.append({s: np.load(f"{RUN}/emb_l14_{s}.npy") for s in SPLITS})
        elif name == "mc":   # 5-crop CLIP ViT-B-16, mean of per-crop L2-normed embeddings, re-normed
            src = RUN if os.path.exists(f"{RUN}/emb_clipmc_train.npy") else MC
            parts.append({s: l2(np.load(f"{src}/emb_clipmc_{s}.npy").mean(1)) for s in SPLITS})
        else: raise ValueError(name)
    return {s: np.concatenate([p[s] for p in parts], 1).astype(np.float32) for s in SPLITS}

def clean_val_mask(val_paths):
    """val rows without md5 / phash<=6 match to any train image (same rule as ensemble.py)."""
    meta = pd.read_csv(f"{ROOT}/runs/eda/meta.csv", dtype=str)
    trp = set(pd.read_csv(f"{ROOT}/split/train.csv").path)
    def ph(s):
        try: return int(s, 16)
        except Exception: return None
    trn = meta[meta.path.isin(trp)]; trn_md5 = set(trn.md5)
    trn_h = np.array([h for h in trn.phash.map(ph) if h is not None], dtype=np.uint64)
    vl = meta[meta.path.isin(set(val_paths))].set_index("path")
    ok = {}
    for p in val_paths:
        if p not in vl.index: ok[p] = True; continue
        h = ph(vl.at[p, "phash"]); m = vl.at[p, "md5"]
        if m in trn_md5 or h is None: ok[p] = False; continue
        x = trn_h ^ np.uint64(h); d = np.array([bin(int(v)).count("1") for v in x]).min()
        ok[p] = d > 6
    return np.array([ok[p] for p in val_paths])

def metrics(y, p, cmask):
    pred = p.argmax(1)
    return {"macro_f1": float(f1_score(y, pred, average="macro")),
            "macro_f1_clean": float(f1_score(y[cmask], pred[cmask], average="macro")),
            "per_class_f1": dict(zip(CLASSES, map(float, f1_score(y, pred, average=None)))),
            "per_class_f1_clean": dict(zip(CLASSES, map(float, f1_score(y[cmask], pred[cmask], average=None)))),
            "cm": confusion_matrix(y, pred).tolist(), "cm_clean": confusion_matrix(y[cmask], pred[cmask]).tolist()}

class MLP(nn.Module):
    def __init__(self, d, h=512, p=0.3):
        super().__init__(); self.net = nn.Sequential(nn.Dropout(0.1), nn.Linear(d, h), nn.GELU(), nn.Dropout(p), nn.Linear(h, 3))
    def forward(self, x): return self.net(x)

def train_mlp(Xtr, ytr, Xva, yva, Xte, seed, epochs=60, patience=12, lr=1e-3, wd=5e-2, bs=256, ls=0.1, log=print):
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr_t, ytr_t = torch.tensor(Xtr), torch.tensor(ytr); Xva_t, Xte_t = torch.tensor(Xva), torch.tensor(Xte)
    cnt = np.bincount(ytr, minlength=3); w = torch.tensor((len(ytr) / (3 * cnt)) ** 0.5, dtype=torch.float32)
    model = MLP(Xtr.shape[1]); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * int(np.ceil(len(ytr) / bs)), pct_start=0.1)
    lossf = nn.CrossEntropyLoss(weight=w, label_smoothing=ls); best = (-1, None, 0); bad = 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(ytr))
        for i in range(0, len(ytr), bs):
            idx = perm[i:i + bs]; opt.zero_grad(); lossf(model(Xtr_t[idx]), ytr_t[idx]).backward(); opt.step(); sched.step()
        model.eval()
        with torch.no_grad(): f1 = f1_score(yva, model(Xva_t).argmax(1).numpy(), average="macro")
        if f1 > best[0]: best = (f1, {k: v.clone() for k, v in model.state_dict().items()}, ep); bad = 0
        else: bad += 1
        if bad >= patience: break
    log(f"  seed{seed} stop ep{ep} best valF1={best[0]:.4f}@{best[2]}")
    model.load_state_dict(best[1]); model.eval()
    with torch.no_grad(): return torch.softmax(model(Xva_t), 1).numpy(), torch.softmax(model(Xte_t), 1).numpy(), best[2]

def fit_svm(C, Xtr, ytr, Xva, Xte):
    t = time.time(); clf = SVC(C=C, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=0).fit(Xtr, ytr)
    assert list(clf.classes_) == [0, 1, 2]
    return C, clf.predict_proba(Xva), clf.predict_proba(Xte), time.time() - t

def finalize(name, suffix, P, yva, cmask, results, log):
    z = np.load(f"{RUN}/probs/{name}.npz"); pva, pte = z["val"], z["test"]
    log(f"SELECTED{suffix}: {name}  MacroF1={results[name]['macro_f1']:.4f} clean={results[name]['macro_f1_clean']:.4f}")
    log(classification_report(yva, pva.argmax(1), target_names=CLASSES, digits=4))
    log("cm full:\n" + str(np.array(results[name]["cm"])) + "\ncm clean:\n" + str(np.array(results[name]["cm_clean"])))
    pd.DataFrame({"path": P["val"]["paths"], "label": P["val"]["labels"], "p_Aman": pva[:, 0], "p_Rusak": pva[:, 1], "p_Lainnya": pva[:, 2]}).to_csv(f"{RUN}/val_probs{suffix}.csv", index=False)
    fn = [os.path.basename(p) for p in P["test"]["paths"]]
    pd.DataFrame({"filename": fn, "p_Aman": pte[:, 0], "p_Rusak": pte[:, 1], "p_Lainnya": pte[:, 2]}).to_csv(f"{RUN}/test_probs{suffix}.csv", index=False)
    samp = pd.read_csv(f"{ROOT}/data/sample_submission.csv")
    sub = samp[["filename"]].merge(pd.DataFrame({"filename": fn, "label": [CLASSES[i] for i in pte.argmax(1)]}), on="filename", how="left")
    assert len(sub) == 3057 and sub.label.notna().all(), (len(sub), sub.label.isna().sum())
    sub.to_csv(f"{RUN}/submission{suffix}.csv", index=False)
    log(f"test pred distribution{suffix}: {sub.label.value_counts().to_dict()} rows={len(sub)}")
    results["_selected" + suffix] = name

def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--feat", default="l14"); ap.add_argument("--models", default="svm,mlp")
    ap.add_argument("--seeds", type=int, default=3); ap.add_argument("--Cs", default="3,10,30"); ap.add_argument("--jobs", type=int, default=3)
    ap.add_argument("--finalize", action="store_true"); ap.add_argument("--name", default=None); ap.add_argument("--suffix", default="")
    a = ap.parse_args()
    logf = open(f"{RUN}/train_{'finalize' if a.finalize else a.feat}.log", "a")
    def log(s): print(s, flush=True); logf.write(s + "\n"); logf.flush()
    P = {s: json.load(open(f"{FM}/paths_{s}.json")) for s in SPLITS}
    ytr = np.array([C2I[l] for l in P["train"]["labels"]]); yva = np.array([C2I[l] for l in P["val"]["labels"]])
    cmask = clean_val_mask(P["val"]["paths"])
    if a.finalize:   # merge per-feature result files (parallel train.py runs each write their own)
        import glob; results = {}
        for f in sorted(glob.glob(f"{RUN}/results_*.json")): results.update(json.load(open(f)))
        resf = f"{RUN}/results.json"; results.update({k: v for k, v in (json.load(open(resf)) if os.path.exists(resf) else {}).items() if k.startswith("_")})
        R = {k: v for k, v in results.items() if not k.startswith("_")}
        name = a.name or max(R, key=lambda k: R[k]["macro_f1"])
        finalize(name, a.suffix, P, yva, cmask, results, log); json.dump(results, open(resf, "w"), indent=1); return
    resf = f"{RUN}/results_{a.feat}.json"; results = json.load(open(resf)) if os.path.exists(resf) else {}
    F = load_feats(a.feat)
    log(f"=== feat={a.feat} dims={F['train'].shape} models={a.models} clean_val={int(cmask.sum())}/{len(cmask)} (per class {np.bincount(yva[cmask], minlength=3).tolist()}) {time.strftime('%H:%M:%S')}")
    sc = StandardScaler().fit(F["train"]); Xtr, Xva, Xte = (sc.transform(F[s]).astype(np.float32) for s in SPLITS)
    def save(name, pva, pte, extra):
        m = metrics(yva, pva, cmask); m.update(extra); m["feat"] = a.feat; results[name] = m
        np.savez(f"{RUN}/probs/{name}.npz", val=pva.astype(np.float32), test=pte.astype(np.float32))
        json.dump(results, open(resf, "w"), indent=1)
        log(f"{name:40s} F1={m['macro_f1']:.4f} clean={m['macro_f1_clean']:.4f} perclass={ {k: round(v,4) for k,v in m['per_class_f1'].items()} } clean_perclass={ {k: round(v,4) for k,v in m['per_class_f1_clean'].items()} } t={extra.get('seconds',0):.0f}s")
    if "mlp" in a.models:
        t0 = time.time(); pvas, ptes = [], []
        for seed in range(a.seeds):
            t1 = time.time(); pva, pte, bep = train_mlp(Xtr, ytr, Xva, yva, Xte, seed, log=log)
            save(f"MLP[{a.feat}] seed={seed}", pva, pte, {"seconds": time.time() - t1, "best_epoch": bep}); pvas.append(pva); ptes.append(pte)
        save(f"MLP[{a.feat}] avg{a.seeds}", np.mean(pvas, 0), np.mean(ptes, 0), {"seconds": time.time() - t0})
    if "svm" in a.models:
        Cs = [float(c) for c in a.Cs.split(",")]
        for C, pva, pte, sec in Parallel(n_jobs=a.jobs)(delayed(fit_svm)(C, Xtr, ytr, Xva, Xte) for C in Cs):
            save(f"SVM-RBF[{a.feat}] C={C:g} balanced", pva, pte, {"seconds": sec})
    log(f"=== done feat={a.feat} {time.strftime('%H:%M:%S')}")

if __name__ == "__main__":
    main()

In [ ]:
os.makedirs("runs/emb-r2/probs", exist_ok=True)   # train.py menyimpan probs/<varian>.npz tanpa membuat foldernya
!{PY} runs/emb-r2/extract_l14.py --half --nw 0
!{PY} runs/emb-r2/extract_multicrop.py --half --nw 0
# !{PY} runs/emb-r2/extract_dino448.py --probe 256 --half   # hanya probe laju, tidak dipakai

In [ ]:
!{PY} runs/emb-r2/train.py --feat l14 --models svm,mlp
!{PY} runs/emb-r2/train.py --feat l14+base --models svm,mlp
!{PY} runs/emb-r2/train.py --feat mc+base --models mlp
!{PY} runs/emb-r2/train.py --feat l14+mc+base --models mlp

Nama varian diberikan eksplisit karena urutan run saat lomba: `--finalize` tanpa `--name` sekarang akan memilih
`MLP[l14+mc+base] seed=1` (0,9227, dijalankan belakangan), bukan `SVM-RBF[l14+base] C=30` yang tercatat di REPORT.

In [ ]:
!{PY} runs/emb-r2/train.py --finalize --name "SVM-RBF[l14+base] C=30 balanced"
!{PY} runs/emb-r2/train.py --finalize --name "MLP[mc+base] avg3" --suffix _mlp

## 10. Ensemble final (`ensemble.py`)

`ensemble.py` membaca `runs/<nama>/val_probs.csv` dan `test_probs.csv`, mencari bobot (grid 0-3) dan bias logit
Rusak/Lainnya di **val bersih**, merata-ratakan prediksi antar gambar test kembar, lalu menimpa 467 jawaban dengan
`test_pseudo_labels_strong.csv`. Hasil: `runs/ensemble/submission.csv`.

**[jembatan]** Kedua model MLP menulis hasilnya sebagai `val_probs_mlp.csv` / `test_probs_mlp.csv`, sedangkan
`ensemble.py` hanya membaca `val_probs.csv`. Sel berikut menyalinnya ke folder run tersendiri
(`runs/emb-heads-mlp`, `runs/emb-r2-mlp`). Cara persis yang dipakai saat lomba tidak tercatat di paket.

Catatan dari BACA-INI: cek aturan lomba soal model pretrained dan pemanfaatan gambar test yang identik dengan
train. Kalau override duplikat dilarang, matikan blok `test_pseudo_labels_strong.csv` di `ensemble.py`.

In [ ]:
for src, dst in [("runs/emb-heads", "runs/emb-heads-mlp"), ("runs/emb-r2", "runs/emb-r2-mlp")]:
    os.makedirs(dst, exist_ok=True)
    for kind in ["val", "test"]:
        shutil.copy(f"{src}/{kind}_probs_mlp.csv", f"{dst}/{kind}_probs.csv")
    print(dst, sorted(os.listdir(dst)))

In [ ]:
%%simpan ensemble.py
"""Ensemble semua runs/*/val_probs.csv + test_probs.csv.
- bobot & bias logit dipilih di val BERSIH (val tanpa near-dup phash<=6 / md5 ke train fold)
- test: rata-rata prob antar test yang kembar (md5 / phash<=6), lalu override pseudo-label kuat dari EDA
tulis runs/ensemble/submission.csv"""
import sys, os, glob, itertools, numpy as np, pandas as pd
from sklearn.metrics import f1_score, confusion_matrix
C=["Aman","Rusak","Lainnya"]; P=[f"p_{c}" for c in C]; L={c:i for i,c in enumerate(C)}
runs=[d for d in sorted(glob.glob("runs/*")) if os.path.exists(f"{d}/val_probs.csv") and os.path.exists(f"{d}/test_probs.csv")]
if len(sys.argv)>1: runs=[r for r in runs if os.path.basename(r) in sys.argv[1:]]
val=pd.read_csv("split/val.csv"); test=pd.read_csv("data/sample_submission.csv")[["filename"]]
y=val["label"].map(L).values

# ---- val bersih & grup test kembar dari meta EDA
meta=pd.read_csv("runs/eda/meta.csv",dtype=str)
trpaths=set(pd.read_csv("split/train.csv").path); vlpaths=set(val.path)
trn=meta[meta.path.isin(trpaths)]; vl=meta[meta.path.isin(vlpaths)]; tst=meta[meta.split=="test"]
print("meta fold:",len(trn),len(vl),len(tst))
def ph(s):
    try: return int(s,16)
    except: return None
trn_h=np.array([h for h in trn.phash.map(ph) if h is not None],dtype=object); trn_md5=set(trn.md5)
def hd(a,b): return bin(a^b).count("1")
clean=[]
for p,m,h in zip(vl.path,vl.md5,vl.phash.map(ph)):
    ok = m not in trn_md5 and h is not None and min(hd(h,t) for t in trn_h)>6
    clean.append((p,ok))
clean=dict(clean); cmask=np.array([clean.get(p,True) for p in val.path])
print(f"val total {len(val)}, val bersih {cmask.sum()} (Aman/Rusak/Lainnya = {np.bincount(y[cmask],minlength=3)})")
# grup test kembar: union-find md5 + phash<=6
tf=list(tst.filename); th=list(tst.phash.map(ph)); tm=list(tst.md5)
par=list(range(len(tf)))
def find(i):
    while par[i]!=i: par[i]=par[par[i]]; i=par[i]
    return i
bymd5={}
for i,m in enumerate(tm): bymd5.setdefault(m,[]).append(i)
for g in bymd5.values():
    for i in g[1:]: par[find(i)]=find(g[0])
for i in range(len(tf)):
    if th[i] is None: continue
    for j in range(i+1,len(tf)):
        if th[j] is not None and hd(th[i],th[j])<=6: par[find(j)]=find(i)
grp={f:find(i) for i,f in enumerate(tf)}
test["grp"]=test.filename.map(grp)
print("grup test kembar (size>1):",(test.groupby("grp").size()>1).sum())

# ---- muat runs
V={}; T={}
for r in runs:
    v=pd.read_csv(f"{r}/val_probs.csv"); v["path"]=v["path"].astype(str)
    v=val[["path"]].merge(v[["path"]+P],on="path",how="left")
    t=test[["filename"]].merge(pd.read_csv(f"{r}/test_probs.csv")[["filename"]+P],on="filename",how="left")
    mv=v[P].isna().any(axis=1).sum(); mt=t[P].isna().any(axis=1).sum()
    V[r]=v[P].fillna(1/3).values; T[r]=t[P].fillna(1/3).values
    pr=V[r].argmax(1)
    print(f"{os.path.basename(r):16s} val F1={f1_score(y,pr,average='macro'):.4f}  BERSIH F1={f1_score(y[cmask],pr[cmask],average='macro'):.4f}  per-class(bersih)={np.round(f1_score(y[cmask],pr[cmask],average=None),3)}  missing={mv}/{mt}")
names=list(V); n=len(names)
def comb(D,w): return sum(wi*np.log(D[k]+1e-6) for wi,k in zip(w,names))/sum(w)
best=(-1,None,None); grid=[0,1,2,3] if n<=4 else [0,1,2]
for w in itertools.product(grid,repeat=n):
    if sum(w)==0: continue
    lv=comb(V,w)[cmask]
    for b2 in np.arange(-2.0,2.01,0.25):
        for b1 in np.arange(-2.0,1.51,0.25):
            f=f1_score(y[cmask],(lv+np.array([0,b1,b2])).argmax(1),average="macro")
            if f>best[0]: best=(f,w,(b1,b2))
f,w,(b1,b2)=best; bias=np.array([0,b1,b2])
print(f"\nBEST ensemble BERSIH val F1={f:.4f} weights={dict(zip([os.path.basename(k) for k in names],w))} bias Rusak={b1} Lainnya={b2}")
pv=(comb(V,w)+bias).argmax(1)
print("full-val F1",round(f1_score(y,pv,average="macro"),4),"| bersih per-class",np.round(f1_score(y[cmask],pv[cmask],average=None),4))
print(confusion_matrix(y[cmask],pv[cmask]))
# ---- test
lt=comb(T,w)
lt=pd.DataFrame(lt).groupby(test.grp.values).transform("mean").values   # rata-rata antar kembar
pt=(lt+bias).argmax(1)
out=test[["filename"]].copy(); out["label"]=[C[i] for i in pt]
ps=pd.read_csv("runs/eda/test_pseudo_labels_strong.csv")
m=out.merge(ps[["test_filename","label"]].rename(columns={"test_filename":"filename","label":"pl"}),on="filename",how="left")
chg=((m.pl.notna())&(m.pl!=m.label)).sum()
out["label"]=np.where(m.pl.notna(),m.pl,m.label)
os.makedirs("runs/ensemble",exist_ok=True); out.to_csv("runs/ensemble/submission.csv",index=False)
print(f"\noverride pseudo-label: {m.pl.notna().sum()} file, {chg} berubah")
print("test label distribution:",out["label"].value_counts().to_dict(),"rows",len(out))

In [ ]:
!{PY} ensemble.py emb-r2-mlp emb-heads-mlp timm-convnext convnext-r2

## 11. Verifikasi

Bandingkan `runs/ensemble/submission.csv` hasil notebook dengan `submission.csv` final di paket
(3.057 baris). Hasil latih ulang di mesin lain bisa sedikit berbeda (GPU, versi library, seed DataLoader).

In [ ]:
import pandas as pd
paket = pd.read_csv("submission_paket.csv")
print("submission paket:", len(paket), "baris", paket.label.value_counts().to_dict())
if os.path.exists("runs/ensemble/submission.csv"):
    baru = pd.read_csv("runs/ensemble/submission.csv")
    m = paket.merge(baru, on="filename", suffixes=("_paket", "_baru"))
    print("baris cocok:", len(m), "| label sama:", f"{(m.label_paket == m.label_baru).mean():.2%}")
    print(pd.crosstab(m.label_paket, m.label_baru))